In [ ]:
# @title S(H)ARP — Start analysis and upload input files { display-mode: "form" }
# @markdown Run this cell first to initialize S(H)ARP and upload the input files.

from pathlib import Path
from IPython.display import display, HTML
import os
import sys
import re
import json
import time
import html
import shutil
import hashlib
import zipfile
import tarfile
import gzip
import platform

# =====================================================================
# Internal options
# =====================================================================

UPLOAD_FILES_NOW = True

DEFAULT_ORGANISM_NAME = "Streptomyces sp."
DEFAULT_STRAIN_NAME = "unknown"

RESET_PREVIOUS_INPUTS = True

REQUIRE_BAKTA_FOR_FASTA_ONLY = True
ALLOW_PRODIGAL_FALLBACK_FOR_DEBUG = False
PREFER_EXISTING_ANNOTATION_WHEN_AVAILABLE = True

# =====================================================================
# S(H)ARP identity
# =====================================================================

SHARP_PROJECT_NAME = "sharp_igem_usp_brazil_2026"
SHARP_PROJECT_TITLE = "S(H)ARP"
SHARP_PROJECT_FULL_NAME = "Streptomyces Hidden Antibiotic Regulated Pathways"
SHARP_PROJECT_SUBTITLE = "BGC Regulatory Region Finder"
SHARP_TEAM = "iGEM USP-Brazil 2026"
SHARP_VERSION = "0.1.0-dev"
SHARP_INITIALIZED_AT = time.strftime("%Y-%m-%d %H:%M:%S")

ANALYSIS_NAME = globals().get("ANALYSIS_NAME", "sharp_run_01")

# =====================================================================
# Public workflow policy
# =====================================================================

SHARP_INPUT_POLICY = {
    "recommended_input": "annotated_genome_package_or_genome_fasta",
    "supported_modes": {
        "genome_fasta_only": {
            "required_files": ["genome.fasta"],
            "annotation_backend": "bakta_preferred",
            "requires_bakta_db": True,
        },
        "annotated_genome_package": {
            "required_files": [
                "genome.fasta",
                "annotation.gff3_or_genbank",
                "proteins.faa",
            ],
            "annotation_backend": "existing_annotation",
            "requires_bakta_db": False,
        },
        "demo_package": {
            "required_files": [
                "genome.fasta",
                "annotation.gff3_or_genbank",
                "proteins.faa",
                "heptarepeats2.meme",
                "sarp_custom.hmm",
            ],
            "annotation_backend": "existing_annotation",
            "requires_bakta_db": False,
        },
    },
    "backend_priority": [
        "existing_annotation",
        "bakta",
        "prodigal_debug_fallback",
    ],
}

# =====================================================================
# Runtime detection
# =====================================================================

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CONTENT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

# =====================================================================
# Workspace paths
# =====================================================================

BASE_DIR = CONTENT_DIR / SHARP_PROJECT_NAME

DATA_DIR = BASE_DIR / "data"
UPLOADS_DIR = DATA_DIR / "uploads"
NORMALIZED_DIR = DATA_DIR / "normalized"

DB_DIR = BASE_DIR / "databases"
RESULTS_DIR = BASE_DIR / "results"
TMP_DIR = BASE_DIR / "tmp"
MODULES_DIR = BASE_DIR / "modules"
CONFIG_DIR = BASE_DIR / "config"

RUN_DIR = RESULTS_DIR / ANALYSIS_NAME

ANNOTATION_DIR = RUN_DIR / "annotation"
BAKTA_OUTPUT_DIR = ANNOTATION_DIR / "bakta"

TABLES_DIR = RUN_DIR / "tables"
REPORT_DIR = RUN_DIR / "report"
LOGS_DIR = RUN_DIR / "logs"
FIMO_DIR = RUN_DIR / "fimo"
HMM_DIR_RUN = RUN_DIR / "hmm"
DOMAIN_DIR_RUN = RUN_DIR / "domains"
EMBEDDING_DIR_RUN = RUN_DIR / "embeddings"

MOTIFS_DIR = DB_DIR / "motifs"
HMM_DIR = DB_DIR / "hmm"
DOMAIN_MODELS_DIR = DB_DIR / "domain_models"
EMBEDDINGS_DIR = DB_DIR / "embeddings"
REPORT_ASSETS_DIR = DB_DIR / "report_assets"
BAKTA_DB_PARENT_DIR = DB_DIR / "bakta_db"

for path in [
    BASE_DIR,
    DATA_DIR,
    UPLOADS_DIR,
    NORMALIZED_DIR,
    DB_DIR,
    RESULTS_DIR,
    TMP_DIR,
    MODULES_DIR,
    CONFIG_DIR,
    RUN_DIR,
    ANNOTATION_DIR,
    BAKTA_OUTPUT_DIR,
    TABLES_DIR,
    REPORT_DIR,
    LOGS_DIR,
    FIMO_DIR,
    HMM_DIR_RUN,
    DOMAIN_DIR_RUN,
    EMBEDDING_DIR_RUN,
    MOTIFS_DIR,
    HMM_DIR,
    DOMAIN_MODELS_DIR,
    EMBEDDINGS_DIR,
    REPORT_ASSETS_DIR,
    BAKTA_DB_PARENT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

# =====================================================================
# Environment paths used by later cells
# =====================================================================

MAMBA_ROOT_PREFIX = BASE_DIR / "micromamba"
MAMBA_BIN = BASE_DIR / "bin" / "micromamba"
MAMBA_ENV_DIR = BASE_DIR / "envs" / "sharp_bio"
MAMBA_ENV_BIN = MAMBA_ENV_DIR / "bin"

KNOWN_ENV_BINS = [
    Path("/content/sharp_env/bin"),
    MAMBA_ENV_BIN,
    BASE_DIR / "envs" / "sharp_bio" / "bin",
    Path("/usr/local/bin"),
    Path("/usr/bin"),
    Path("/bin"),
]

for env_bin in KNOWN_ENV_BINS:
    if env_bin.exists():
        env_bin_text = str(env_bin)
        current_path = os.environ.get("PATH", "")

        if env_bin_text not in current_path.split(os.pathsep):
            os.environ["PATH"] = env_bin_text + os.pathsep + current_path

os.environ["MAMBA_ROOT_PREFIX"] = str(MAMBA_ROOT_PREFIX)

# =====================================================================
# Rotifer integration settings
# =====================================================================

ROTIFER_REPO_URL = "https://github.com/leepusp/rotifer.git"
ROTIFER_BRANCH = "master"
ROTIFER_DIR = CONTENT_DIR / "rotifer"
ROTIFER_LIB = ROTIFER_DIR / "lib"

if ROTIFER_LIB.exists():
    if str(ROTIFER_LIB) not in sys.path:
        sys.path.insert(0, str(ROTIFER_LIB))

    os.environ["PYTHONPATH"] = (
        str(ROTIFER_LIB)
        + os.pathsep
        + os.environ.get("PYTHONPATH", "")
    )

if str(MODULES_DIR) not in sys.path:
    sys.path.insert(0, str(MODULES_DIR))

# =====================================================================
# Internal resources
# =====================================================================

SHARP_INTERNAL_RESOURCES = {
    "heptamer_meme": MOTIFS_DIR / "heptarepeats2.meme",
    "sarp_hmm": HMM_DIR / "sarp_custom.hmm",
    "domain_models_hmm": DOMAIN_MODELS_DIR / "domain_models.hmm",
    "embedding_model": EMBEDDINGS_DIR / "model",
    "embedding_reference": EMBEDDINGS_DIR / "reference_embeddings.parquet",
    "scoring_config": CONFIG_DIR / "sharp_scoring_config.json",
    "report_module": MODULES_DIR / "operon_fig_colab.py",
    "bakta_db_parent": BAKTA_DB_PARENT_DIR,
}

DEFAULT_SCORING_CONFIG = {
    "version": SHARP_VERSION,
    "fimo_pvalue_threshold": 1e-4,
    "sarp_hmm_evalue_threshold": 1e-3,
    "neighborhood_before_genes": 10,
    "neighborhood_after_genes": 10,
    "domain_step_status": "pending_models",
    "embedding_step_status": "pending_models",
    "annotation_policy": SHARP_INPUT_POLICY,
}

if not SHARP_INTERNAL_RESOURCES["scoring_config"].exists():
    SHARP_INTERNAL_RESOURCES["scoring_config"].write_text(
        json.dumps(DEFAULT_SCORING_CONFIG, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

# =====================================================================
# Shared notebook context
# =====================================================================

SHARP_PATHS = {
    "content_dir": CONTENT_DIR,
    "base_dir": BASE_DIR,
    "data_dir": DATA_DIR,
    "uploads_dir": UPLOADS_DIR,
    "normalized_dir": NORMALIZED_DIR,
    "db_dir": DB_DIR,
    "results_dir": RESULTS_DIR,
    "tmp_dir": TMP_DIR,
    "modules_dir": MODULES_DIR,
    "config_dir": CONFIG_DIR,
    "run_dir": RUN_DIR,
    "annotation_dir": ANNOTATION_DIR,
    "bakta_output_dir": BAKTA_OUTPUT_DIR,
    "tables_dir": TABLES_DIR,
    "report_dir": REPORT_DIR,
    "logs_dir": LOGS_DIR,
    "fimo_dir": FIMO_DIR,
    "hmm_run_dir": HMM_DIR_RUN,
    "domain_run_dir": DOMAIN_DIR_RUN,
    "embedding_run_dir": EMBEDDING_DIR_RUN,
    "motifs_dir": MOTIFS_DIR,
    "hmm_dir": HMM_DIR,
    "domain_models_dir": DOMAIN_MODELS_DIR,
    "embeddings_dir": EMBEDDINGS_DIR,
    "report_assets_dir": REPORT_ASSETS_DIR,
    "bakta_db_parent_dir": BAKTA_DB_PARENT_DIR,
    "mamba_root_prefix": MAMBA_ROOT_PREFIX,
    "mamba_bin": MAMBA_BIN,
    "mamba_env_dir": MAMBA_ENV_DIR,
    "mamba_env_bin": MAMBA_ENV_BIN,
    "rotifer_dir": ROTIFER_DIR,
    "rotifer_lib": ROTIFER_LIB,
}

# =====================================================================
# Default placeholders consumed by later cells
# =====================================================================

workflow = "sharp_not_initialized"
annotation_mode = "pending_input"
input_mode = "pending_input"

GENOME_FASTA = NORMALIZED_DIR / "sharp_input_genome.fna"
GENOME_ANNOTATION = NORMALIZED_DIR / "sharp_input_annotation.gff3"
PROTEIN_FASTA = NORMALIZED_DIR / "sharp_input_proteins.faa"

INPUT_FILES = {}

SHARP_SAMPLE = {
    "project_name": SHARP_PROJECT_NAME,
    "analysis_name": ANALYSIS_NAME,
    "organism_name": DEFAULT_ORGANISM_NAME,
    "strain_name": DEFAULT_STRAIN_NAME,
}

SHARP_INPUT_CONTRACT = {
    "status": "pending_input",
    "mode": "not_selected",
    "workflow": workflow,
    "annotation_mode": annotation_mode,
    "genome_fasta": str(GENOME_FASTA),
    "annotation_file": str(GENOME_ANNOTATION),
    "protein_fasta": str(PROTEIN_FASTA),
}

cfg = {
    "project": SHARP_PROJECT_NAME,
    "analysis_name": ANALYSIS_NAME,
    "paths": SHARP_PATHS,
    "internal_resources": SHARP_INTERNAL_RESOURCES,
    "input_policy": SHARP_INPUT_POLICY,
    "input_contract": SHARP_INPUT_CONTRACT,
}

# =====================================================================
# JSON helpers
# =====================================================================

def json_ready(value):
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}

    if isinstance(value, list):
        return [json_ready(v) for v in value]

    if isinstance(value, tuple):
        return [json_ready(v) for v in value]

    return value


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(json_ready(data), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

# =====================================================================
# Public UI
# =====================================================================

def display_sharp_cover():
    display(HTML(f"""
    <style>
      .sharp-cover {{
        --sharp-bg:#97003F;
        --sharp-bg-dark:#760032;
        --sharp-deep:#4D0022;
        --sharp-pink:#D8AFC0;
        --sharp-soft:#F1DDE6;
        --sharp-white:#FFFFFF;
        --sharp-text:#2D0015;
        --sharp-muted:#5E2038;
        font-family:Arial, Helvetica, sans-serif;
      }}

      .sharp-cover .hero {{
        border-radius:24px;
        overflow:hidden;
        background:
          radial-gradient(circle at 14% 16%, rgba(216,175,192,0.28), transparent 24%),
          radial-gradient(circle at 84% 22%, rgba(255,255,255,0.16), transparent 28%),
          linear-gradient(135deg,var(--sharp-bg),var(--sharp-deep));
        color:white;
        padding:34px 38px;
        margin:12px 0 14px 0;
        box-shadow:0 18px 50px rgba(77,0,34,0.22);
      }}

      .sharp-cover .wordmark {{
        font-family:"Bodoni 72","Bodoni MT",Didot,Georgia,"Times New Roman",serif;
        font-size:58px;
        line-height:0.95;
        font-weight:800;
        letter-spacing:0.5px;
        margin:0 0 8px 0;
      }}

      .sharp-cover .full {{
        font-size:18px;
        font-weight:750;
        color:var(--sharp-soft);
        margin:0 0 8px 0;
      }}

      .sharp-cover .subtitle {{
        font-size:14px;
        color:var(--sharp-pink);
        margin:0 0 18px 0;
      }}

      .sharp-cover .meta {{
        display:flex;
        flex-wrap:wrap;
        gap:8px;
        margin-top:16px;
      }}

      .sharp-cover .pill {{
        display:inline-block;
        border:1px solid rgba(255,255,255,0.28);
        background:rgba(255,255,255,0.10);
        color:white;
        border-radius:999px;
        padding:7px 12px;
        font-size:12px;
        font-weight:800;
      }}

      .sharp-cover .instruction {{
        border-radius:18px;
        border:1px solid rgba(151,0,63,0.22);
        background:linear-gradient(135deg,#FFF7FA,#F7E4EC);
        color:var(--sharp-text);
        padding:16px 18px;
        margin:12px 0;
      }}

      .sharp-cover .instruction-title {{
        color:var(--sharp-bg-dark);
        font-size:17px;
        font-weight:900;
        margin-bottom:7px;
      }}

      .sharp-cover .instruction-text {{
        color:var(--sharp-muted);
        font-size:13.5px;
        line-height:1.55;
      }}

      .sharp-cover code {{
        color:var(--sharp-bg-dark);
        font-family:Consolas, Monaco, monospace;
      }}

      @media (prefers-color-scheme: dark) {{
        .sharp-cover .instruction {{
          background:linear-gradient(135deg,#211018,#320017);
          border-color:rgba(216,175,192,0.24);
          color:#F7E4EC;
        }}

        .sharp-cover .instruction-title {{
          color:#FFFFFF;
        }}

        .sharp-cover .instruction-text {{
          color:#D8AFC0;
        }}

        .sharp-cover code {{
          color:#FFFFFF;
        }}
      }}
    </style>

    <div class="sharp-cover">
      <div class="hero">
        <div class="wordmark">S(H)ARP</div>
        <div class="full">{html.escape(SHARP_PROJECT_FULL_NAME)}</div>
        <div class="subtitle">{html.escape(SHARP_PROJECT_SUBTITLE)}</div>
        <div class="meta">
          <span class="pill">{html.escape(SHARP_TEAM)}</span>
          <span class="pill">version {html.escape(SHARP_VERSION)}</span>
          <span class="pill">analysis {html.escape(ANALYSIS_NAME)}</span>
        </div>
      </div>

      <div class="instruction">
        <div class="instruction-title">Step 1 — Select input files below</div>
        <div class="instruction-text">
          Recommended input: <code>genome.fasta + annotation.gff3/gbff + proteins.faa</code>.<br>
          Genome-only FASTA is accepted, but it requires Bakta and a Bakta database in Step 2.<br>
          ZIP/TAR packages are accepted and will be extracted automatically.
        </div>
      </div>
    </div>
    """))


def display_sharp_panel(chip, title, detail, status="running", percent=None):
    status_color = {
        "running": "#A15C00",
        "ok": "#0A7F37",
        "warn": "#B42318",
        "neutral": "#760032",
    }.get(status, "#760032")

    bar_html = ""

    if percent is not None:
        percent = max(0, min(100, int(percent)))
        bar_html = f"""
        <div class="bar-outer"><div class="bar-inner" style="width:{percent}%;"></div></div>
        <div class="tiny">Progress: <code>{percent}%</code></div>
        """

    display(HTML(f"""
    <style>
      .sharp-panel {{
        --sharp-bg:#97003F;
        --sharp-bg-dark:#760032;
        --sharp-card:#FFF7FA;
        --sharp-card-soft:#F7E4EC;
        --sharp-border:rgba(151,0,63,0.25);
        --sharp-text:#2D0015;
        --sharp-muted:#5E2038;
        font-family:Arial, Helvetica, sans-serif;
      }}

      .sharp-panel .box {{
        border-radius:18px;
        border:1px solid var(--sharp-border);
        background:linear-gradient(135deg,var(--sharp-card),var(--sharp-card-soft));
        padding:16px 18px;
        color:var(--sharp-text);
        margin:12px 0;
      }}

      .sharp-panel .chip {{
        display:inline-block;
        background:var(--sharp-bg);
        color:white;
        border-radius:999px;
        padding:5px 11px;
        font-size:12px;
        font-weight:850;
        font-family:Consolas,monospace;
        margin-bottom:9px;
      }}

      .sharp-panel .title {{
        color:{status_color};
        font-size:18px;
        font-weight:900;
        margin-bottom:4px;
      }}

      .sharp-panel .text {{
        color:var(--sharp-muted);
        font-size:13.5px;
        line-height:1.5;
      }}

      .sharp-panel .tiny {{
        color:var(--sharp-muted);
        font-size:12px;
        margin-top:4px;
      }}

      .sharp-panel .bar-outer {{
        width:100%;
        height:12px;
        border-radius:999px;
        background:#F1DDE6;
        border:1px solid rgba(151,0,63,0.18);
        margin:10px 0 8px 0;
        overflow:hidden;
      }}

      .sharp-panel .bar-inner {{
        height:100%;
        background:linear-gradient(90deg,#97003F,#D8AFC0);
      }}

      .sharp-panel code {{
        font-family:Consolas, Monaco, monospace;
        color:var(--sharp-bg-dark);
      }}

      @media (prefers-color-scheme: dark) {{
        .sharp-panel .box {{
          background:linear-gradient(135deg,#211018,#320017);
          border-color:rgba(216,175,192,0.24);
          color:#F7E4EC;
        }}

        .sharp-panel .text,
        .sharp-panel .tiny {{
          color:#D8AFC0;
        }}

        .sharp-panel code {{
          color:#FFFFFF;
        }}
      }}
    </style>

    <div class="sharp-panel">
      <div class="box">
        <div class="chip">{html.escape(str(chip))}</div>
        <div class="title">{html.escape(str(title))}</div>
        {bar_html}
        <div class="text">{detail}</div>
      </div>
    </div>
    """))

# =====================================================================
# Show cover, then immediately show the upload selector
# =====================================================================

display_sharp_cover()

# =====================================================================
# Optional clean start
# =====================================================================

if RESET_PREVIOUS_INPUTS:
    for path in [
        UPLOADS_DIR,
        NORMALIZED_DIR,
    ]:
        if path.exists():
            shutil.rmtree(path)

        path.mkdir(parents=True, exist_ok=True)

# =====================================================================
# Input helpers
# =====================================================================

def file_sha256(path, block_size=1024 * 1024):
    path = Path(path)
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        while True:
            block = handle.read(block_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def open_text_maybe_gzip(path):
    path = Path(path)

    if path.name.lower().endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8", errors="replace")

    return open(path, "r", encoding="utf-8", errors="replace")


def copy_maybe_decompress(source_path, destination_path):
    source_path = Path(source_path)
    destination_path = Path(destination_path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if source_path.name.lower().endswith(".gz"):
        with gzip.open(source_path, "rb") as src, open(destination_path, "wb") as dst:
            shutil.copyfileobj(src, dst)
    else:
        shutil.copy2(source_path, destination_path)

    return destination_path


def safe_extract_zip(archive_path, extract_dir):
    archive_path = Path(archive_path)
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    extracted = []

    with zipfile.ZipFile(archive_path, "r") as archive:
        for member in archive.infolist():
            member_path = extract_dir / member.filename

            try:
                member_path.resolve().relative_to(extract_dir.resolve())
            except ValueError:
                continue

            archive.extract(member, extract_dir)

    extracted.extend([p for p in extract_dir.rglob("*") if p.is_file()])

    return extracted


def safe_extract_tar(archive_path, extract_dir):
    archive_path = Path(archive_path)
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    extracted = []

    with tarfile.open(archive_path, "r:*") as archive:
        safe_members = []

        for member in archive.getmembers():
            member_path = extract_dir / member.name

            try:
                member_path.resolve().relative_to(extract_dir.resolve())
            except ValueError:
                continue

            safe_members.append(member)

        archive.extractall(extract_dir, members=safe_members)

    extracted.extend([p for p in extract_dir.rglob("*") if p.is_file()])

    return extracted


def extract_archive_if_needed(path):
    path = Path(path)
    extracted_paths = []

    lower = path.name.lower()

    if lower.endswith(".zip"):
        extract_dir = UPLOADS_DIR / f"{path.stem}_extracted"
        extracted_paths.extend(safe_extract_zip(path, extract_dir))

    elif lower.endswith((".tar", ".tar.gz", ".tgz", ".tar.xz")):
        extract_dir = UPLOADS_DIR / f"{path.name.replace('.', '_')}_extracted"
        extracted_paths.extend(safe_extract_tar(path, extract_dir))

    return extracted_paths


def normalize_suffixes(path):
    name = Path(path).name.lower()

    if name.endswith(".gz"):
        name = name[:-3]

    return Path(name).suffix.lower(), name


def is_fasta_file(path):
    suffix, name = normalize_suffixes(path)

    return suffix in {
        ".fa",
        ".fas",
        ".fasta",
        ".fna",
    }


def is_protein_fasta_file(path):
    suffix, name = normalize_suffixes(path)

    if suffix in {
        ".faa",
        ".pep",
        ".protein",
    }:
        return True

    if name.endswith((
        "_proteins.fasta",
        "_protein.fasta",
        "proteins.fasta",
        "protein.fasta",
        "_proteins.fa",
        "_protein.fa",
        "proteins.fa",
        "protein.fa",
    )):
        return True

    return False


def is_annotation_file(path):
    suffix, name = normalize_suffixes(path)

    return suffix in {
        ".gff",
        ".gff3",
        ".gb",
        ".gbk",
        ".gbff",
        ".genbank",
    }


def is_meme_file(path):
    return Path(path).name.lower().endswith(".meme")


def is_hmm_file(path):
    return Path(path).name.lower().endswith(".hmm")


def simple_fasta_stats(path):
    path = Path(path)

    records = 0
    total_bp = 0
    current_len = 0
    first_id = ""

    with open_text_maybe_gzip(path) as handle:
        for line in handle:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if records > 0:
                    total_bp += current_len

                records += 1
                current_len = 0

                if not first_id:
                    first_id = line[1:].split()[0]
            else:
                current_len += len(re.sub(r"[^A-Za-z]", "", line))

    if records > 0:
        total_bp += current_len

    return {
        "records": records,
        "total_bp": total_bp,
        "first_id": first_id,
    }


def choose_genome_fasta(candidates):
    if not candidates:
        return None

    scored = []

    for path in candidates:
        stats = simple_fasta_stats(path)
        name = Path(path).name.lower()

        score = 0
        score += stats["total_bp"]

        if name.endswith((".fna", ".fa", ".fasta", ".fna.gz", ".fa.gz", ".fasta.gz")):
            score += 10_000

        if any(token in name for token in ["genome", "assembly", "contig", "chromosome"]):
            score += 50_000

        if "protein" in name or name.endswith((".faa", ".faa.gz")):
            score -= 1_000_000

        scored.append((score, path, stats))

    scored = sorted(scored, key=lambda item: item[0], reverse=True)

    return scored[0][1]


def choose_annotation_file(candidates):
    if not candidates:
        return None

    def score(path):
        name = Path(path).name.lower()
        value = 0

        if name.endswith((".gff3", ".gff3.gz")):
            value += 100
        elif name.endswith((".gff", ".gff.gz")):
            value += 90
        elif name.endswith((".gbff", ".gbff.gz")):
            value += 70
        elif name.endswith((".gbk", ".gbk.gz")):
            value += 60
        elif name.endswith((".gb", ".gb.gz")):
            value += 50

        if "annotation" in name:
            value += 20

        return value

    return sorted(candidates, key=score, reverse=True)[0]


def choose_protein_fasta(candidates):
    if not candidates:
        return None

    def score(path):
        name = Path(path).name.lower()
        value = 0

        if name.endswith((".faa", ".faa.gz")):
            value += 100

        if "protein" in name or "proteins" in name:
            value += 50

        return value

    return sorted(candidates, key=score, reverse=True)[0]


def copy_optional_internal_resources(candidate_files):
    copied = []

    meme_files = [p for p in candidate_files if is_meme_file(p)]
    hmm_files = [p for p in candidate_files if is_hmm_file(p)]
    py_files = [p for p in candidate_files if Path(p).name == "operon_fig_colab.py"]

    if meme_files and not Path(SHARP_INTERNAL_RESOURCES["heptamer_meme"]).exists():
        shutil.copy2(meme_files[0], SHARP_INTERNAL_RESOURCES["heptamer_meme"])
        copied.append("motif")

    if hmm_files:
        sarp_candidates = [
            p for p in hmm_files
            if any(token in Path(p).name.lower() for token in ["sarp", "btad", "regulator"])
        ]

        domain_candidates = [
            p for p in hmm_files
            if any(token in Path(p).name.lower() for token in ["domain", "models", "all_models", "pfam"])
        ]

        if sarp_candidates and not Path(SHARP_INTERNAL_RESOURCES["sarp_hmm"]).exists():
            shutil.copy2(sarp_candidates[0], SHARP_INTERNAL_RESOURCES["sarp_hmm"])
            copied.append("sarp_hmm")

        if domain_candidates and not Path(SHARP_INTERNAL_RESOURCES["domain_models_hmm"]).exists():
            shutil.copy2(domain_candidates[0], SHARP_INTERNAL_RESOURCES["domain_models_hmm"])
            copied.append("domain_hmm")

    if py_files and not Path(SHARP_INTERNAL_RESOURCES["report_module"]).exists():
        shutil.copy2(py_files[0], SHARP_INTERNAL_RESOURCES["report_module"])
        copied.append("report_module")

    return copied

# =====================================================================
# Upload files
# =====================================================================

uploaded_paths = []

if UPLOAD_FILES_NOW:
    try:
        from google.colab import files

        uploaded = files.upload()

        for filename, content in uploaded.items():
            destination = UPLOADS_DIR / Path(filename).name
            destination.write_bytes(content)
            uploaded_paths.append(destination)

    except Exception:
        uploaded_paths = [p for p in UPLOADS_DIR.rglob("*") if p.is_file()]

else:
    uploaded_paths = [p for p in UPLOADS_DIR.rglob("*") if p.is_file()]

all_candidate_files = list(uploaded_paths)

for path in uploaded_paths:
    all_candidate_files.extend(extract_archive_if_needed(path))

all_candidate_files = [
    p for p in UPLOADS_DIR.rglob("*")
    if p.is_file()
]

if not all_candidate_files:
    display_sharp_panel(
        chip="input",
        title="No input files were uploaded.",
        detail=(
            "Upload at least one genome FASTA file, or an annotated package containing "
            "<code>FASTA + GFF3/GenBank + FAA</code>."
        ),
        status="warn",
        percent=100,
    )

    raise RuntimeError(
        "No input files were uploaded. Upload a genome FASTA or annotated genome package."
    )

# =====================================================================
# Detect input files
# =====================================================================

fasta_candidates = [
    p for p in all_candidate_files
    if is_fasta_file(p)
]

protein_candidates = [
    p for p in all_candidate_files
    if is_protein_fasta_file(p)
]

annotation_candidates = [
    p for p in all_candidate_files
    if is_annotation_file(p)
]

genome_source = choose_genome_fasta(fasta_candidates)
annotation_source = choose_annotation_file(annotation_candidates)
protein_source = choose_protein_fasta(protein_candidates)

if genome_source is None:
    display_sharp_panel(
        chip="input",
        title="Genome FASTA was not detected.",
        detail=(
            "Upload a file with one of these extensions: "
            "<code>.fna</code>, <code>.fa</code>, or <code>.fasta</code>."
        ),
        status="warn",
        percent=100,
    )

    raise RuntimeError(
        "No genome FASTA was detected. Upload a .fna, .fa, or .fasta file."
    )

# =====================================================================
# Normalize selected files
# =====================================================================

GENOME_FASTA = NORMALIZED_DIR / "sharp_input_genome.fna"
GENOME_ANNOTATION = NORMALIZED_DIR / "sharp_input_annotation.gff3"
PROTEIN_FASTA = NORMALIZED_DIR / "sharp_input_proteins.faa"

copy_maybe_decompress(genome_source, GENOME_FASTA)

annotation_format = ""

if annotation_source is not None:
    annotation_suffix, annotation_name = normalize_suffixes(annotation_source)

    if annotation_suffix in {".gb", ".gbk", ".gbff", ".genbank"}:
        GENOME_ANNOTATION = NORMALIZED_DIR / "sharp_input_annotation.gbff"
        annotation_format = "genbank"
    else:
        GENOME_ANNOTATION = NORMALIZED_DIR / "sharp_input_annotation.gff3"
        annotation_format = "gff3"

    copy_maybe_decompress(annotation_source, GENOME_ANNOTATION)

if protein_source is not None:
    copy_maybe_decompress(protein_source, PROTEIN_FASTA)

copied_internal_resources = copy_optional_internal_resources(all_candidate_files)

# =====================================================================
# Resolve workflow mode
# =====================================================================

has_annotation = annotation_source is not None and Path(GENOME_ANNOTATION).exists()
has_proteins = protein_source is not None and Path(PROTEIN_FASTA).exists()

has_demo_resources = (
    Path(SHARP_INTERNAL_RESOURCES["heptamer_meme"]).exists()
    or Path(SHARP_INTERNAL_RESOURCES["sarp_hmm"]).exists()
    or Path(SHARP_INTERNAL_RESOURCES["domain_models_hmm"]).exists()
)

if has_annotation and has_proteins and PREFER_EXISTING_ANNOTATION_WHEN_AVAILABLE:
    workflow = "sharp_existing_annotation"
    annotation_mode = "use_existing_annotation"

    if has_demo_resources:
        input_mode = "demo_package"
    else:
        input_mode = "annotated_genome_package"

    requires_bakta_db = False

elif has_annotation and not has_proteins:
    workflow = "sharp_input_incomplete"
    annotation_mode = "pending_missing_proteins"
    input_mode = "incomplete_annotated_package"
    requires_bakta_db = False

elif has_proteins and not has_annotation:
    workflow = "sharp_input_incomplete"
    annotation_mode = "pending_missing_annotation"
    input_mode = "incomplete_annotated_package"
    requires_bakta_db = False

else:
    workflow = "sharp_auto_from_genome"
    annotation_mode = "auto_from_genome"
    input_mode = "genome_fasta_only"
    requires_bakta_db = bool(REQUIRE_BAKTA_FOR_FASTA_ONLY)

genome_stats = simple_fasta_stats(GENOME_FASTA)
genome_sha256 = file_sha256(GENOME_FASTA)

# =====================================================================
# Shared input globals for downstream cells
# =====================================================================

INPUT_FILES = {
    "genome_source": str(genome_source),
    "annotation_source": str(annotation_source) if annotation_source else "",
    "protein_source": str(protein_source) if protein_source else "",
    "uploaded_files": [str(p) for p in uploaded_paths],
    "all_detected_files": [str(p) for p in all_candidate_files],
    "copied_internal_resources": copied_internal_resources,
}

SHARP_SAMPLE = {
    "project_name": SHARP_PROJECT_NAME,
    "analysis_name": ANALYSIS_NAME,
    "organism_name": DEFAULT_ORGANISM_NAME,
    "strain_name": DEFAULT_STRAIN_NAME,
    "genome_sha256": genome_sha256,
    "genome_hash_short": genome_sha256[:16],
    "input_mode": input_mode,
}

SHARP_INPUT_CONTRACT = {
    "status": "ready" if workflow != "sharp_input_incomplete" else "incomplete",
    "workflow": workflow,
    "mode": input_mode,
    "annotation_mode": annotation_mode,
    "requires_bakta_db": requires_bakta_db,
    "genome_fasta": str(GENOME_FASTA),
    "annotation_file": str(GENOME_ANNOTATION) if has_annotation else "",
    "annotation_format": annotation_format,
    "protein_fasta": str(PROTEIN_FASTA) if has_proteins else "",
    "genome_records": genome_stats["records"],
    "genome_total_bp": genome_stats["total_bp"],
    "genome_first_id": genome_stats["first_id"],
    "genome_sha256": genome_sha256,
    "uploaded_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}

cfg = {
    "project": SHARP_PROJECT_NAME,
    "analysis_name": ANALYSIS_NAME,
    "paths": SHARP_PATHS,
    "internal_resources": SHARP_INTERNAL_RESOURCES,
    "input_policy": SHARP_INPUT_POLICY,
    "input_contract": SHARP_INPUT_CONTRACT,
}

# =====================================================================
# Notebook context files
# =====================================================================

SHARP_NOTEBOOK_CONTEXT = {
    "project": SHARP_PROJECT_NAME,
    "title": SHARP_PROJECT_TITLE,
    "full_name": SHARP_PROJECT_FULL_NAME,
    "subtitle": SHARP_PROJECT_SUBTITLE,
    "team": SHARP_TEAM,
    "version": SHARP_VERSION,
    "initialized_at": SHARP_INITIALIZED_AT,
    "analysis_name": ANALYSIS_NAME,
    "in_colab": IN_COLAB,
    "python_version": sys.version.split()[0],
    "platform": platform.platform(),
    "workflow": workflow,
    "annotation_mode": annotation_mode,
    "input_mode": input_mode,
    "input_policy": SHARP_INPUT_POLICY,
    "input_contract": SHARP_INPUT_CONTRACT,
    "require_bakta_for_fasta_only": REQUIRE_BAKTA_FOR_FASTA_ONLY,
    "allow_prodigal_fallback_for_debug": ALLOW_PRODIGAL_FALLBACK_FOR_DEBUG,
    "prefer_existing_annotation_when_available": PREFER_EXISTING_ANNOTATION_WHEN_AVAILABLE,
    "paths": SHARP_PATHS,
    "internal_resources": SHARP_INTERNAL_RESOURCES,
}

SHARP_CONTEXT_FILE = CONFIG_DIR / "sharp_notebook_context.json"
INPUT_CONTRACT_FILE = CONFIG_DIR / "sharp_input_contract.json"
INPUT_FILES_FILE = CONFIG_DIR / "sharp_input_files.json"

write_json(SHARP_CONTEXT_FILE, SHARP_NOTEBOOK_CONTEXT)
write_json(INPUT_CONTRACT_FILE, SHARP_INPUT_CONTRACT)
write_json(INPUT_FILES_FILE, INPUT_FILES)

# =====================================================================
# Final public input detection panel
# =====================================================================

if workflow == "sharp_existing_annotation":
    public_status = "Input package ready."
    public_detail = (
        "Annotated genome package detected. "
        "Step 2 can skip Bakta DB."
    )
    public_class = "ok"

elif workflow == "sharp_auto_from_genome":
    public_status = "Genome FASTA ready."
    public_detail = (
        "Genome-only input detected. "
        "Step 2 will prepare the annotation backend."
    )
    public_class = "ok"

else:
    public_status = "Input needs attention."
    public_detail = (
        "Genome FASTA was detected, but the annotation package is incomplete."
    )
    public_class = "warn"

resource_note = ""

if copied_internal_resources:
    resource_note = (
        "<br>Internal resources detected: "
        + "<code>"
        + html.escape(", ".join(copied_internal_resources))
        + "</code>."
    )

display_sharp_panel(
    chip="detected input",
    title=public_status,
    detail=(
        f"{html.escape(public_detail)}<br>"
        f"Input mode: <code>{html.escape(input_mode)}</code><br>"
        f"Annotation mode: <code>{html.escape(annotation_mode)}</code><br>"
        f"Bakta DB required in Step 2: <code>{requires_bakta_db}</code><br>"
        f"Genome size: <code>{genome_stats['total_bp']:,} bp</code><br>"
        f"Genome records: <code>{genome_stats['records']}</code>"
        f"{resource_note}"
    ),
    status=public_class,
    percent=100,
)

print("S(H)ARP initialized and input loaded.")
print("workflow:", workflow)
print("annotation_mode:", annotation_mode)
print("input_mode:", input_mode)
print("requires_bakta_db:", requires_bakta_db)
print("genome_fasta:", GENOME_FASTA)
print("annotation_file:", str(GENOME_ANNOTATION) if has_annotation else "")
print("protein_fasta:", str(PROTEIN_FASTA) if has_proteins else "")
print("genome_records:", genome_stats["records"])
print("genome_bp:", genome_stats["total_bp"])
print("genome_sha256:", genome_sha256)
print("copied_internal_resources:", copied_internal_resources)
print("next step: run Step 2 — Prepare S(H)ARP environment.")

In [ ]:
# @title S(H)ARP — Prepare environment { display-mode: "form" }

# =====================================================================
# Imports
# =====================================================================

from pathlib import Path
from IPython.display import display, HTML
import hashlib
import html
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tarfile
import time
import urllib.request

# =====================================================================
# Internal public-policy constants
# =====================================================================

# Public interface policy:
# - Annotated input package: use provided annotation.
# - Genome-only FASTA: use cached Bakta DB light.
# - A protein FASTA without GFF3/GBFF does not define an annotated package.
# - Prodigal fallback is disabled in this validation phase.
# - The UI shows a single live status panel.
# - Bakta compatibility patching is integrated here to keep the notebook flow clean.
# - AMRFinderPlus internal DB preparation is integrated here because Bakta requires it.
# - MEME Suite/FIMO is installed here because motif scanning is part of the S(H)ARP flow.

INSTALL_REQUIRED_PYTHON_PACKAGES = True
INSTALL_REQUIRED_SYSTEM_TOOLS = True

AUTO_INSTALL_BAKTA_FOR_GENOME_ONLY = True
AUTO_INSTALL_MEME_SUITE_FOR_FIMO = True
REQUIRE_FIMO_FOR_MOTIF_STEP = True

DOWNLOAD_BAKTA_DB_FOR_GENOME_ONLY = True
FORCE_REFRESH_BAKTA_DB_ARCHIVE = False
FORCE_REEXTRACT_BAKTA_DB = False
FORCE_AMRFINDER_UPDATE = False
USE_ZENODO_FALLBACK_IF_GITHUB_FAILS = True

INSTALL_BUNDLED_SHARP_RESOURCES = True
OVERWRITE_BUNDLED_RESOURCES = False

GENOME_ONLY_ANNOTATION_ENGINE = "Bakta cached light DB"
USE_PRODIGAL_FOR_GENOME_ONLY = False
USE_BAKTA_FOR_GENOME_ONLY = True
REQUIRE_ANNOTATED_PACKAGE_FOR_GENOME_ONLY = False

PREFER_EXISTING_ANNOTATION_WHEN_AVAILABLE = bool(
    globals().get("PREFER_EXISTING_ANNOTATION_WHEN_AVAILABLE", True)
)

# =====================================================================
# Single-panel UI
# =====================================================================

SHARP_COLORS = {
    "background": "#97003F",
    "dark": "#760032",
    "deep": "#4D0022",
    "pink": "#D8AFC0",
    "soft": "#F1DDE6",
    "white": "#FFFFFF",
}

SHARP_FONT_STACK = (
    "Bodoni 72, Bodoni MT, Didot, Georgia, Times New Roman, serif"
)

SHARP_STATUS_HANDLE = None
SHARP_STATUS_HISTORY = []

def safe_html(value):
    """Escape a value for HTML rendering."""
    return html.escape(str(value), quote=True)

def compact_path(value, max_len=95):
    """Shorten long paths for compact UI display."""
    value = str(value)
    if len(value) <= max_len:
        return value
    return value[:42] + "..." + value[-42:]

def unique_list(values):
    """Return unique values preserving order."""
    output = []
    seen = set()

    for value in values:
        if value not in seen:
            output.append(value)
            seen.add(value)

    return output

def render_status_panel(
    phase,
    message,
    progress,
    rows=None,
    warnings=None,
    blockers=None,
    done=False,
):
    """Render the single live S(H)ARP setup panel."""
    rows = rows or []
    warnings = unique_list(warnings or [])
    blockers = unique_list(blockers or [])

    progress_value = max(0, min(100, int(progress)))

    status_badge = "READY" if done and not blockers else ("BLOCKED" if blockers else "RUNNING")
    status_background = SHARP_COLORS["pink"] if done and not blockers else ("#FFD6D6" if blockers else SHARP_COLORS["soft"])
    status_color = SHARP_COLORS["deep"]

    history_items = ""
    if SHARP_STATUS_HISTORY:
        history_items = "".join(
            f"""
            <div style="display:flex; gap:8px; align-items:center; padding:2px 0;">
              <span style="color:{SHARP_COLORS['pink']};">●</span>
              <span>{safe_html(item)}</span>
            </div>
            """
            for item in SHARP_STATUS_HISTORY[-9:]
        )

    rows_html = ""
    if rows:
        row_items = []
        for key, value in rows:
            row_items.append(
                f"""
                <div style="display:grid; grid-template-columns:210px 1fr; gap:12px; padding:6px 0; border-bottom:1px solid rgba(255,255,255,0.12);">
                  <div style="font-weight:700; color:{SHARP_COLORS['soft']};">{safe_html(key)}</div>
                  <div style="color:{SHARP_COLORS['white']};"><code>{safe_html(compact_path(value))}</code></div>
                </div>
                """
            )

        rows_html = f"""
        <div style="margin-top:14px; font-size:13px;">
          {''.join(row_items)}
        </div>
        """

    warning_html = ""
    if warnings:
        warning_items = "".join(f"<li>{safe_html(item)}</li>" for item in warnings[:8])
        extra = ""
        if len(warnings) > 8:
            extra = f"<li>{safe_html(str(len(warnings) - 8) + ' additional warning(s) hidden for compact display.')}</li>"

        warning_html = f"""
        <div style="margin-top:14px; padding:10px 12px; border-radius:12px; background:rgba(255,255,255,0.12);">
          <div style="font-weight:800; color:{SHARP_COLORS['soft']}; margin-bottom:6px;">Pending optional steps</div>
          <ul style="margin:0; padding-left:20px; line-height:1.45;">{warning_items}{extra}</ul>
        </div>
        """

    blocker_html = ""
    if blockers:
        blocker_items = "".join(f"<li>{safe_html(item)}</li>" for item in blockers)

        blocker_html = f"""
        <div style="margin-top:14px; padding:10px 12px; border-radius:12px; background:#FFE4E4; color:#4D0022;">
          <div style="font-weight:900; margin-bottom:6px;">Blocking setup issues</div>
          <ul style="margin:0; padding-left:20px; line-height:1.45;">{blocker_items}</ul>
        </div>
        """

    return HTML(f"""
    <div style="
      background:linear-gradient(135deg,{SHARP_COLORS['background']},{SHARP_COLORS['dark']});
      color:{SHARP_COLORS['white']};
      padding:20px 22px;
      border-radius:18px;
      border:1px solid rgba(255,255,255,0.22);
      box-shadow:0 10px 32px rgba(0,0,0,0.20);
      margin:12px 0;
      font-family:Inter,Arial,sans-serif;
    ">
      <div style="display:flex; justify-content:space-between; gap:16px; align-items:flex-start;">
        <div>
          <div style="font-family:{SHARP_FONT_STACK}; font-size:30px; letter-spacing:0.5px;">
            S(H)ARP
          </div>
          <div style="font-size:12px; color:{SHARP_COLORS['soft']}; text-transform:uppercase; letter-spacing:1.6px; margin-top:2px;">
            Environment setup
          </div>
        </div>
        <div style="
          background:{status_background};
          color:{status_color};
          border-radius:999px;
          padding:7px 12px;
          font-weight:900;
          font-size:12px;
          letter-spacing:1px;
        ">
          {safe_html(status_badge)}
        </div>
      </div>

      <div style="font-size:21px; font-weight:900; margin-top:14px;">
        {safe_html(phase)}
      </div>

      <div style="margin-top:8px; font-size:14px; line-height:1.45;">
        {safe_html(message)}
      </div>

      <div style="margin-top:14px;">
        <div style="font-size:12px; color:{SHARP_COLORS['soft']}; margin-bottom:6px;">
          Progress: <code>{progress_value}%</code>
        </div>
        <div style="height:10px; background:{SHARP_COLORS['deep']}; border-radius:999px; overflow:hidden;">
          <div style="height:10px; width:{progress_value}%; background:{SHARP_COLORS['pink']}; transition:width 0.25s ease;"></div>
        </div>
      </div>

      <div style="margin-top:14px; font-size:13px; color:{SHARP_COLORS['soft']};">
        {history_items}
      </div>

      {rows_html}
      {warning_html}
      {blocker_html}
    </div>
    """)

def update_status(
    phase,
    message,
    progress,
    rows=None,
    warnings=None,
    blockers=None,
    done=False,
    history_label=None,
):
    """Update the single live S(H)ARP status panel."""
    global SHARP_STATUS_HANDLE

    if history_label:
        if not SHARP_STATUS_HISTORY or SHARP_STATUS_HISTORY[-1] != history_label:
            SHARP_STATUS_HISTORY.append(history_label)

    panel = render_status_panel(
        phase=phase,
        message=message,
        progress=progress,
        rows=rows,
        warnings=warnings,
        blockers=blockers,
        done=done,
    )

    if SHARP_STATUS_HANDLE is None:
        SHARP_STATUS_HANDLE = display(panel, display_id=True)
    else:
        SHARP_STATUS_HANDLE.update(panel)

update_status(
    phase="Preparing environment",
    message="S(H)ARP is initializing the automatic public workflow.",
    progress=3,
    history_label="Setup started",
)

# =====================================================================
# Runtime and path setup
# =====================================================================

def is_colab_runtime():
    """Return True when running inside Google Colab."""
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

IS_COLAB = is_colab_runtime()
RUNTIME_NAME = "colab" if IS_COLAB else "local_or_hpc"
HOSTNAME = platform.node()

PROJECT_DIR = Path(globals().get("PROJECT_DIR", "/content/sharp_igem_usp_brazil_2026")).resolve()
DATA_DIR = Path(globals().get("DATA_DIR", PROJECT_DIR / "data")).resolve()
INPUT_DIR = Path(globals().get("INPUT_DIR", DATA_DIR / "input")).resolve()
NORMALIZED_DIR = Path(globals().get("NORMALIZED_DIR", DATA_DIR / "normalized")).resolve()
CONFIG_DIR = Path(globals().get("CONFIG_DIR", PROJECT_DIR / "config")).resolve()
DATABASES_DIR = Path(globals().get("DATABASES_DIR", PROJECT_DIR / "databases")).resolve()
RESULTS_DIR = Path(globals().get("RESULTS_DIR", PROJECT_DIR / "results")).resolve()

RUN_NAME = str(globals().get("RUN_NAME", "sharp_run_01"))
RUN_DIR = Path(globals().get("RUN_DIR", RESULTS_DIR / RUN_NAME)).resolve()
LOG_DIR = Path(globals().get("LOG_DIR", RUN_DIR / "logs")).resolve()
TABLE_DIR = Path(globals().get("TABLE_DIR", RUN_DIR / "tables")).resolve()
REPORT_DIR = Path(globals().get("REPORT_DIR", RUN_DIR / "report")).resolve()

for directory in [
    PROJECT_DIR,
    DATA_DIR,
    INPUT_DIR,
    NORMALIZED_DIR,
    CONFIG_DIR,
    DATABASES_DIR,
    RESULTS_DIR,
    RUN_DIR,
    LOG_DIR,
    TABLE_DIR,
    REPORT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SHARP_CONTEXT_FILE = Path(
    globals().get("SHARP_CONTEXT_FILE", CONFIG_DIR / "sharp_notebook_context.json")
)
INPUT_CONTRACT_FILE = Path(
    globals().get("INPUT_CONTRACT_FILE", CONFIG_DIR / "sharp_input_contract.json")
)
INPUT_FILES_FILE = Path(
    globals().get("INPUT_FILES_FILE", CONFIG_DIR / "sharp_input_files.json")
)

update_status(
    phase="Runtime detected",
    message="Runtime paths were initialized.",
    progress=8,
    rows=[
        ("Runtime", RUNTIME_NAME),
        ("Project directory", PROJECT_DIR),
        ("Run directory", RUN_DIR),
    ],
    history_label="Runtime paths ready",
)

# =====================================================================
# General utility functions
# =====================================================================

def read_json_if_exists(path):
    """Read a JSON file if it exists."""
    path = Path(path)
    if not path.exists():
        return None

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def write_json(path, data):
    """Write formatted JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

def command_path(command):
    """Resolve a command from PATH."""
    return shutil.which(command) or ""

def run_cmd(command, label=None, cwd=None, env=None, check=True, capture=True, timeout=None):
    """Run a command and capture logs."""
    if isinstance(command, str):
        printable = command
        shell = True
    else:
        printable = " ".join(str(x) for x in command)
        shell = False

    log_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", label or "command").strip("_")
    stdout_path = LOG_DIR / f"{log_name}.stdout.txt"
    stderr_path = LOG_DIR / f"{log_name}.stderr.txt"

    started = time.time()

    process = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        shell=shell,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
        timeout=timeout,
    )

    elapsed = time.time() - started

    stdout_text = process.stdout or ""
    stderr_text = process.stderr or ""

    if capture:
        stdout_path.write_text(stdout_text, encoding="utf-8")
        stderr_path.write_text(stderr_text, encoding="utf-8")

    record = {
        "label": label or "",
        "command": printable,
        "returncode": process.returncode,
        "elapsed_seconds": round(elapsed, 3),
        "stdout_path": str(stdout_path) if capture else "",
        "stderr_path": str(stderr_path) if capture else "",
    }

    if check and process.returncode != 0:
        message = [
            f"Command failed: {label or printable}",
            f"returncode={process.returncode}",
            f"stdout={stdout_path}",
            f"stderr={stderr_path}",
        ]

        if stderr_text:
            message.append(stderr_text[-4000:])

        raise RuntimeError("\n".join(message))

    return record

def pip_install(packages):
    """Install Python packages with pip."""
    if not packages:
        return None

    command = [sys.executable, "-m", "pip", "install", "-q"] + list(packages)

    return run_cmd(
        command,
        label="pip_install_" + "_".join(p.split("==")[0] for p in packages),
        check=True,
        capture=True,
        timeout=1800,
    )

def apt_install(packages):
    """Install apt packages in Colab/Debian-like environments."""
    if not packages:
        return None

    if not command_path("apt-get"):
        return {
            "label": "apt_install_skipped",
            "returncode": None,
            "reason": "apt-get not available",
        }

    run_cmd(
        "apt-get update -qq",
        label="apt_get_update",
        check=True,
        capture=True,
        timeout=900,
    )

    command = (
        "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq "
        + " ".join(packages)
    )

    return run_cmd(
        command,
        label="apt_install_" + "_".join(packages[:8]),
        check=False,
        capture=True,
        timeout=2400,
    )

def url_download(url, output_path, label=None, fallback_urls=None):
    """Download a URL with curl when possible, using fallback URLs if needed."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    urls = [url] + list(fallback_urls or [])
    last_error = None

    for idx, candidate_url in enumerate(urls, start=1):
        try:
            if command_path("curl"):
                run_cmd(
                    [
                        "curl",
                        "-L",
                        "--fail",
                        "--retry",
                        "5",
                        "--retry-delay",
                        "10",
                        candidate_url,
                        "-o",
                        str(output_path),
                    ],
                    label=label or f"download_{idx}",
                    check=True,
                    capture=True,
                    timeout=None,
                )
            else:
                with urllib.request.urlopen(candidate_url, timeout=60) as response:
                    with output_path.open("wb") as handle:
                        shutil.copyfileobj(response, handle)

            if output_path.exists() and output_path.stat().st_size > 0:
                return {
                    "ok": True,
                    "url": candidate_url,
                    "path": str(output_path),
                    "size_bytes": output_path.stat().st_size,
                }

            last_error = f"Downloaded file is empty: {candidate_url}"

        except Exception as exc:
            last_error = str(exc)

    return {
        "ok": False,
        "url": url,
        "path": str(output_path),
        "error": last_error or "download failed",
    }

def load_remote_json(url, fallback=None, output_path=None):
    """Download and parse a remote JSON file."""
    fallback = fallback or {}
    output_path = Path(output_path) if output_path else None

    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            text = response.read().decode("utf-8")

        data = json.loads(text)

        if output_path:
            output_path.parent.mkdir(parents=True, exist_ok=True)
            output_path.write_text(json.dumps(data, indent=2), encoding="utf-8")

        return data, None

    except Exception as exc:
        if output_path:
            output_path.parent.mkdir(parents=True, exist_ok=True)
            output_path.write_text(json.dumps(fallback, indent=2), encoding="utf-8")

        return fallback, str(exc)

def file_md5(path, chunk_size=1024 * 1024 * 8):
    """Compute MD5 checksum."""
    checksum = hashlib.md5()

    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            checksum.update(chunk)

    return checksum.hexdigest()

def file_sha256(path, chunk_size=1024 * 1024 * 8):
    """Compute SHA256 checksum."""
    checksum = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            checksum.update(chunk)

    return checksum.hexdigest()

def safe_extract_tar_xz(archive_path, extract_parent):
    """Extract a .tar.xz archive safely."""
    archive_path = Path(archive_path)
    extract_parent = Path(extract_parent)
    extract_parent.mkdir(parents=True, exist_ok=True)
    extract_root = extract_parent.resolve()

    with tarfile.open(archive_path, "r:xz") as tar:
        members = tar.getmembers()

        for member in members:
            member_path = (extract_parent / member.name).resolve()
            if not str(member_path).startswith(str(extract_root)):
                raise RuntimeError(f"Unsafe tar member path: {member.name}")

        try:
            tar.extractall(extract_parent, filter="data")
        except TypeError:
            tar.extractall(extract_parent)

def chmod_readable_tree(path):
    """Make an extracted directory tree readable by the runtime user."""
    path = Path(path)

    if not path.exists():
        return

    for item in path.rglob("*"):
        try:
            if item.is_dir():
                item.chmod(0o755)
            elif item.is_file():
                item.chmod(0o644)
        except Exception:
            pass

    try:
        path.chmod(0o755)
    except Exception:
        pass

def validate_bakta_db(db_path):
    """Validate required Bakta DB light files."""
    db_path = Path(db_path)

    required_files = [
        "version.json",
        "antifam.h3f",
        "antifam.h3i",
        "antifam.h3m",
        "antifam.h3p",
        "pfam.h3f",
        "pfam.h3i",
        "pfam.h3m",
        "pfam.h3p",
        "amrfinderplus-db",
    ]

    missing_or_unreadable = []

    for name in required_files:
        candidate = db_path / name

        if not candidate.exists() or not os.access(candidate, os.R_OK):
            missing_or_unreadable.append(name)

    return missing_or_unreadable

def extract_bakta_db_light(archive_path, extract_parent, force=False):
    """Extract Bakta DB light and validate required files."""
    archive_path = Path(archive_path)
    extract_parent = Path(extract_parent)
    expected_db_dir = extract_parent / "db-light"
    version_json = expected_db_dir / "version.json"

    if force and expected_db_dir.exists():
        shutil.rmtree(expected_db_dir)

    if expected_db_dir.exists():
        chmod_readable_tree(expected_db_dir)
        missing_or_unreadable = validate_bakta_db(expected_db_dir)

        if not missing_or_unreadable and version_json.exists():
            return {
                "ok": True,
                "db_path": str(expected_db_dir),
                "version_json": str(version_json),
                "extracted": False,
                "missing_or_unreadable": [],
            }

        shutil.rmtree(expected_db_dir)

    safe_extract_tar_xz(archive_path, extract_parent)

    if not version_json.exists():
        candidates = list(extract_parent.glob("*/version.json"))
        if candidates:
            version_json = candidates[0]
            expected_db_dir = version_json.parent

    if not version_json.exists():
        raise RuntimeError(
            f"Bakta DB extraction completed, but version.json was not found under {extract_parent}"
        )

    chmod_readable_tree(expected_db_dir)

    missing_or_unreadable = validate_bakta_db(expected_db_dir)

    if missing_or_unreadable:
        raise RuntimeError(
            "Extracted Bakta DB has missing or unreadable files: "
            + ", ".join(missing_or_unreadable)
        )

    return {
        "ok": True,
        "db_path": str(expected_db_dir),
        "version_json": str(version_json),
        "extracted": True,
        "missing_or_unreadable": [],
    }

def status_dict(command, policy):
    """Build a tool status dictionary."""
    path = command_path(command)
    return {
        "available": bool(path),
        "path": path,
        "policy": policy,
    }

def add_to_path(path):
    """Prepend a path to PATH if it exists."""
    path = Path(path)

    if path.exists():
        current = os.environ.get("PATH", "")
        parts = current.split(os.pathsep)

        if str(path) not in parts:
            os.environ["PATH"] = str(path) + os.pathsep + current

def run_command(command, check=True):
    """Run a command and return the completed process."""
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            "Command failed:\n"
            + " ".join(map(str, command))
            + "\n\nSTDOUT:\n"
            + result.stdout
            + "\n\nSTDERR:\n"
            + result.stderr
        )

    return result

def command_version(command, version_args=None):
    """Return a compact command version string."""
    if not command:
        return ""

    version_args = version_args or ["--version"]

    try:
        result = subprocess.run(
            [command] + list(version_args),
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=False,
            timeout=60,
        )

        output = (result.stdout or "").strip().splitlines()
        if output:
            return output[0].strip()

        return f"returncode={result.returncode}"

    except Exception as exc:
        return f"version_check_failed: {exc}"

# =====================================================================
# Read Cell 1 input contract
# =====================================================================

context = read_json_if_exists(SHARP_CONTEXT_FILE) or {}
input_contract = read_json_if_exists(INPUT_CONTRACT_FILE) or {}
input_files = read_json_if_exists(INPUT_FILES_FILE) or {}

GENOME_FASTA = Path(
    globals().get(
        "GENOME_FASTA",
        input_contract.get("genome_fasta", input_files.get("genome_fasta", "")),
    )
) if (
    globals().get("GENOME_FASTA")
    or input_contract.get("genome_fasta")
    or input_files.get("genome_fasta")
) else None

ANNOTATION_FILE = Path(
    globals().get(
        "ANNOTATION_FILE",
        input_contract.get("annotation_file", input_files.get("annotation_file", "")),
    )
) if (
    globals().get("ANNOTATION_FILE")
    or input_contract.get("annotation_file")
    or input_files.get("annotation_file")
) else None

PROTEIN_FASTA = Path(
    globals().get(
        "PROTEIN_FASTA",
        input_contract.get("protein_fasta", input_files.get("protein_fasta", "")),
    )
) if (
    globals().get("PROTEIN_FASTA")
    or input_contract.get("protein_fasta")
    or input_files.get("protein_fasta")
) else None

CDS_FASTA = Path(
    globals().get(
        "CDS_FASTA",
        input_contract.get("cds_fasta", input_files.get("cds_fasta", "")),
    )
) if (
    globals().get("CDS_FASTA")
    or input_contract.get("cds_fasta")
    or input_files.get("cds_fasta")
) else None

has_genome = bool(GENOME_FASTA and GENOME_FASTA.exists())
has_annotation = bool(ANNOTATION_FILE and ANNOTATION_FILE.exists())
has_proteins = bool(PROTEIN_FASTA and PROTEIN_FASTA.exists())
has_cds = bool(CDS_FASTA and CDS_FASTA.exists())

if not has_genome:
    raise RuntimeError(
        "No genome FASTA was found from Cell 1. Run the input/upload cell before running setup."
    )

# =====================================================================
# Automatic workflow decision
# =====================================================================

dependency_warnings = []
resource_warnings = []
dependency_blockers = []

if has_annotation and has_proteins and PREFER_EXISTING_ANNOTATION_WHEN_AVAILABLE:
    workflow = "sharp_existing_annotation"
    input_mode = "annotated_genome_package"
    annotation_mode = "use_existing_annotation"
    annotation_backend = "existing_annotation"
    annotation_action = "use_existing_annotation"
    requires_bakta_db = False
    requires_bakta_execution = False

elif has_annotation and not has_proteins:
    workflow = "sharp_input_incomplete"
    input_mode = "incomplete_annotated_package"
    annotation_mode = "pending_missing_proteins"
    annotation_backend = "incomplete_annotated_package"
    annotation_action = "upload_protein_fasta"
    requires_bakta_db = False
    requires_bakta_execution = False
    dependency_blockers.append(
        "Annotation GFF3/GBFF was provided, but protein FASTA was not found. "
        "Upload FAA together with the annotation, or upload only the genome FASTA and let S(H)ARP run Bakta."
    )

else:
    workflow = "sharp_auto_from_genome"
    input_mode = "genome_fasta_only"
    annotation_mode = "auto_from_genome"
    annotation_backend = "bakta_cached_light_db"
    annotation_action = "download_extract_bakta_db_then_run_bakta_in_core"
    requires_bakta_db = True
    requires_bakta_execution = True

    if has_proteins and not has_annotation:
        resource_warnings.append(
            "Protein FASTA was uploaded without GFF3/GBFF annotation. "
            "S(H)ARP will treat this as genome-only input and run Bakta automatically."
        )

update_status(
    phase="Input detected",
    message="S(H)ARP selected the analysis path from the uploaded files.",
    progress=15,
    rows=[
        ("Genome FASTA", GENOME_FASTA),
        ("Annotation file", ANNOTATION_FILE if has_annotation else "not provided"),
        ("Protein FASTA", PROTEIN_FASTA if has_proteins else "not provided"),
        ("CDS FASTA", CDS_FASTA if has_cds else "not provided"),
        ("Input mode", input_mode),
        ("Annotation backend", annotation_backend),
    ],
    warnings=resource_warnings,
    blockers=dependency_blockers,
    history_label="Input classified",
)

# =====================================================================
# Install Python dependencies and system tools
# =====================================================================

update_status(
    phase="Installing dependencies",
    message="Installing lightweight Python packages and system tools.",
    progress=25,
    rows=[
        ("Input mode", input_mode),
        ("Annotation backend", annotation_backend),
    ],
    warnings=resource_warnings,
    blockers=dependency_blockers,
    history_label="Dependency installation started",
)

if INSTALL_REQUIRED_PYTHON_PACKAGES:
    try:
        pip_install(["numpy", "pandas", "biopython", "requests", "tqdm"])
    except Exception as exc:
        dependency_blockers.append(f"Python dependency installation failed: {exc}")

if INSTALL_REQUIRED_SYSTEM_TOOLS and IS_COLAB:
    apt_packages = [
        "hmmer",
        "curl",
        "xz-utils",
        "bzip2",
    ]

    apt_result = apt_install(apt_packages)

    if apt_result and apt_result.get("returncode") not in (0, None):
        dependency_warnings.append(
            "Some apt packages may not have installed cleanly. Check setup logs in the run directory."
        )

elif INSTALL_REQUIRED_SYSTEM_TOOLS and not IS_COLAB:
    dependency_warnings.append(
        "System package auto-install was skipped outside Colab. Existing PATH tools will be used."
    )

# =====================================================================
# Local micromamba tool environments
# =====================================================================

CONDA_ENV_DIR = PROJECT_DIR / "envs" / "bakta"
CONDA_ENV_BIN = CONDA_ENV_DIR / "bin"

MEME_ENV_DIR = PROJECT_DIR / "envs" / "meme"
MEME_ENV_BIN = MEME_ENV_DIR / "bin"

MICROMAMBA_ROOT = PROJECT_DIR / "envs" / "micromamba"
MICROMAMBA_BIN = MICROMAMBA_ROOT / "bin" / "micromamba"

add_to_path(CONDA_ENV_BIN)
add_to_path(MEME_ENV_BIN)

def install_micromamba_if_needed():
    """Install micromamba locally inside the project directory."""
    if MICROMAMBA_BIN.exists():
        return str(MICROMAMBA_BIN)

    MICROMAMBA_ROOT.mkdir(parents=True, exist_ok=True)

    install_command = (
        f'mkdir -p "{MICROMAMBA_ROOT}" && '
        f'curl -L --fail "https://micro.mamba.pm/api/micromamba/linux-64/latest" | '
        f'tar -xvj -C "{MICROMAMBA_ROOT}" bin/micromamba'
    )

    run_cmd(
        install_command,
        label="install_micromamba",
        check=True,
        capture=True,
        timeout=900,
    )

    if not MICROMAMBA_BIN.exists():
        raise RuntimeError(f"micromamba was not installed at {MICROMAMBA_BIN}")

    return str(MICROMAMBA_BIN)

def install_bakta_conda_env_if_needed():
    """Install Bakta in a local micromamba environment."""
    bakta_candidate = CONDA_ENV_BIN / "bakta"

    if bakta_candidate.exists():
        add_to_path(CONDA_ENV_BIN)
        return str(bakta_candidate)

    micromamba = install_micromamba_if_needed()

    command = [
        micromamba,
        "create",
        "-y",
        "-p",
        str(CONDA_ENV_DIR),
        "-c",
        "conda-forge",
        "-c",
        "bioconda",
        "bakta=1.11.3",
    ]

    env = os.environ.copy()
    env["MAMBA_ROOT_PREFIX"] = str(MICROMAMBA_ROOT)

    run_cmd(
        command,
        label="create_bakta_micromamba_env",
        env=env,
        check=True,
        capture=True,
        timeout=3600,
    )

    add_to_path(CONDA_ENV_BIN)

    if not bakta_candidate.exists():
        raise RuntimeError(f"Bakta was not installed at {bakta_candidate}")

    return str(bakta_candidate)

def install_meme_suite_env_if_needed():
    """Install MEME Suite in a local micromamba environment for FIMO."""
    fimo_candidate = MEME_ENV_BIN / "fimo"

    if fimo_candidate.exists():
        add_to_path(MEME_ENV_BIN)
        return str(fimo_candidate)

    micromamba = install_micromamba_if_needed()

    command = [
        micromamba,
        "create",
        "-y",
        "-p",
        str(MEME_ENV_DIR),
        "-c",
        "conda-forge",
        "-c",
        "bioconda",
        "meme",
    ]

    env = os.environ.copy()
    env["MAMBA_ROOT_PREFIX"] = str(MICROMAMBA_ROOT)

    run_cmd(
        command,
        label="create_meme_suite_micromamba_env",
        env=env,
        check=True,
        capture=True,
        timeout=3600,
    )

    add_to_path(MEME_ENV_BIN)

    if not fimo_candidate.exists():
        raise RuntimeError(f"FIMO was not installed at {fimo_candidate}")

    return str(fimo_candidate)

# =====================================================================
# Bakta and FIMO installation
# =====================================================================

if requires_bakta_execution and AUTO_INSTALL_BAKTA_FOR_GENOME_ONLY:
    if not command_path("bakta"):
        update_status(
            phase="Installing Bakta",
            message="Genome-only FASTA requires Bakta. S(H)ARP is installing it in a local notebook environment.",
            progress=35,
            rows=[
                ("Environment", CONDA_ENV_DIR),
                ("Input mode", input_mode),
            ],
            warnings=resource_warnings + dependency_warnings,
            blockers=dependency_blockers,
            history_label="Bakta installation started",
        )

        if IS_COLAB:
            try:
                install_bakta_conda_env_if_needed()
            except Exception as exc:
                dependency_blockers.append(f"Bakta installation failed: {exc}")
        else:
            dependency_warnings.append(
                "Bakta auto-install is only enabled for Colab. Existing PATH tools will be used outside Colab."
            )

add_to_path(CONDA_ENV_BIN)

if REQUIRE_FIMO_FOR_MOTIF_STEP and AUTO_INSTALL_MEME_SUITE_FOR_FIMO:
    if not command_path("fimo"):
        update_status(
            phase="Installing MEME Suite / FIMO",
            message="FIMO is required for S(H)ARP motif scanning. S(H)ARP is installing MEME Suite in a local notebook environment.",
            progress=39,
            rows=[
                ("Environment", MEME_ENV_DIR),
                ("Tool", "fimo"),
                ("Conda package", "bioconda::meme"),
            ],
            warnings=resource_warnings + dependency_warnings,
            blockers=dependency_blockers,
            history_label="MEME Suite/FIMO installation started",
        )

        if IS_COLAB:
            try:
                install_meme_suite_env_if_needed()
            except Exception as exc:
                dependency_blockers.append(f"MEME Suite/FIMO installation failed: {exc}")
        else:
            dependency_warnings.append(
                "MEME Suite/FIMO auto-install is only enabled for Colab. Existing PATH tools will be used outside Colab."
            )

add_to_path(MEME_ENV_BIN)

# =====================================================================
# Tool detection
# =====================================================================

SHARP_TOOL_STATUS = {
    "python": {
        "available": True,
        "path": sys.executable,
        "version": sys.version.split()[0],
        "policy": "runtime",
    },
    "bakta": status_dict("bakta", "genome_only_annotation"),
    "bakta_db": status_dict("bakta_db", "genome_only_annotation"),
    "amrfinder": status_dict("amrfinder", "bakta_dependency"),
    "amrfinder_update": status_dict("amrfinder_update", "bakta_dependency_setup"),
    "hmmsearch": status_dict("hmmsearch", "sarp_hmm_step"),
    "hmmscan": status_dict("hmmscan", "domain_step"),
    "hmmpress": status_dict("hmmpress", "domain_step"),
    "fimo": status_dict("fimo", "motif_step"),
    "meme": status_dict("meme", "motif_step"),
    "diamond": status_dict("diamond", "bakta_dependency"),
    "blastp": status_dict("blastp", "bakta_dependency"),
    "makeblastdb": status_dict("makeblastdb", "bakta_dependency"),
    "cmscan": status_dict("cmscan", "bakta_dependency"),
    "cmpress": status_dict("cmpress", "bakta_dependency"),
    "prodigal": status_dict("prodigal", "bakta_dependency_internal"),
    "aragorn": status_dict("aragorn", "bakta_dependency"),
    "tRNAscan-SE": status_dict("tRNAscan-SE", "bakta_dependency"),
}

BAKTA_BIN = SHARP_TOOL_STATUS["bakta"]["path"]
BAKTA_DB_BIN = SHARP_TOOL_STATUS["bakta_db"]["path"]
AMRFINDER_BIN = SHARP_TOOL_STATUS["amrfinder"]["path"]
AMRFINDER_UPDATE_BIN = SHARP_TOOL_STATUS["amrfinder_update"]["path"]
FIMO_BIN = SHARP_TOOL_STATUS["fimo"]["path"]
MEME_BIN = SHARP_TOOL_STATUS["meme"]["path"]

FIMO_VERSION = command_version(FIMO_BIN) if FIMO_BIN else ""
MEME_VERSION = command_version(MEME_BIN) if MEME_BIN else ""

if requires_bakta_execution and not BAKTA_BIN:
    dependency_blockers.append(
        "Genome-only input requires Bakta, but the bakta command is not available."
    )

if requires_bakta_execution and not AMRFINDER_UPDATE_BIN:
    dependency_blockers.append(
        "Genome-only input requires amrfinder_update to prepare Bakta's AMRFinderPlus database."
    )

if REQUIRE_FIMO_FOR_MOTIF_STEP and not FIMO_BIN:
    dependency_blockers.append(
        "S(H)ARP motif scanning requires FIMO, but the fimo command is not available."
    )

update_status(
    phase="Tools detected",
    message="S(H)ARP checked the runtime tools needed for the selected workflow.",
    progress=43,
    rows=[
        ("Bakta", BAKTA_BIN or "not required / not available"),
        ("amrfinder_update", AMRFINDER_UPDATE_BIN or "not required / not available"),
        ("FIMO", FIMO_BIN or "not available"),
        ("hmmsearch", SHARP_TOOL_STATUS["hmmsearch"]["path"] or "not available"),
        ("hmmscan", SHARP_TOOL_STATUS["hmmscan"]["path"] or "not available"),
    ],
    warnings=resource_warnings + dependency_warnings,
    blockers=dependency_blockers,
    history_label="Runtime tools checked",
)

# =====================================================================
# Patch Bakta PyHMMER compatibility
# =====================================================================

BAKTA_PATCH_STATUS = {
    "needed": bool(requires_bakta_execution),
    "status": "not_required" if not requires_bakta_execution else "pending",
    "target_files": [],
    "backup_files": [],
    "validation_stdout": "",
    "removed_failed_bakta_run_dir": "",
    "error": "",
}

def locate_bakta_package_dir(bakta_python):
    """Locate the installed Bakta package directory using the Bakta environment Python."""
    probe = run_command(
        [
            str(bakta_python),
            "-c",
            "import bakta; from pathlib import Path; print(Path(bakta.__file__).parent)",
        ],
        check=True,
    )

    package_dir = Path(probe.stdout.strip())

    if not package_dir.exists():
        raise RuntimeError(f"Bakta package directory was not found: {package_dir}")

    return package_dir

def remove_existing_sharp_helper(text):
    """Remove previous S(H)ARP compatibility helper block if present."""
    start_marker = "# S(H)ARP compatibility helper start"
    end_marker = "# S(H)ARP compatibility helper end"

    while start_marker in text and end_marker in text:
        start = text.index(start_marker)
        end = text.index(end_marker) + len(end_marker)

        if end < len(text) and text[end:end + 1] == "\n":
            end += 1

        text = text[:start] + text[end:]

    return text

def insert_helper_after_imports(text):
    """Insert _sharp_decode helper after imports."""
    helper = (
        "# S(H)ARP compatibility helper start\n"
        "def _sharp_decode(value):\n"
        "    \"\"\"Decode bytes-like values while preserving plain strings.\"\"\"\n"
        "    if hasattr(value, 'decode'):\n"
        "        return value.decode()\n"
        "    return str(value)\n"
        "# S(H)ARP compatibility helper end\n"
        "\n"
    )

    lines = text.splitlines()
    insert_index = 0

    for index, line in enumerate(lines):
        stripped = line.strip()

        if stripped.startswith("import ") or stripped.startswith("from "):
            insert_index = index + 1
            continue

        if insert_index > 0 and stripped and not stripped.startswith("#"):
            break

    lines.insert(insert_index, helper.rstrip("\n"))

    return "\n".join(lines) + "\n"

def patch_bakta_decode_calls_in_text(text):
    """Patch direct .decode() calls in a Bakta source file."""
    text = remove_existing_sharp_helper(text)

    previous_targeted_patch = "(hit.name.decode() if hasattr(hit.name, 'decode') else str(hit.name))"
    text = text.replace(previous_targeted_patch, "_sharp_decode(hit.name)")

    decode_pattern = re.compile(
        r"(?<![A-Za-z0-9_])"
        r"([A-Za-z_][A-Za-z0-9_]*(?:\.[A-Za-z_][A-Za-z0-9_]*)*)"
        r"\.decode\(\)"
    )

    patched_text = decode_pattern.sub(r"_sharp_decode(\1)", text)

    if "_sharp_decode(" in patched_text:
        patched_text = insert_helper_after_imports(patched_text)

    return patched_text

def list_remaining_direct_decode_calls(text):
    """List remaining direct .decode() calls outside the S(H)ARP helper block."""
    text_without_helper = remove_existing_sharp_helper(text)
    remaining = []

    for line_number, line in enumerate(text_without_helper.splitlines(), start=1):
        if ".decode()" in line:
            remaining.append(f"L{line_number}: {line.strip()}")

    return remaining

def clean_failed_bakta_run_dir_for_current_genome():
    """Remove previous failed Bakta output for the current genome hash."""
    try:
        genome_hash_short = file_sha256(GENOME_FASTA)[:16]
        bakta_run_dir = RUN_DIR / "bakta" / genome_hash_short

        if bakta_run_dir.exists():
            shutil.rmtree(bakta_run_dir)
            return str(bakta_run_dir)

    except Exception:
        return ""

    return ""

def patch_bakta_pyhmmer_compatibility():
    """Patch local Bakta source files for PyHMMER bytes/str compatibility."""
    if not BAKTA_BIN:
        raise RuntimeError("Bakta executable is not available for patching.")

    bakta_bin_path = Path(BAKTA_BIN)

    if not bakta_bin_path.exists():
        raise RuntimeError(f"Bakta executable was not found: {bakta_bin_path}")

    bakta_python = bakta_bin_path.parent / "python"

    if not bakta_python.exists():
        raise RuntimeError(f"Bakta environment Python was not found: {bakta_python}")

    bakta_package_dir = locate_bakta_package_dir(bakta_python)

    candidate_files = sorted(
        path for path in bakta_package_dir.rglob("*.py")
        if path.is_file()
        and "__pycache__" not in path.parts
        and ".sharp_backup_" not in path.name
    )

    patched_files = []
    backup_files = []
    already_safe_files = []
    remaining_decode_report = []

    for source_file in candidate_files:
        original_text = source_file.read_text(encoding="utf-8")

        if ".decode()" not in original_text and "_sharp_decode(" not in original_text:
            continue

        patched_text = patch_bakta_decode_calls_in_text(original_text)

        if patched_text != original_text:
            backup_path = source_file.with_suffix(
                source_file.suffix + f".sharp_backup_{int(time.time())}"
            )
            shutil.copy2(source_file, backup_path)
            source_file.write_text(patched_text, encoding="utf-8")

            patched_files.append(str(source_file))
            backup_files.append(str(backup_path))
        else:
            already_safe_files.append(str(source_file))

    files_to_validate = sorted(set(patched_files + already_safe_files))

    for source_file in files_to_validate:
        source_path = Path(source_file)
        final_text = source_path.read_text(encoding="utf-8")
        remaining_direct_decode_calls = list_remaining_direct_decode_calls(final_text)

        for item in remaining_direct_decode_calls:
            remaining_decode_report.append(f"{source_path}: {item}")

    if remaining_decode_report:
        raise RuntimeError(
            "Direct .decode() calls remain in Bakta source files:\n"
            + "\n".join(remaining_decode_report[:40])
        )

    for source_file in patched_files:
        run_command(
            [
                str(bakta_python),
                "-m",
                "py_compile",
                str(source_file),
            ],
            check=True,
        )

    validation_script = r"""
import bakta
import bakta.features.orf as orf
import bakta.features.cds as cds
from pathlib import Path

pkg = Path(bakta.__file__).parent
remaining = []

def remove_helper(text):
    marker_start = "# S(H)ARP compatibility helper start"
    marker_end = "# S(H)ARP compatibility helper end"

    while marker_start in text and marker_end in text:
        start = text.index(marker_start)
        end = text.index(marker_end) + len(marker_end)

        if end < len(text) and text[end:end + 1] == "\n":
            end += 1

        text = text[:start] + text[end:]

    return text

for path in pkg.rglob("*.py"):
    if "__pycache__" in path.parts or ".sharp_backup_" in path.name:
        continue

    text = remove_helper(path.read_text())

    if ".decode()" in text:
        remaining.append(str(path))

assert not remaining, remaining

print("bakta_package_decode_patch_validation=PASS")
print("bakta_package_dir=" + str(pkg))
print("orf_file=" + str(Path(orf.__file__)))
print("cds_file=" + str(Path(cds.__file__)))
"""

    validation = run_command(
        [
            str(bakta_python),
            "-c",
            validation_script,
        ],
        check=True,
    )

    removed_failed_run = clean_failed_bakta_run_dir_for_current_genome()

    if patched_files:
        patch_status = "patched"
    elif already_safe_files:
        patch_status = "already_patched"
    else:
        patch_status = "not_needed_no_decode_calls_found"

    return {
        "needed": True,
        "status": patch_status,
        "target_files": patched_files + already_safe_files,
        "backup_files": backup_files,
        "validation_stdout": validation.stdout,
        "removed_failed_bakta_run_dir": removed_failed_run,
        "error": "",
    }

if requires_bakta_execution and BAKTA_BIN:
    update_status(
        phase="Patching Bakta compatibility",
        message="S(H)ARP is patching local Bakta package files for PyHMMER bytes/str compatibility in Colab.",
        progress=48,
        rows=[
            ("Bakta", BAKTA_BIN),
            ("Scope", "all Bakta Python files with direct .decode() calls"),
            ("Purpose", "avoid PyHMMER bytes/str failures in ORF, CDS/Pfam, and related steps"),
        ],
        warnings=resource_warnings + dependency_warnings,
        blockers=dependency_blockers,
        history_label="Bakta compatibility patch started",
    )

    try:
        BAKTA_PATCH_STATUS = patch_bakta_pyhmmer_compatibility()
    except Exception as exc:
        BAKTA_PATCH_STATUS = {
            "needed": True,
            "status": "failed",
            "target_files": [],
            "backup_files": [],
            "validation_stdout": "",
            "removed_failed_bakta_run_dir": "",
            "error": str(exc),
        }
        dependency_blockers.append(f"Bakta compatibility patch failed: {exc}")

# =====================================================================
# Download bundled S(H)ARP resources
# =====================================================================

SHARP_RESOURCE_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "leepusp/sharp-colab/main/resources"
)

SHARP_INTERNAL_RESOURCES = {
    "heptamer_meme": str(DATABASES_DIR / "motifs" / "heptarepeats2.meme"),
    "sarp_hmm": str(DATABASES_DIR / "hmm" / "sarp_custom.hmm"),
    "domain_models_hmm": str(DATABASES_DIR / "domain_models" / "domain_models.hmm"),
    "embedding_reference": str(DATABASES_DIR / "embeddings" / "reference_embeddings.parquet"),
    "scoring_config": str(CONFIG_DIR / "sharp_scoring_config.json"),
    "report_module": str(PROJECT_DIR / "resources" / "modules" / "operon_fig_colab.py"),
}

SHARP_BUNDLED_RESOURCES = {
    "heptamer_meme": {
        "url": f"{SHARP_RESOURCE_BASE_URL}/motifs/heptarepeats2.meme",
        "path": SHARP_INTERNAL_RESOURCES["heptamer_meme"],
        "required_for_complete_results": True,
        "purpose": "FIMO motif search for heptamer repeats / regulatory motifs.",
    },
    "sarp_hmm": {
        "url": f"{SHARP_RESOURCE_BASE_URL}/hmm/sarp_custom.hmm",
        "path": SHARP_INTERNAL_RESOURCES["sarp_hmm"],
        "required_for_complete_results": True,
        "purpose": "HMM search for SARP/BtaD-like regulators.",
    },
    "domain_models_hmm": {
        "url": f"{SHARP_RESOURCE_BASE_URL}/domain_models/domain_models.hmm",
        "path": SHARP_INTERNAL_RESOURCES["domain_models_hmm"],
        "required_for_complete_results": True,
        "purpose": "Domain evidence with hmmscan.",
    },
    "embedding_reference": {
        "url": f"{SHARP_RESOURCE_BASE_URL}/embeddings/reference_embeddings.parquet",
        "path": SHARP_INTERNAL_RESOURCES["embedding_reference"],
        "required_for_complete_results": True,
        "purpose": "Reference embedding table for similarity/scoring.",
    },
    "scoring_config": {
        "url": f"{SHARP_RESOURCE_BASE_URL}/config/sharp_scoring_config.json",
        "path": SHARP_INTERNAL_RESOURCES["scoring_config"],
        "required_for_complete_results": True,
        "purpose": "S(H)ARP scoring thresholds and neighborhood parameters.",
    },
    "report_module": {
        "url": f"{SHARP_RESOURCE_BASE_URL}/modules/operon_fig_colab.py",
        "path": SHARP_INTERNAL_RESOURCES["report_module"],
        "required_for_complete_results": False,
        "purpose": "Optional operon/neighborhood visualization module.",
    },
}

def write_default_scoring_config(path):
    """Write a minimal scoring config if the repository config is unavailable."""
    default_config = {
        "version": "0.1.0",
        "neighborhood": {
            "max_distance_bp": 20000,
            "max_genes_each_side": 12,
        },
        "evidence_weights": {
            "heptamer_motif": 1.0,
            "sarp_hmm": 2.0,
            "domain_model": 1.0,
            "embedding_similarity": 1.0,
        },
        "thresholds": {
            "candidate_region_min_score": 2.0,
            "sarp_hmm_max_evalue": 1e-5,
            "domain_hmm_max_evalue": 1e-3,
            "fimo_max_qvalue": 0.05,
        },
    }

    write_json(path, default_config)

SHARP_RESOURCE_STATUS = {}

update_status(
    phase="Loading S(H)ARP resources",
    message="Small bundled S(H)ARP resources are being checked and downloaded when available.",
    progress=53,
    rows=[
        ("Resource source", SHARP_RESOURCE_BASE_URL),
        ("Config directory", CONFIG_DIR),
    ],
    warnings=resource_warnings + dependency_warnings,
    blockers=dependency_blockers,
    history_label="Resource loading started",
)

if INSTALL_BUNDLED_SHARP_RESOURCES:
    for key, item in SHARP_BUNDLED_RESOURCES.items():
        output_path = Path(item["path"])
        output_path.parent.mkdir(parents=True, exist_ok=True)

        if (
            output_path.exists()
            and output_path.stat().st_size > 0
            and not OVERWRITE_BUNDLED_RESOURCES
        ):
            SHARP_RESOURCE_STATUS[key] = {
                "available": True,
                "path": str(output_path),
                "source": "existing",
                "purpose": item["purpose"],
                "required_for_complete_results": item["required_for_complete_results"],
            }
            continue

        result = url_download(
            item["url"],
            output_path,
            label=f"download_resource_{key}",
        )

        available = bool(
            result.get("ok")
            and output_path.exists()
            and output_path.stat().st_size > 0
        )

        if not available and key == "scoring_config":
            write_default_scoring_config(output_path)
            available = True
            result = {
                "ok": True,
                "url": "built_in_default",
                "path": str(output_path),
                "size_bytes": output_path.stat().st_size,
            }

        SHARP_RESOURCE_STATUS[key] = {
            "available": available,
            "path": str(output_path),
            "source": result.get("url", ""),
            "purpose": item["purpose"],
            "required_for_complete_results": item["required_for_complete_results"],
            "download_error": result.get("error", ""),
        }

else:
    for key, item in SHARP_BUNDLED_RESOURCES.items():
        output_path = Path(item["path"])
        SHARP_RESOURCE_STATUS[key] = {
            "available": output_path.exists() and output_path.stat().st_size > 0,
            "path": str(output_path),
            "source": "not_installed_this_run",
            "purpose": item["purpose"],
            "required_for_complete_results": item["required_for_complete_results"],
        }

if not SHARP_RESOURCE_STATUS.get("heptamer_meme", {}).get("available"):
    resource_warnings.append(
        "heptarepeats2.meme is not loaded; motif search will be pending."
    )

if not SHARP_RESOURCE_STATUS.get("sarp_hmm", {}).get("available"):
    resource_warnings.append(
        "SARP HMM file is not loaded; SARP HMM step will be pending."
    )

if not SHARP_RESOURCE_STATUS.get("domain_models_hmm", {}).get("available"):
    resource_warnings.append(
        "Domain HMM database is not loaded; domain step will be pending."
    )

if not SHARP_RESOURCE_STATUS.get("embedding_reference", {}).get("available"):
    resource_warnings.append(
        "Embedding reference table is not loaded; embedding score step will be pending."
    )

if not SHARP_RESOURCE_STATUS.get("report_module", {}).get("available"):
    resource_warnings.append(
        "Optional report module is not loaded; the default report builder will be used."
    )

# =====================================================================
# Bakta DB light release config
# =====================================================================

BAKTA_CACHE_CONFIG_URL = (
    "https://raw.githubusercontent.com/"
    "leepusp/sharp-colab/main/resources/config/bakta_light_cache.json"
)

BAKTA_CACHE_CONFIG_FALLBACK = {
    "version": "0.2.0",
    "policy": {
        "prodigal_enabled": False,
        "genome_only_annotation_backend": "bakta_cached_light_db",
        "annotated_package_preferred_for_fast_runs": True,
    },
    "bakta": {
        "db_type": "light",
        "db_major": 6,
        "db_minor": 0,
        "archive_name": "db-light.tar.xz",
        "archive_size_bytes": 1344199496,
        "expected_md5": "4a6e059ded39e9c5537ef4137d2f5648",
        "expected_sha256": "dab28b58fbf51fde4b72793c1edf7a9de6530cfb501d2dd5a72b16b8ecb4c705",
        "github_release_url": "https://github.com/leepusp/sharp-colab/releases/download/bakta-db-light-v6.0/db-light.tar.xz",
        "github_md5_url": "https://github.com/leepusp/sharp-colab/releases/download/bakta-db-light-v6.0/db-light.tar.xz.md5",
        "github_sha256_url": "https://github.com/leepusp/sharp-colab/releases/download/bakta-db-light-v6.0/db-light.tar.xz.sha256",
        "zenodo_fallback_url": "https://zenodo.org/records/14916843/files/db-light.tar.xz?download=1",
        "zenodo_record": "14916843",
        "doi": "10.5281/zenodo.14916843",
    },
    "colab": {
        "download_dir": str(DATABASES_DIR / "bakta_cache"),
        "extract_dir": str(DATABASES_DIR / "bakta"),
        "run_from_extracted_local_content_only": True,
    },
}

BAKTA_CACHE_CONFIG_PATH = CONFIG_DIR / "bakta_light_cache.json"

BAKTA_LIGHT_CACHE_CONFIG, config_warning = load_remote_json(
    BAKTA_CACHE_CONFIG_URL,
    fallback=BAKTA_CACHE_CONFIG_FALLBACK,
    output_path=BAKTA_CACHE_CONFIG_PATH,
)

if config_warning:
    resource_warnings.append(
        "Could not load remote Bakta cache config from GitHub; built-in fallback config will be used."
    )

BAKTA_ARCHIVE_NAME = BAKTA_LIGHT_CACHE_CONFIG["bakta"]["archive_name"]
BAKTA_EXPECTED_MD5 = BAKTA_LIGHT_CACHE_CONFIG["bakta"]["expected_md5"]
BAKTA_EXPECTED_SHA256 = BAKTA_LIGHT_CACHE_CONFIG["bakta"].get("expected_sha256", "")
BAKTA_RELEASE_URL = BAKTA_LIGHT_CACHE_CONFIG["bakta"]["github_release_url"]
BAKTA_ZENODO_FALLBACK_URL = BAKTA_LIGHT_CACHE_CONFIG["bakta"].get("zenodo_fallback_url", "")

BAKTA_DOWNLOAD_DIR = Path(
    BAKTA_LIGHT_CACHE_CONFIG.get("colab", {}).get(
        "download_dir",
        str(DATABASES_DIR / "bakta_cache"),
    )
)

BAKTA_EXTRACT_PARENT = Path(
    BAKTA_LIGHT_CACHE_CONFIG.get("colab", {}).get(
        "extract_dir",
        str(DATABASES_DIR / "bakta"),
    )
)

BAKTA_ARCHIVE_PATH = BAKTA_DOWNLOAD_DIR / BAKTA_ARCHIVE_NAME
BAKTA_DB_PATH = ""
BAKTA_DB_READY = False
BAKTA_DB_VERSION = {}
BAKTA_DB_SOURCE_URL = ""
BAKTA_DB_VALIDATION = {
    "required_files_checked": [],
    "missing_or_unreadable": [],
}

AMRFINDER_DB_READY = False
AMRFINDER_DB_PATH = ""
AMRFINDER_SETUP_STATUS = {
    "ready": False,
    "path": "",
    "amrfinder_update": AMRFINDER_UPDATE_BIN,
    "status": "not_required" if not requires_bakta_execution else "pending",
    "sentinel": "",
    "error": "",
}

# =====================================================================
# Download and extract Bakta DB only for genome-only FASTA
# =====================================================================

BAKTA_RUNTIME_ARGS = []
BAKTA_RUN_PATH_PREFIX = ""

def build_bakta_cds_focused_args(bakta_bin):
    """Build optional Bakta arguments while preserving CDS/protein annotation."""
    if not bakta_bin:
        return []

    try:
        result = subprocess.run(
            [bakta_bin, "--help"],
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            check=False,
        )
        help_text = result.stdout or ""
    except Exception:
        return []

    candidate_flags = [
        "--skip-trna",
        "--skip-tmrna",
        "--skip-rrna",
        "--skip-ncrna",
        "--skip-ncrna-region",
        "--skip-crispr",
        "--skip-sorf",
        "--skip-gap",
        "--skip-ori",
        "--skip-plot",
    ]

    return [flag for flag in candidate_flags if flag in help_text]

def prepare_amrfinderplus_database_for_bakta(bakta_db_path):
    """Prepare AMRFinderPlus internal DB required by Bakta after manual DB extraction."""
    bakta_db_path = Path(bakta_db_path)
    amrfinder_db_path = bakta_db_path / "amrfinderplus-db"
    sentinel_path = amrfinder_db_path / ".sharp_amrfinder_update_ok.json"

    if not bakta_db_path.exists():
        raise RuntimeError(f"Bakta DB path does not exist: {bakta_db_path}")

    if not amrfinder_db_path.exists():
        raise RuntimeError(f"AMRFinderPlus DB directory does not exist: {amrfinder_db_path}")

    amrfinder_update_bin = command_path("amrfinder_update")

    if not amrfinder_update_bin:
        raise RuntimeError("amrfinder_update executable was not found in PATH.")

    if (
        sentinel_path.exists()
        and not FORCE_AMRFINDER_UPDATE
        and os.access(amrfinder_db_path, os.R_OK)
    ):
        try:
            sentinel = read_json_if_exists(sentinel_path) or {}
            if sentinel.get("status") == "ready":
                return {
                    "ready": True,
                    "path": str(amrfinder_db_path),
                    "amrfinder_update": amrfinder_update_bin,
                    "status": "already_prepared",
                    "sentinel": str(sentinel_path),
                    "error": "",
                }
        except Exception:
            pass

    update_status(
        phase="Preparing AMRFinderPlus DB",
        message="Bakta requires AMRFinderPlus to initialize its internal database after manual Bakta DB extraction.",
        progress=88,
        rows=[
            ("amrfinder_update", amrfinder_update_bin),
            ("AMRFinderPlus DB", amrfinder_db_path),
        ],
        warnings=resource_warnings + dependency_warnings,
        blockers=dependency_blockers,
        history_label="AMRFinderPlus DB preparation started",
    )

    run_cmd(
        [
            amrfinder_update_bin,
            "--force_update",
            "--database",
            str(amrfinder_db_path),
        ],
        label="amrfinder_update_bakta_internal_db",
        check=True,
        capture=True,
        timeout=3600,
    )

    chmod_readable_tree(amrfinder_db_path)

    if not amrfinder_db_path.exists() or not os.access(amrfinder_db_path, os.R_OK):
        raise RuntimeError(f"AMRFinderPlus DB is not readable after update: {amrfinder_db_path}")

    sentinel_payload = {
        "status": "ready",
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "amrfinder_update": amrfinder_update_bin,
        "database": str(amrfinder_db_path),
    }

    write_json(sentinel_path, sentinel_payload)

    return {
        "ready": True,
        "path": str(amrfinder_db_path),
        "amrfinder_update": amrfinder_update_bin,
        "status": "prepared",
        "sentinel": str(sentinel_path),
        "error": "",
    }

if requires_bakta_db:
    if not DOWNLOAD_BAKTA_DB_FOR_GENOME_ONLY:
        dependency_blockers.append(
            "Genome-only FASTA requires Bakta DB light, but automatic DB download is disabled."
        )
    else:
        update_status(
            phase="Preparing Bakta DB light",
            message="The database will be downloaded from the S(H)ARP GitHub Release if it is not already cached.",
            progress=65,
            rows=[
                ("Release URL", BAKTA_RELEASE_URL),
                ("Archive path", BAKTA_ARCHIVE_PATH),
                ("Expected MD5", BAKTA_EXPECTED_MD5),
            ],
            warnings=resource_warnings + dependency_warnings,
            blockers=dependency_blockers,
            history_label="Bakta DB preparation started",
        )

        if FORCE_REFRESH_BAKTA_DB_ARCHIVE and BAKTA_ARCHIVE_PATH.exists():
            BAKTA_ARCHIVE_PATH.unlink()

        archive_valid = False

        if BAKTA_ARCHIVE_PATH.exists() and BAKTA_ARCHIVE_PATH.stat().st_size > 0:
            try:
                update_status(
                    phase="Validating cached Bakta archive",
                    message="A previous Bakta DB archive was found. S(H)ARP is validating its checksum.",
                    progress=68,
                    rows=[
                        ("Archive path", BAKTA_ARCHIVE_PATH),
                        ("Expected MD5", BAKTA_EXPECTED_MD5),
                    ],
                    warnings=resource_warnings + dependency_warnings,
                    blockers=dependency_blockers,
                    history_label="Cached archive validation started",
                )

                current_md5 = file_md5(BAKTA_ARCHIVE_PATH)
                archive_valid = current_md5 == BAKTA_EXPECTED_MD5

                if not archive_valid:
                    resource_warnings.append(
                        "Existing Bakta DB archive failed MD5 validation and will be downloaded again."
                    )
                    BAKTA_ARCHIVE_PATH.unlink()

            except Exception:
                archive_valid = False

        if not archive_valid:
            fallback_urls = []
            if USE_ZENODO_FALLBACK_IF_GITHUB_FAILS and BAKTA_ZENODO_FALLBACK_URL:
                fallback_urls.append(BAKTA_ZENODO_FALLBACK_URL)

            update_status(
                phase="Downloading Bakta DB light",
                message="Downloading the cached Bakta DB light release asset. This is the longest setup step.",
                progress=70,
                rows=[
                    ("Source", BAKTA_RELEASE_URL),
                    ("Destination", BAKTA_ARCHIVE_PATH),
                    ("Archive size", "about 1.3 GB"),
                ],
                warnings=resource_warnings + dependency_warnings,
                blockers=dependency_blockers,
                history_label="Bakta DB download started",
            )

            download_result = url_download(
                BAKTA_RELEASE_URL,
                BAKTA_ARCHIVE_PATH,
                label="download_bakta_db_light_release_asset",
                fallback_urls=fallback_urls,
            )

            if not download_result.get("ok"):
                dependency_blockers.append(
                    "Failed to download Bakta DB light archive: "
                    + str(download_result.get("error", "unknown error"))
                )
            else:
                BAKTA_DB_SOURCE_URL = download_result.get("url", "")

        if BAKTA_ARCHIVE_PATH.exists() and BAKTA_ARCHIVE_PATH.stat().st_size > 0:
            update_status(
                phase="Validating Bakta DB light",
                message="S(H)ARP is validating MD5 and SHA256 checksums before extraction.",
                progress=76,
                rows=[
                    ("Archive", BAKTA_ARCHIVE_PATH),
                    ("Expected MD5", BAKTA_EXPECTED_MD5),
                ],
                warnings=resource_warnings + dependency_warnings,
                blockers=dependency_blockers,
                history_label="Checksum validation started",
            )

            actual_md5 = file_md5(BAKTA_ARCHIVE_PATH)

            if actual_md5 != BAKTA_EXPECTED_MD5:
                dependency_blockers.append(
                    f"Bakta DB light MD5 mismatch: expected {BAKTA_EXPECTED_MD5}, got {actual_md5}"
                )

            if BAKTA_EXPECTED_SHA256:
                actual_sha256 = file_sha256(BAKTA_ARCHIVE_PATH)

                if actual_sha256 != BAKTA_EXPECTED_SHA256:
                    dependency_blockers.append(
                        f"Bakta DB light SHA256 mismatch: expected {BAKTA_EXPECTED_SHA256}, got {actual_sha256}"
                    )

            if not dependency_blockers:
                update_status(
                    phase="Extracting and validating Bakta DB light",
                    message="The archive is valid. S(H)ARP is extracting the database and checking required runtime files.",
                    progress=82,
                    rows=[
                        ("Archive", BAKTA_ARCHIVE_PATH),
                        ("Extract parent", BAKTA_EXTRACT_PARENT),
                    ],
                    warnings=resource_warnings + dependency_warnings,
                    blockers=dependency_blockers,
                    history_label="Bakta DB extraction started",
                )

                try:
                    extraction = extract_bakta_db_light(
                        BAKTA_ARCHIVE_PATH,
                        BAKTA_EXTRACT_PARENT,
                        force=FORCE_REEXTRACT_BAKTA_DB,
                    )

                    BAKTA_DB_PATH = extraction["db_path"]
                    BAKTA_DB_READY = bool(Path(BAKTA_DB_PATH).exists())

                    version_json_path = Path(extraction["version_json"])
                    BAKTA_DB_VERSION = read_json_if_exists(version_json_path) or {}

                    BAKTA_DB_VALIDATION = {
                        "required_files_checked": [
                            "version.json",
                            "antifam.h3f",
                            "antifam.h3i",
                            "antifam.h3m",
                            "antifam.h3p",
                            "pfam.h3f",
                            "pfam.h3i",
                            "pfam.h3m",
                            "pfam.h3p",
                            "amrfinderplus-db",
                        ],
                        "missing_or_unreadable": [],
                        "extracted_this_run": bool(extraction.get("extracted", False)),
                    }

                    os.environ["BAKTA_DB"] = str(BAKTA_DB_PATH)

                except Exception as exc:
                    dependency_blockers.append(f"Bakta DB extraction/validation failed: {exc}")

        else:
            dependency_blockers.append(
                "Bakta DB light archive was not found after the download step."
            )

    if requires_bakta_execution and BAKTA_DB_READY:
        try:
            AMRFINDER_SETUP_STATUS = prepare_amrfinderplus_database_for_bakta(BAKTA_DB_PATH)
            AMRFINDER_DB_READY = bool(AMRFINDER_SETUP_STATUS["ready"])
            AMRFINDER_DB_PATH = AMRFINDER_SETUP_STATUS["path"]
        except Exception as exc:
            AMRFINDER_DB_READY = False
            AMRFINDER_DB_PATH = str(Path(BAKTA_DB_PATH) / "amrfinderplus-db") if BAKTA_DB_PATH else ""
            AMRFINDER_SETUP_STATUS = {
                "ready": False,
                "path": AMRFINDER_DB_PATH,
                "amrfinder_update": command_path("amrfinder_update"),
                "status": "failed",
                "sentinel": "",
                "error": str(exc),
            }
            dependency_blockers.append(f"AMRFinderPlus DB preparation failed: {exc}")

    if BAKTA_BIN:
        BAKTA_RUN_PATH_PREFIX = str(Path(BAKTA_BIN).parent)
        BAKTA_RUNTIME_ARGS = build_bakta_cds_focused_args(BAKTA_BIN)

    if requires_bakta_execution and not BAKTA_BIN:
        dependency_blockers.append(
            "Genome-only FASTA requires Bakta execution, but bakta is not available."
        )

    if requires_bakta_execution and not BAKTA_DB_READY:
        dependency_blockers.append(
            "Genome-only FASTA requires Bakta DB light, but the DB is not ready."
        )

    if requires_bakta_execution and not AMRFINDER_DB_READY:
        dependency_blockers.append(
            "Genome-only FASTA requires AMRFinderPlus DB preparation, but it is not ready."
        )

else:
    BAKTA_DB_READY = False
    BAKTA_DB_PATH = ""
    AMRFINDER_DB_READY = False
    AMRFINDER_DB_PATH = ""

# =====================================================================
# Final dependency status
# =====================================================================

critical_tools_ready = True

if requires_bakta_execution:
    critical_tools_ready = bool(
        BAKTA_BIN
        and BAKTA_DB_READY
        and AMRFINDER_DB_READY
    )

if REQUIRE_FIMO_FOR_MOTIF_STEP:
    critical_tools_ready = bool(critical_tools_ready and FIMO_BIN)

SHARP_DEPENDENCIES_READY = bool(critical_tools_ready and not dependency_blockers)

SHARP_PYTHON_STATUS = {
    "python": sys.executable,
    "version": sys.version,
    "runtime": RUNTIME_NAME,
    "hostname": HOSTNAME,
    "platform": platform.platform(),
}

SHARP_BACKEND = {
    "workflow": workflow,
    "input_mode": input_mode,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "annotation_action": annotation_action,
    "genome_only_annotation_engine": GENOME_ONLY_ANNOTATION_ENGINE,
    "prodigal_enabled": False,
    "use_prodigal_for_genome_only": False,
    "use_bakta_for_genome_only": bool(USE_BAKTA_FOR_GENOME_ONLY),
    "requires_bakta_db": bool(requires_bakta_db),
    "requires_bakta_execution": bool(requires_bakta_execution),
    "bakta_bin": BAKTA_BIN,
    "bakta_db_bin": BAKTA_DB_BIN,
    "bakta_db_path": BAKTA_DB_PATH,
    "bakta_db_ready": bool(BAKTA_DB_READY),
    "bakta_db_version": BAKTA_DB_VERSION,
    "bakta_db_validation": BAKTA_DB_VALIDATION,
    "bakta_runtime_args": BAKTA_RUNTIME_ARGS,
    "bakta_run_path_prefix": BAKTA_RUN_PATH_PREFIX,
    "bakta_cache_config_url": BAKTA_CACHE_CONFIG_URL,
    "bakta_cache_config_path": str(BAKTA_CACHE_CONFIG_PATH),
    "bakta_release_url": BAKTA_RELEASE_URL,
    "bakta_archive_path": str(BAKTA_ARCHIVE_PATH),
    "bakta_expected_md5": BAKTA_EXPECTED_MD5,
    "bakta_expected_sha256": BAKTA_EXPECTED_SHA256,
    "bakta_pyhmmer_patch": BAKTA_PATCH_STATUS,
    "amrfinder_bin": AMRFINDER_BIN,
    "amrfinder_update_bin": AMRFINDER_UPDATE_BIN,
    "amrfinder_db_path": AMRFINDER_DB_PATH,
    "amrfinder_db_ready": bool(AMRFINDER_DB_READY),
    "amrfinder_setup_status": AMRFINDER_SETUP_STATUS,
    "fimo_bin": FIMO_BIN,
    "fimo_ready": bool(FIMO_BIN),
    "fimo_version": FIMO_VERSION,
    "meme_bin": MEME_BIN,
    "meme_version": MEME_VERSION,
    "meme_env_dir": str(MEME_ENV_DIR),
    "meme_env_bin": str(MEME_ENV_BIN),
    "require_fimo_for_motif_step": bool(REQUIRE_FIMO_FOR_MOTIF_STEP),
    "download_bakta_db_for_genome_only": bool(DOWNLOAD_BAKTA_DB_FOR_GENOME_ONLY),
}

SHARP_DEPENDENCY_STATUS = {
    "ready": bool(SHARP_DEPENDENCIES_READY),
    "runtime": RUNTIME_NAME,
    "workflow": workflow,
    "input_mode": input_mode,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "tool_status": SHARP_TOOL_STATUS,
    "python_status": SHARP_PYTHON_STATUS,
    "backend": SHARP_BACKEND,
    "warnings": unique_list(dependency_warnings + resource_warnings),
    "blockers": unique_list(dependency_blockers),
}

SHARP_DEPENDENCY_MANIFEST = RUN_DIR / f"{RUN_NAME}_dependency_manifest.json"
SHARP_RESOURCE_MANIFEST = CONFIG_DIR / "sharp_bundled_resource_manifest.json"

write_json(SHARP_DEPENDENCY_MANIFEST, SHARP_DEPENDENCY_STATUS)
write_json(SHARP_RESOURCE_MANIFEST, SHARP_RESOURCE_STATUS)

context_update = {
    "project_dir": str(PROJECT_DIR),
    "data_dir": str(DATA_DIR),
    "input_dir": str(INPUT_DIR),
    "normalized_dir": str(NORMALIZED_DIR),
    "config_dir": str(CONFIG_DIR),
    "databases_dir": str(DATABASES_DIR),
    "results_dir": str(RESULTS_DIR),
    "run_name": RUN_NAME,
    "run_dir": str(RUN_DIR),
    "log_dir": str(LOG_DIR),
    "table_dir": str(TABLE_DIR),
    "report_dir": str(REPORT_DIR),
    "runtime": RUNTIME_NAME,
    "hostname": HOSTNAME,
    "genome_fasta": str(GENOME_FASTA),
    "annotation_file": str(ANNOTATION_FILE) if has_annotation else "",
    "protein_fasta": str(PROTEIN_FASTA) if has_proteins else "",
    "cds_fasta": str(CDS_FASTA) if has_cds else "",
    "workflow": workflow,
    "input_mode": input_mode,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "bakta_bin": BAKTA_BIN,
    "bakta_db_path": BAKTA_DB_PATH,
    "bakta_db_ready": bool(BAKTA_DB_READY),
    "bakta_pyhmmer_patch": BAKTA_PATCH_STATUS,
    "amrfinder_db_path": AMRFINDER_DB_PATH,
    "amrfinder_db_ready": bool(AMRFINDER_DB_READY),
    "amrfinder_setup_status": AMRFINDER_SETUP_STATUS,
    "fimo_bin": FIMO_BIN,
    "fimo_ready": bool(FIMO_BIN),
    "fimo_version": FIMO_VERSION,
    "dependencies_ready": bool(SHARP_DEPENDENCIES_READY),
    "dependency_manifest": str(SHARP_DEPENDENCY_MANIFEST),
    "resource_manifest": str(SHARP_RESOURCE_MANIFEST),
}

context.update(context_update)
write_json(SHARP_CONTEXT_FILE, context)

globals().update({
    "PROJECT_DIR": PROJECT_DIR,
    "DATA_DIR": DATA_DIR,
    "INPUT_DIR": INPUT_DIR,
    "NORMALIZED_DIR": NORMALIZED_DIR,
    "CONFIG_DIR": CONFIG_DIR,
    "DATABASES_DIR": DATABASES_DIR,
    "RESULTS_DIR": RESULTS_DIR,
    "RUN_NAME": RUN_NAME,
    "RUN_DIR": RUN_DIR,
    "LOG_DIR": LOG_DIR,
    "TABLE_DIR": TABLE_DIR,
    "REPORT_DIR": REPORT_DIR,
    "GENOME_FASTA": GENOME_FASTA,
    "ANNOTATION_FILE": ANNOTATION_FILE if has_annotation else None,
    "PROTEIN_FASTA": PROTEIN_FASTA if has_proteins else None,
    "CDS_FASTA": CDS_FASTA if has_cds else None,
    "workflow": workflow,
    "input_mode": input_mode,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "annotation_action": annotation_action,
    "SHARP_PYTHON_STATUS": SHARP_PYTHON_STATUS,
    "SHARP_TOOL_STATUS": SHARP_TOOL_STATUS,
    "SHARP_INTERNAL_RESOURCES": SHARP_INTERNAL_RESOURCES,
    "SHARP_RESOURCE_STATUS": SHARP_RESOURCE_STATUS,
    "SHARP_BACKEND": SHARP_BACKEND,
    "SHARP_DEPENDENCIES_READY": SHARP_DEPENDENCIES_READY,
    "SHARP_DEPENDENCY_STATUS": SHARP_DEPENDENCY_STATUS,
    "SHARP_DEPENDENCY_MANIFEST": SHARP_DEPENDENCY_MANIFEST,
    "SHARP_RESOURCE_MANIFEST": SHARP_RESOURCE_MANIFEST,
    "BAKTA_BIN": BAKTA_BIN,
    "BAKTA_DB_BIN": BAKTA_DB_BIN,
    "BAKTA_DB_PATH": BAKTA_DB_PATH,
    "BAKTA_DB_READY": BAKTA_DB_READY,
    "BAKTA_DB_VERSION": BAKTA_DB_VERSION,
    "BAKTA_DB_VALIDATION": BAKTA_DB_VALIDATION,
    "BAKTA_PATCH_STATUS": BAKTA_PATCH_STATUS,
    "BAKTA_RUN_PATH_PREFIX": BAKTA_RUN_PATH_PREFIX,
    "BAKTA_RUNTIME_ARGS": BAKTA_RUNTIME_ARGS,
    "BAKTA_LIGHT_CACHE_CONFIG": BAKTA_LIGHT_CACHE_CONFIG,
    "BAKTA_CACHE_CONFIG_URL": BAKTA_CACHE_CONFIG_URL,
    "BAKTA_ARCHIVE_PATH": BAKTA_ARCHIVE_PATH,
    "AMRFINDER_BIN": AMRFINDER_BIN,
    "AMRFINDER_UPDATE_BIN": AMRFINDER_UPDATE_BIN,
    "AMRFINDER_DB_READY": AMRFINDER_DB_READY,
    "AMRFINDER_DB_PATH": AMRFINDER_DB_PATH,
    "AMRFINDER_SETUP_STATUS": AMRFINDER_SETUP_STATUS,
    "FIMO_BIN": FIMO_BIN,
    "FIMO_VERSION": FIMO_VERSION,
    "MEME_BIN": MEME_BIN,
    "MEME_VERSION": MEME_VERSION,
    "MEME_ENV_DIR": MEME_ENV_DIR,
    "MEME_ENV_BIN": MEME_ENV_BIN,
})

final_rows = [
    ("Runtime", RUNTIME_NAME),
    ("Input mode", input_mode),
    ("Annotation mode", annotation_mode),
    ("Annotation backend", annotation_backend),
    ("Genome-only engine", GENOME_ONLY_ANNOTATION_ENGINE if requires_bakta_db else "not needed"),
    ("Bakta", BAKTA_BIN or "not required / not available"),
    ("Bakta patch", BAKTA_PATCH_STATUS.get("status", "not required")),
    ("Bakta patch files", len(BAKTA_PATCH_STATUS.get("target_files", []))),
    ("Bakta DB", BAKTA_DB_PATH if BAKTA_DB_READY else ("not required" if not requires_bakta_db else "not ready")),
    ("Bakta DB type", BAKTA_DB_VERSION.get("type", "not required" if not requires_bakta_db else "unknown")),
    ("Bakta DB date", BAKTA_DB_VERSION.get("date", "not required" if not requires_bakta_db else "unknown")),
    ("AMRFinderPlus DB", AMRFINDER_DB_PATH if AMRFINDER_DB_READY else ("not required" if not requires_bakta_execution else "not ready")),
    ("AMRFinderPlus setup", AMRFINDER_SETUP_STATUS.get("status", "not required")),
    ("FIMO", FIMO_BIN if FIMO_BIN else "not ready"),
    ("FIMO version", FIMO_VERSION or "unknown"),
    ("heptarepeats2.meme", "loaded" if SHARP_RESOURCE_STATUS.get("heptamer_meme", {}).get("available") else "pending"),
    ("Dependencies", "ready" if SHARP_DEPENDENCIES_READY else "blocked"),
    ("Manifest", SHARP_DEPENDENCY_MANIFEST),
]

update_status(
    phase="S(H)ARP environment ready" if SHARP_DEPENDENCIES_READY else "S(H)ARP setup needs attention",
    message="Setup completed. Continue to the core analysis cell when dependencies are ready.",
    progress=100 if SHARP_DEPENDENCIES_READY else 90,
    rows=final_rows,
    warnings=dependency_warnings + resource_warnings,
    blockers=dependency_blockers,
    done=True,
    history_label="Setup finished",
)

print("S(H)ARP setup finished.")
print("dependencies_ready:", SHARP_DEPENDENCIES_READY)
print("workflow:", workflow)
print("input_mode:", input_mode)
print("annotation_backend:", annotation_backend)
print("bakta_bin:", BAKTA_BIN)
print("bakta_patch_status:", BAKTA_PATCH_STATUS)
print("bakta_db_ready:", BAKTA_DB_READY)
print("bakta_db_path:", BAKTA_DB_PATH)
print("bakta_db_version:", BAKTA_DB_VERSION)
print("amrfinder_db_ready:", AMRFINDER_DB_READY)
print("amrfinder_db_path:", AMRFINDER_DB_PATH)
print("amrfinder_setup_status:", AMRFINDER_SETUP_STATUS)
print("fimo_ready:", bool(FIMO_BIN))
print("fimo_bin:", FIMO_BIN)
print("fimo_version:", FIMO_VERSION)
print("meme_env_dir:", MEME_ENV_DIR)
print("bakta_runtime_args:", " ".join(BAKTA_RUNTIME_ARGS))
print("dependency_manifest:", SHARP_DEPENDENCY_MANIFEST)

In [ ]:
# @title Run S(H)ARP core analysis { display-mode: "form" }

# =====================================================================
# Imports
# =====================================================================

from pathlib import Path
from IPython.display import display, HTML
from types import SimpleNamespace
import hashlib
import html
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import traceback

import pandas as pd
from Bio import SeqIO

# Force a non-interactive Matplotlib backend for command-line tools.
# This prevents Colab's inline backend from breaking Bakta subprocesses.
os.environ["MPLBACKEND"] = "Agg"

MPLCONFIG_DIR = Path("/tmp") / "sharp_matplotlib"
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIG_DIR)

# =====================================================================
# Internal analysis defaults
# =====================================================================

BAKTA_THREADS = 2
BAKTA_PREFIX = "sharp_bakta"

FIMO_PVALUE_THRESHOLD = 1e-4
FIMO_REPEAT_DISTANCE_MIN = 2
FIMO_REPEAT_DISTANCE_MAX = 15
FIMO_REPEAT_PRE_FILTER_MAX = 20

SARP_HMM_EVALUE_THRESHOLD = 1e-3
SARP_HMM_SCORE_THRESHOLD = 10

NEIGHBORHOOD_BEFORE_GENES = 10
NEIGHBORHOOD_AFTER_GENES = 10

DEBUG_CORE = False

# =====================================================================
# Single-panel UI
# =====================================================================

SHARP_COLORS = {
    "background": "#97003F",
    "dark": "#760032",
    "deep": "#4D0022",
    "pink": "#D8AFC0",
    "soft": "#F1DDE6",
    "white": "#FFFFFF",
}

SHARP_FONT_STACK = (
    "Bodoni 72, Bodoni MT, Didot, Georgia, Times New Roman, serif"
)

CORE_STATUS_HANDLE = None
CORE_STATUS_HISTORY = []
CORE_EVENTS = []

def safe_html(value):
    """Escape a value for HTML rendering."""
    return html.escape(str(value), quote=True)

def compact_path(value, max_len=95):
    """Shorten long paths for compact UI display."""
    value = str(value)
    if len(value) <= max_len:
        return value
    return value[:42] + "..." + value[-42:]

def unique_list(values):
    """Return unique values preserving order."""
    output = []
    seen = set()

    for value in values:
        if value not in seen:
            output.append(value)
            seen.add(value)

    return output

def render_core_panel(
    phase,
    message,
    progress,
    rows=None,
    warnings=None,
    blockers=None,
    done=False,
):
    """Render the single live core-analysis panel."""
    rows = rows or []
    warnings = unique_list(warnings or [])
    blockers = unique_list(blockers or [])

    progress_value = max(0, min(100, int(progress)))

    status_badge = "READY" if done and not blockers else ("FAILED" if blockers else "RUNNING")
    status_background = SHARP_COLORS["pink"] if done and not blockers else ("#FFD6D6" if blockers else SHARP_COLORS["soft"])
    status_color = SHARP_COLORS["deep"]

    history_html = ""
    if CORE_STATUS_HISTORY:
        history_html = "".join(
            f"""
            <div style="display:flex; gap:8px; align-items:center; padding:2px 0;">
              <span style="color:{SHARP_COLORS['pink']};">●</span>
              <span>{safe_html(item)}</span>
            </div>
            """
            for item in CORE_STATUS_HISTORY[-8:]
        )

    rows_html = ""
    if rows:
        row_items = []
        for key, value in rows:
            row_items.append(
                f"""
                <div style="display:grid; grid-template-columns:210px 1fr; gap:12px; padding:6px 0; border-bottom:1px solid rgba(255,255,255,0.12);">
                  <div style="font-weight:700; color:{SHARP_COLORS['soft']};">{safe_html(key)}</div>
                  <div style="color:{SHARP_COLORS['white']};"><code>{safe_html(compact_path(value))}</code></div>
                </div>
                """
            )

        rows_html = f"""
        <div style="margin-top:14px; font-size:13px;">
          {''.join(row_items)}
        </div>
        """

    warning_html = ""
    if warnings:
        warning_items = "".join(f"<li>{safe_html(item)}</li>" for item in warnings[:8])
        extra = ""
        if len(warnings) > 8:
            extra = f"<li>{safe_html(str(len(warnings) - 8) + ' additional warning(s) hidden for compact display.')}</li>"

        warning_html = f"""
        <div style="margin-top:14px; padding:10px 12px; border-radius:12px; background:rgba(255,255,255,0.12);">
          <div style="font-weight:800; color:{SHARP_COLORS['soft']}; margin-bottom:6px;">Pending optional steps</div>
          <ul style="margin:0; padding-left:20px; line-height:1.45;">{warning_items}{extra}</ul>
        </div>
        """

    blocker_html = ""
    if blockers:
        blocker_items = "".join(f"<li>{safe_html(item)}</li>" for item in blockers)
        blocker_html = f"""
        <div style="margin-top:14px; padding:10px 12px; border-radius:12px; background:#FFE4E4; color:#4D0022;">
          <div style="font-weight:900; margin-bottom:6px;">Core analysis issue</div>
          <ul style="margin:0; padding-left:20px; line-height:1.45;">{blocker_items}</ul>
        </div>
        """

    return HTML(f"""
    <div style="
      background:linear-gradient(135deg,{SHARP_COLORS['background']},{SHARP_COLORS['dark']});
      color:{SHARP_COLORS['white']};
      padding:20px 22px;
      border-radius:18px;
      border:1px solid rgba(255,255,255,0.22);
      box-shadow:0 10px 32px rgba(0,0,0,0.20);
      margin:12px 0;
      font-family:Inter,Arial,sans-serif;
    ">
      <div style="display:flex; justify-content:space-between; gap:16px; align-items:flex-start;">
        <div>
          <div style="font-family:{SHARP_FONT_STACK}; font-size:30px; letter-spacing:0.5px;">
            S(H)ARP
          </div>
          <div style="font-size:12px; color:{SHARP_COLORS['soft']}; text-transform:uppercase; letter-spacing:1.6px; margin-top:2px;">
            Core analysis
          </div>
        </div>
        <div style="
          background:{status_background};
          color:{status_color};
          border-radius:999px;
          padding:7px 12px;
          font-weight:900;
          font-size:12px;
          letter-spacing:1px;
        ">
          {safe_html(status_badge)}
        </div>
      </div>

      <div style="font-size:21px; font-weight:900; margin-top:14px;">
        {safe_html(phase)}
      </div>

      <div style="margin-top:8px; font-size:14px; line-height:1.45;">
        {safe_html(message)}
      </div>

      <div style="margin-top:14px;">
        <div style="font-size:12px; color:{SHARP_COLORS['soft']}; margin-bottom:6px;">
          Progress: <code>{progress_value}%</code>
        </div>
        <div style="height:10px; background:{SHARP_COLORS['deep']}; border-radius:999px; overflow:hidden;">
          <div style="height:10px; width:{progress_value}%; background:{SHARP_COLORS['pink']}; transition:width 0.25s ease;"></div>
        </div>
      </div>

      <div style="margin-top:14px; font-size:13px; color:{SHARP_COLORS['soft']};">
        {history_html}
      </div>

      {rows_html}
      {warning_html}
      {blocker_html}
    </div>
    """)

def update_core_status(
    phase,
    message,
    progress,
    rows=None,
    warnings=None,
    blockers=None,
    done=False,
    history_label=None,
):
    """Update the single live core-analysis panel."""
    global CORE_STATUS_HANDLE

    if history_label:
        if not CORE_STATUS_HISTORY or CORE_STATUS_HISTORY[-1] != history_label:
            CORE_STATUS_HISTORY.append(history_label)

    panel = render_core_panel(
        phase=phase,
        message=message,
        progress=progress,
        rows=rows,
        warnings=warnings,
        blockers=blockers,
        done=done,
    )

    if CORE_STATUS_HANDLE is None:
        CORE_STATUS_HANDLE = display(panel, display_id=True)
    else:
        CORE_STATUS_HANDLE.update(panel)

def record_core_event(step, status, detail=""):
    """Record a core-analysis event."""
    CORE_EVENTS.append({
        "time": time.strftime("%Y-%m-%d %H:%M:%S"),
        "step": str(step),
        "status": str(status),
        "detail": str(detail),
    })

update_core_status(
    phase="Starting core analysis",
    message="S(H)ARP is preparing annotation, evidence tables, neighborhoods, and candidate regions.",
    progress=3,
    history_label="Core analysis started",
)

# =====================================================================
# General helpers
# =====================================================================

def read_json_if_exists(path):
    """Read a JSON file if it exists."""
    path = Path(path)
    if not path.exists():
        return None

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None

def write_json(path, data):
    """Write formatted JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(data, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

def path_to_string(value):
    """Convert Path objects inside nested structures to strings."""
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {k: path_to_string(v) for k, v in value.items()}
    if isinstance(value, list):
        return [path_to_string(v) for v in value]
    return value

def file_sha256(path, block_size=1024 * 1024):
    """Compute SHA256 digest for a file."""
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()

def command_available(command_name):
    """Resolve a command using tool status, PATH, and known notebook envs."""
    if "SHARP_TOOL_STATUS" in globals():
        item = SHARP_TOOL_STATUS.get(command_name, {})
        candidate = item.get("path", "")

        if candidate and Path(candidate).exists():
            return str(candidate)

    detected = shutil.which(command_name)

    if detected:
        return detected

    legacy_candidates = [
        Path("/content/sharp_env/bin") / command_name,
        Path("/content/sharp_igem_usp_brazil_2026/envs/sharp_bio/bin") / command_name,
        Path("/content/sharp_igem_usp_brazil_2026/envs/bakta/bin") / command_name,
        Path("/content/sharp_igem_usp_brazil_2026/envs/meme/bin") / command_name,
    ]

    for candidate in legacy_candidates:
        if candidate.exists():
            return str(candidate)

    return None

def tail_text(text, max_lines=18):
    """Return the last lines of a text block."""
    return "\n".join(str(text).splitlines()[-max_lines:])

def run_cmd(cmd, step_label, shell=False, env=None, check=False, update_every=20):
    """Run an external command and write a log file."""
    LOG_DIR.mkdir(parents=True, exist_ok=True)

    safe_label = re.sub(r"[^A-Za-z0-9._-]+", "_", step_label.lower()).strip("_")
    log_file = LOG_DIR / f"{safe_label}_{int(time.time())}.log"

    printable = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))

    record_core_event(step_label, "running", printable)

    update_core_status(
        phase=step_label,
        message="External command is running. Logs are being captured for the final package.",
        progress=45,
        rows=[
            ("Command", printable),
            ("Log file", log_file),
        ],
        history_label=step_label,
    )

    start = time.time()

    with log_file.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            cmd,
            shell=shell,
            stdout=log_handle,
            stderr=subprocess.STDOUT,
            text=True,
            env=env,
        )

        while process.poll() is None:
            time.sleep(update_every)

        returncode = process.returncode

    elapsed = time.time() - start

    try:
        output_text = log_file.read_text(encoding="utf-8", errors="replace")
    except Exception:
        output_text = ""

    if returncode == 0:
        record_core_event(step_label, "ok", f"log={log_file}; elapsed={elapsed:.0f}s")
    else:
        record_core_event(
            step_label,
            "failed",
            f"returncode={returncode}; log={log_file}; tail={tail_text(output_text)}",
        )

    if check and returncode != 0:
        raise RuntimeError(
            f"Command failed: {printable}\n"
            f"Log file: {log_file}\n"
            f"Last log lines:\n{tail_text(output_text)}"
        )

    return SimpleNamespace(
        returncode=returncode,
        stdout=output_text,
        log_file=str(log_file),
        elapsed=elapsed,
    )

def parse_gff_attributes(attr_text):
    """Parse GFF3 attributes into a dictionary."""
    result = {}

    for item in str(attr_text).split(";"):
        item = item.strip()

        if not item:
            continue

        if "=" in item:
            key, value = item.split("=", 1)
        elif " " in item:
            key, value = item.split(" ", 1)
        else:
            continue

        result[key.strip()] = value.strip().strip('"')

    return result

def clean_gff_attr(value):
    """Clean a value for GFF3 attribute output."""
    value = "" if value is None else str(value)
    value = value.replace(";", ",")
    value = value.replace("=", ":")
    value = value.replace("\t", " ")
    return value

def normalize_identifier(value):
    """Normalize a sequence or feature identifier."""
    if value is None:
        return ""

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    value = str(value).strip()

    if not value:
        return ""

    value = value.replace("cds-", "")
    value = value.replace("gene-", "")

    return value.strip()

def candidate_ids_from_values(values):
    """Generate candidate IDs from raw values."""
    ids = set()

    for value in values:
        if value is None:
            continue

        try:
            if pd.isna(value):
                continue
        except Exception:
            pass

        value = str(value).strip()

        if not value:
            continue

        candidates = [value]
        candidates.append(value.replace("cds-", ""))
        candidates.append(value.replace("gene-", ""))

        if "|" in value:
            candidates.extend([x.strip() for x in value.split("|") if x.strip()])

        if " " in value:
            candidates.append(value.split()[0].strip())

        for candidate in candidates:
            normalized = normalize_identifier(candidate)

            if normalized:
                ids.add(candidate)
                ids.add(normalized)

    return ids

def build_id_to_pid_map(cds_df):
    """Build a mapping from external identifiers to standardized S(H)ARP PIDs."""
    id_to_pid = {}

    candidate_cols = [
        "pid",
        "pid_original",
        "protein_fasta_id",
        "protein_id",
        "locus_tag",
        "gene",
        "id_attr",
        "name_attr",
    ]

    for _, row in cds_df.iterrows():
        pid = str(row.get("pid", "")).strip()

        if not pid:
            continue

        values = []

        for col in candidate_cols:
            if col in cds_df.columns:
                values.append(row.get(col, ""))

        for candidate in candidate_ids_from_values(values):
            id_to_pid[candidate] = pid
            id_to_pid[normalize_identifier(candidate)] = pid

    return id_to_pid

def map_external_id_to_pid(raw_id, id_to_pid):
    """Map an external identifier to a standardized PID."""
    if raw_id is None:
        return None

    raw_id = str(raw_id).strip()

    if not raw_id:
        return None

    candidates = [raw_id, normalize_identifier(raw_id)]
    candidates.extend(sorted(candidate_ids_from_values([raw_id])))

    for candidate in candidates:
        if candidate in id_to_pid:
            return id_to_pid[candidate]

    return None

def contig_lengths_from_fasta(fasta_path):
    """Build a contig table from FASTA."""
    rows = []

    for record in SeqIO.parse(str(fasta_path), "fasta"):
        rows.append({
            "nucleotide": record.id,
            "nlen": len(record.seq),
            "description": record.description,
        })

    return pd.DataFrame(rows)

def count_real_rows(path):
    """Count real rows in a TSV, treating pending tables as zero."""
    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return 0

    try:
        df = pd.read_csv(path, sep="\t")
    except Exception:
        return 0

    if df.empty:
        return 0

    if "status" in df.columns:
        statuses = set(df["status"].dropna().astype(str).str.lower())
        if statuses and statuses <= {"pending"}:
            return 0

    return int(len(df))

# =====================================================================
# Resolve state from Cell 2
# =====================================================================

PROJECT_DIR = Path(globals().get("PROJECT_DIR", "/content/sharp_igem_usp_brazil_2026")).resolve()
CONFIG_DIR = Path(globals().get("CONFIG_DIR", PROJECT_DIR / "config")).resolve()
SHARP_CONTEXT_FILE = Path(globals().get("SHARP_CONTEXT_FILE", CONFIG_DIR / "sharp_notebook_context.json"))

context = read_json_if_exists(SHARP_CONTEXT_FILE) or {}

DATA_DIR = Path(globals().get("DATA_DIR", context.get("data_dir", PROJECT_DIR / "data"))).resolve()
INPUT_DIR = Path(globals().get("INPUT_DIR", context.get("input_dir", DATA_DIR / "input"))).resolve()
NORMALIZED_DIR = Path(globals().get("NORMALIZED_DIR", context.get("normalized_dir", DATA_DIR / "normalized"))).resolve()
DATABASES_DIR = Path(globals().get("DATABASES_DIR", context.get("databases_dir", PROJECT_DIR / "databases"))).resolve()
RESULTS_DIR = Path(globals().get("RESULTS_DIR", context.get("results_dir", PROJECT_DIR / "results"))).resolve()

RUN_NAME = str(globals().get("RUN_NAME", context.get("run_name", "sharp_run_01")))
RUN_DIR = Path(globals().get("RUN_DIR", context.get("run_dir", RESULTS_DIR / RUN_NAME))).resolve()
LOG_DIR = Path(globals().get("LOG_DIR", context.get("log_dir", RUN_DIR / "logs"))).resolve()
TABLE_DIR = Path(globals().get("TABLE_DIR", context.get("table_dir", RUN_DIR / "tables"))).resolve()
REPORT_DIR = Path(globals().get("REPORT_DIR", context.get("report_dir", RUN_DIR / "report"))).resolve()

ANNOTATION_DIR = Path(globals().get("ANNOTATION_DIR", RUN_DIR / "annotation")).resolve()
FIMO_DIR = RUN_DIR / "fimo"
HMM_DIR_RUN = RUN_DIR / "hmm"
DOMAIN_DIR_RUN = RUN_DIR / "domains"
EMBEDDING_DIR_RUN = RUN_DIR / "embeddings"
BAKTA_OUTPUT_DIR = RUN_DIR / "bakta"
TMP_DIR = RUN_DIR / "tmp"

for directory in [
    PROJECT_DIR,
    DATA_DIR,
    INPUT_DIR,
    NORMALIZED_DIR,
    CONFIG_DIR,
    DATABASES_DIR,
    RESULTS_DIR,
    RUN_DIR,
    LOG_DIR,
    TABLE_DIR,
    REPORT_DIR,
    ANNOTATION_DIR,
    FIMO_DIR,
    HMM_DIR_RUN,
    DOMAIN_DIR_RUN,
    EMBEDDING_DIR_RUN,
    BAKTA_OUTPUT_DIR,
    TMP_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

workflow = str(globals().get("workflow", context.get("workflow", "")))
input_mode = str(globals().get("input_mode", context.get("input_mode", "")))
annotation_mode = str(globals().get("annotation_mode", context.get("annotation_mode", "")))
annotation_backend = str(globals().get("annotation_backend", context.get("annotation_backend", "")))

SHARP_BACKEND = dict(globals().get("SHARP_BACKEND", {}))
if not annotation_backend:
    annotation_backend = str(SHARP_BACKEND.get("annotation_backend", ""))

SHARP_DEPENDENCIES_READY = bool(globals().get("SHARP_DEPENDENCIES_READY", context.get("dependencies_ready", False)))
SHARP_DEPENDENCY_STATUS = dict(globals().get("SHARP_DEPENDENCY_STATUS", {}))
SHARP_TOOL_STATUS = dict(globals().get("SHARP_TOOL_STATUS", {}))
SHARP_PYTHON_STATUS = dict(globals().get("SHARP_PYTHON_STATUS", {}))

GENOME_FASTA = Path(
    globals().get("GENOME_FASTA", context.get("genome_fasta", ""))
).resolve()

annotation_file_context = (
    globals().get("ANNOTATION_FILE")
    or globals().get("GENOME_ANNOTATION")
    or context.get("annotation_file", "")
)
protein_fasta_context = (
    globals().get("PROTEIN_FASTA")
    or context.get("protein_fasta", "")
)
cds_fasta_context = (
    globals().get("CDS_FASTA")
    or context.get("cds_fasta", "")
)

ANNOTATION_FILE = Path(annotation_file_context).resolve() if annotation_file_context else None
PROTEIN_FASTA = Path(protein_fasta_context).resolve() if protein_fasta_context else None
CDS_FASTA = Path(cds_fasta_context).resolve() if cds_fasta_context else None

BAKTA_BIN = str(globals().get("BAKTA_BIN", SHARP_BACKEND.get("bakta_bin", "")))
BAKTA_DB_PATH = str(globals().get("BAKTA_DB_PATH", SHARP_BACKEND.get("bakta_db_path", context.get("bakta_db_path", ""))))
BAKTA_DB_READY = bool(globals().get("BAKTA_DB_READY", SHARP_BACKEND.get("bakta_db_ready", context.get("bakta_db_ready", False))))
BAKTA_RUN_PATH_PREFIX = str(globals().get("BAKTA_RUN_PATH_PREFIX", SHARP_BACKEND.get("bakta_run_path_prefix", "")))
BAKTA_RUNTIME_ARGS = list(globals().get("BAKTA_RUNTIME_ARGS", SHARP_BACKEND.get("bakta_runtime_args", [])))

SHARP_INTERNAL_RESOURCES = dict(globals().get("SHARP_INTERNAL_RESOURCES", {}))
SHARP_INTERNAL_RESOURCES.setdefault("heptamer_meme", str(DATABASES_DIR / "motifs" / "heptarepeats2.meme"))
SHARP_INTERNAL_RESOURCES.setdefault("sarp_hmm", str(DATABASES_DIR / "hmm" / "sarp_custom.hmm"))
SHARP_INTERNAL_RESOURCES.setdefault("domain_models_hmm", str(DATABASES_DIR / "domain_models" / "domain_models.hmm"))
SHARP_INTERNAL_RESOURCES.setdefault("embedding_reference", str(DATABASES_DIR / "embeddings" / "reference_embeddings.parquet"))
SHARP_INTERNAL_RESOURCES.setdefault("scoring_config", str(CONFIG_DIR / "sharp_scoring_config.json"))
SHARP_INTERNAL_RESOURCES.setdefault("report_module", str(PROJECT_DIR / "resources" / "modules" / "operon_fig_colab.py"))

SHARP_SAMPLE = dict(globals().get("SHARP_SAMPLE", {}))
SHARP_SAMPLE.setdefault("analysis_name", RUN_NAME)
SHARP_SAMPLE.setdefault("organism_name", globals().get("DEFAULT_ORGANISM_NAME", context.get("organism_name", "Streptomyces sp.")))
SHARP_SAMPLE.setdefault("strain_name", globals().get("DEFAULT_STRAIN_NAME", context.get("strain_name", "unknown")))

if not GENOME_FASTA.exists():
    raise RuntimeError(f"Genome FASTA was not found: {GENOME_FASTA}")

update_core_status(
    phase="Core inputs resolved",
    message="S(H)ARP resolved the setup context from Cell 2.",
    progress=8,
    rows=[
        ("Workflow", workflow),
        ("Input mode", input_mode),
        ("Annotation backend", annotation_backend),
        ("Genome FASTA", GENOME_FASTA),
        ("Bakta DB", BAKTA_DB_PATH or "not required"),
    ],
    history_label="Setup context loaded",
)

# =====================================================================
# Output paths
# =====================================================================

GENOME_SHA256 = file_sha256(GENOME_FASTA)
GENOME_HASH_SHORT = GENOME_SHA256[:16]

BAKTA_RUN_DIR = BAKTA_OUTPUT_DIR / GENOME_HASH_SHORT
BAKTA_RUN_DIR.mkdir(parents=True, exist_ok=True)

SHARP_ANALYSIS_OUTPUTS = {
    "standard_genome_fasta": ANNOTATION_DIR / "sharp_genome.fna",
    "annotation_file": ANNOTATION_DIR / "sharp_annotation.gff3",
    "annotation_genbank": ANNOTATION_DIR / "sharp_annotation.gbff",
    "protein_fasta": ANNOTATION_DIR / "sharp_proteins.faa",
    "cds_nucleotide_fasta": ANNOTATION_DIR / "sharp_cds.ffn",
    "cds_table": ANNOTATION_DIR / "sharp_cds_table.tsv",
    "contig_table": ANNOTATION_DIR / "sharp_contigs.tsv",
    "id_crosswalk": ANNOTATION_DIR / "sharp_id_crosswalk.tsv",

    "fimo_hits": FIMO_DIR / "sharp_fimo_hits.tsv",
    "fimo_to_cds": FIMO_DIR / "sharp_fimo_to_cds.tsv",
    "fimo_repeat_hits": FIMO_DIR / "sharp_fimo_repeat_hits.tsv",
    "fimo_regulatory_hits": FIMO_DIR / "sharp_fimo_regulatory_hits.tsv",

    "sarp_hits": HMM_DIR_RUN / "sharp_sarp_hmm_hits.tsv",
    "domain_hits": DOMAIN_DIR_RUN / "sharp_domain_hits.tsv",
    "embedding_scores": EMBEDDING_DIR_RUN / "sharp_embedding_scores.tsv",

    "neighborhood_seed_table": TABLE_DIR / "sharp_neighborhood_seed_table.tsv",
    "neighborhood_index": TABLE_DIR / "sharp_neighborhood_index.tsv",
    "neighborhood_evidence_links": TABLE_DIR / "sharp_neighborhood_evidence_links.tsv",
    "ndf": TABLE_DIR / "sharp_neighborhood_dataframe.tsv",
    "regions": TABLE_DIR / "sharp_predicted_regions.tsv",

    "pipeline_manifest": RUN_DIR / f"{SHARP_SAMPLE['analysis_name']}_pipeline_manifest.json",
}

# =====================================================================
# Annotation helpers
# =====================================================================

def copy_genome_to_standard_path():
    """Copy the input genome to the standardized output path."""
    destination = SHARP_ANALYSIS_OUTPUTS["standard_genome_fasta"]

    if GENOME_FASTA.resolve() != destination.resolve():
        shutil.copy2(GENOME_FASTA, destination)

    return destination

def locate_bakta_outputs(output_dir, prefix):
    """Locate Bakta outputs in a directory."""
    output_dir = Path(output_dir)

    def first_existing(candidates):
        for candidate in candidates:
            candidate = Path(candidate)
            if candidate.exists() and candidate.stat().st_size > 0:
                return candidate
        return None

    return {
        "gff": first_existing([
            output_dir / f"{prefix}.gff3",
            output_dir / f"{prefix}.gff",
            *sorted(output_dir.glob("*.gff3")),
            *sorted(output_dir.glob("*.gff")),
        ]),
        "gbff": first_existing([
            output_dir / f"{prefix}.gbff",
            output_dir / f"{prefix}.gbk",
            output_dir / f"{prefix}.gb",
            *sorted(output_dir.glob("*.gbff")),
            *sorted(output_dir.glob("*.gbk")),
            *sorted(output_dir.glob("*.gb")),
        ]),
        "faa": first_existing([
            output_dir / f"{prefix}.faa",
            *sorted(output_dir.glob("*.faa")),
        ]),
        "ffn": first_existing([
            output_dir / f"{prefix}.ffn",
            *sorted(output_dir.glob("*.ffn")),
        ]),
        "tsv": first_existing([
            output_dir / f"{prefix}.tsv",
            *sorted(output_dir.glob("*.tsv")),
        ]),
    }

def run_bakta_annotation(genome_fasta):
    """Run Bakta using the cached DB prepared by Cell 2."""
    bakta_bin = BAKTA_BIN or command_available("bakta")
    bakta_db = BAKTA_DB_PATH or SHARP_BACKEND.get("bakta_db_path", "") or os.environ.get("BAKTA_DB", "")

    if not bakta_bin:
        raise RuntimeError("Bakta executable was not found. Rerun Cell 2.")

    if not bakta_db or not Path(bakta_db).exists():
        raise RuntimeError("Bakta DB was not found. Rerun Cell 2.")

    os.environ["BAKTA_DB"] = str(bakta_db)

    BAKTA_RUN_DIR.mkdir(parents=True, exist_ok=True)

    existing = locate_bakta_outputs(BAKTA_RUN_DIR, BAKTA_PREFIX)

    if existing["gff"] and existing["faa"]:
        record_core_event(
            "Bakta annotation",
            "skipped",
            f"existing Bakta outputs detected for genome hash {GENOME_HASH_SHORT}",
        )
        return existing

    if BAKTA_RUN_DIR.exists():
        partial_outputs = locate_bakta_outputs(BAKTA_RUN_DIR, BAKTA_PREFIX)

        if not partial_outputs["gff"] or not partial_outputs["faa"]:
            shutil.rmtree(BAKTA_RUN_DIR)

    BAKTA_RUN_DIR.mkdir(parents=True, exist_ok=True)

    organism = SHARP_SAMPLE.get("organism_name", "")
    strain = SHARP_SAMPLE.get("strain_name", "")

    genus = ""
    if organism and organism.lower() != "unknown":
        first = organism.split()[0].strip()

        if re.match(r"^[A-Za-z][A-Za-z_-]+$", first):
            genus = first

    cmd = [
        str(bakta_bin),
        "--db",
        str(bakta_db),
        "--output",
        str(BAKTA_RUN_DIR),
        "--prefix",
        BAKTA_PREFIX,
        "--threads",
        str(BAKTA_THREADS),
        "--force",
    ]

    for arg in BAKTA_RUNTIME_ARGS:
        if arg and str(arg).strip():
            cmd.append(str(arg).strip())

    if genus:
        cmd.extend(["--genus", genus])

    if strain and strain.lower() != "unknown":
        cmd.extend(["--strain", strain])

    cmd.append(str(genome_fasta))

    bakta_env = os.environ.copy()
    bakta_env["BAKTA_DB"] = str(bakta_db)
    bakta_env["MPLBACKEND"] = "Agg"

    mplconfig_dir = TMP_DIR / "matplotlib"
    mplconfig_dir.mkdir(parents=True, exist_ok=True)
    bakta_env["MPLCONFIGDIR"] = str(mplconfig_dir)

    path_parts = []

    if BAKTA_RUN_PATH_PREFIX:
        path_parts.append(str(BAKTA_RUN_PATH_PREFIX))

    path_parts.append(str(Path(bakta_bin).parent))
    path_parts.append(bakta_env.get("PATH", ""))

    bakta_env["PATH"] = os.pathsep.join(path_parts)

    update_core_status(
        phase="Running Bakta annotation",
        message="Bakta is annotating the genome using the cached light database.",
        progress=30,
        rows=[
            ("Bakta", bakta_bin),
            ("Bakta DB", bakta_db),
            ("Output", BAKTA_RUN_DIR),
            ("Threads", BAKTA_THREADS),
        ],
        history_label="Bakta annotation started",
    )

    run_cmd(
        cmd,
        step_label="Running Bakta annotation",
        shell=False,
        env=bakta_env,
        check=True,
        update_every=30,
    )

    outputs = locate_bakta_outputs(BAKTA_RUN_DIR, BAKTA_PREFIX)

    if not outputs["gff"] or not outputs["faa"]:
        raise RuntimeError("Bakta finished, but GFF/protein FASTA outputs were not detected.")

    return outputs

def parse_gff_to_cds_table(gff_path, contig_table):
    """Parse CDS features from GFF3."""
    contig_lengths = dict(zip(contig_table["nucleotide"], contig_table["nlen"]))

    rows = []

    with open(gff_path, "r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if not line.strip() or line.startswith("#"):
                continue

            parts = line.rstrip("\n").split("\t")

            if len(parts) < 9:
                continue

            seqid, source, feature_type, start, end, score, strand, phase, attrs = parts

            if feature_type.lower() != "cds":
                continue

            attr = parse_gff_attributes(attrs)

            id_attr = attr.get("ID", "")
            name_attr = attr.get("Name", "")

            raw_id = (
                attr.get("protein_id")
                or attr.get("locus_tag")
                or id_attr
                or name_attr
                or f"cds_{len(rows) + 1:06d}"
            )

            rows.append({
                "pid_original": normalize_identifier(raw_id),
                "id_attr": id_attr,
                "name_attr": name_attr,
                "nucleotide": seqid,
                "start": int(start),
                "end": int(end),
                "strand": 1 if strand == "+" else -1 if strand == "-" else 0,
                "gene": attr.get("gene", ""),
                "locus_tag": attr.get("locus_tag", ""),
                "protein_id": attr.get("protein_id", ""),
                "product": attr.get("product", ""),
                "source": source,
                "feature_type": feature_type,
                "nlen": contig_lengths.get(seqid, None),
                "organism": SHARP_SAMPLE.get("organism_name", "unknown"),
            })

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError(f"No CDS features were parsed from GFF: {gff_path}")

    df = df.sort_values(["nucleotide", "start", "end"]).reset_index(drop=True)
    df["gene_index"] = df.groupby("nucleotide").cumcount()
    df["pid"] = [f"sharp_prot_{i + 1:06d}" for i in range(len(df))]
    df["protein_fasta_id"] = ""

    return df

def parse_genbank_to_cds_table_and_gff(genbank_path, contig_table, output_gff):
    """Parse CDS features from GenBank and emit standardized GFF3."""
    contig_lengths = dict(zip(contig_table["nucleotide"], contig_table["nlen"]))

    rows = []
    gff_lines = ["##gff-version 3"]

    for record in SeqIO.parse(str(genbank_path), "genbank"):
        seqid = record.id
        nlen = contig_lengths.get(seqid, len(record.seq))

        for feature in record.features:
            if feature.type.lower() != "cds":
                continue

            q = feature.qualifiers

            start = int(feature.location.start) + 1
            end = int(feature.location.end)
            strand_value = feature.location.strand
            strand_symbol = "+" if strand_value == 1 else "-" if strand_value == -1 else "."
            strand = 1 if strand_value == 1 else -1 if strand_value == -1 else 0

            gene = q.get("gene", [""])[0]
            locus_tag = q.get("locus_tag", [""])[0]
            protein_id = q.get("protein_id", [""])[0]
            product = q.get("product", [""])[0]

            raw_id = protein_id or locus_tag or gene or f"cds_{len(rows) + 1:06d}"

            id_attr = raw_id
            name_attr = gene or locus_tag or protein_id

            rows.append({
                "pid_original": normalize_identifier(raw_id),
                "id_attr": id_attr,
                "name_attr": name_attr,
                "nucleotide": seqid,
                "start": start,
                "end": end,
                "strand": strand,
                "gene": gene,
                "locus_tag": locus_tag,
                "protein_id": protein_id,
                "product": product,
                "source": "genbank",
                "feature_type": "CDS",
                "nlen": nlen,
                "organism": SHARP_SAMPLE.get("organism_name", "unknown"),
            })

            attrs = [
                f"ID={clean_gff_attr(id_attr)}",
                f"Name={clean_gff_attr(name_attr)}",
                f"locus_tag={clean_gff_attr(locus_tag)}",
                f"protein_id={clean_gff_attr(protein_id)}",
                f"product={clean_gff_attr(product)}",
            ]

            gff_lines.append(
                "\t".join([
                    seqid,
                    "genbank",
                    "CDS",
                    str(start),
                    str(end),
                    ".",
                    strand_symbol,
                    "0",
                    ";".join(attrs),
                ])
            )

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError(f"No CDS features were parsed from GenBank: {genbank_path}")

    df = df.sort_values(["nucleotide", "start", "end"]).reset_index(drop=True)
    df["gene_index"] = df.groupby("nucleotide").cumcount()
    df["pid"] = [f"sharp_prot_{i + 1:06d}" for i in range(len(df))]
    df["protein_fasta_id"] = ""

    output_gff.write_text("\n".join(gff_lines) + "\n", encoding="utf-8")

    return df

def standardize_protein_fasta(source_faa, cds_table, output_faa):
    """Write standardized protein FASTA and update CDS table with protein FASTA IDs."""
    source_faa = Path(source_faa)
    output_faa = Path(output_faa)

    if not source_faa.exists():
        raise FileNotFoundError(f"Protein FASTA not found: {source_faa}")

    records = list(SeqIO.parse(str(source_faa), "fasta"))
    cds_table = cds_table.copy()

    id_to_index = {}

    for idx, row in cds_table.iterrows():
        values = [
            row.get("pid_original", ""),
            row.get("protein_id", ""),
            row.get("locus_tag", ""),
            row.get("gene", ""),
            row.get("id_attr", ""),
            row.get("name_attr", ""),
        ]

        for candidate in candidate_ids_from_values(values):
            if candidate not in id_to_index:
                id_to_index[candidate] = idx

    matched_indices = set()
    record_to_pid = {}

    for record in records:
        rec_id = str(record.id).strip()
        idx = None

        for candidate in candidate_ids_from_values([rec_id, record.description]):
            if candidate in id_to_index:
                idx = id_to_index[candidate]
                break

        if idx is not None and idx not in matched_indices:
            matched_indices.add(idx)
            pid = cds_table.loc[idx, "pid"]
            record_to_pid[record.id] = pid
            cds_table.loc[idx, "protein_fasta_id"] = rec_id

    if len(records) == len(cds_table) and len(matched_indices) < max(1, int(0.5 * len(records))):
        record_to_pid = {}
        cds_table["protein_fasta_id"] = ""

        for record, idx in zip(records, cds_table.index):
            pid = cds_table.loc[idx, "pid"]
            record_to_pid[record.id] = pid
            cds_table.loc[idx, "protein_fasta_id"] = str(record.id).strip()

    with output_faa.open("w", encoding="utf-8") as handle:
        for record in records:
            pid = record_to_pid.get(record.id, normalize_identifier(record.id) or record.id)
            record.id = pid
            record.name = pid
            record.description = pid
            SeqIO.write(record, handle, "fasta")

    return output_faa, cds_table

def standardize_cds_fasta(source_ffn, cds_table, output_ffn):
    """Write standardized nucleotide CDS FASTA when available."""
    source_ffn = Path(source_ffn) if source_ffn else None
    output_ffn = Path(output_ffn)

    if source_ffn is None or not source_ffn.exists():
        output_ffn.write_text("", encoding="utf-8")
        return output_ffn

    records = list(SeqIO.parse(str(source_ffn), "fasta"))

    if len(records) == len(cds_table):
        with output_ffn.open("w", encoding="utf-8") as handle:
            for record, (_, row) in zip(records, cds_table.iterrows()):
                record.id = row["pid"]
                record.name = row["pid"]
                record.description = row["pid"]
                SeqIO.write(record, handle, "fasta")
    else:
        shutil.copy2(source_ffn, output_ffn)

    return output_ffn

def obtain_annotation_contract():
    """Build the standardized S(H)ARP annotation contract."""
    genome_fasta = copy_genome_to_standard_path()
    contig_table = contig_lengths_from_fasta(genome_fasta)

    contig_table.to_csv(
        SHARP_ANALYSIS_OUTPUTS["contig_table"],
        sep="\t",
        index=False,
    )

    if annotation_mode == "use_existing_annotation" or annotation_backend == "existing_annotation":
        annotation_source = "existing_annotation"

        if ANNOTATION_FILE is None:
            raise FileNotFoundError("Existing annotation file was not provided.")

        if PROTEIN_FASTA is None:
            raise FileNotFoundError("Existing protein FASTA was not provided.")

        source_annotation = Path(ANNOTATION_FILE)
        source_faa = Path(PROTEIN_FASTA)
        source_ffn = Path(CDS_FASTA) if CDS_FASTA and Path(CDS_FASTA).exists() else None

        if not source_annotation.exists():
            raise FileNotFoundError(f"Existing annotation file not found: {source_annotation}")

        if not source_faa.exists():
            raise FileNotFoundError(f"Existing protein FASTA not found: {source_faa}")

        lower = source_annotation.name.lower()

        if lower.endswith((".gff", ".gff3")):
            shutil.copy2(source_annotation, SHARP_ANALYSIS_OUTPUTS["annotation_file"])

            cds_table = parse_gff_to_cds_table(
                SHARP_ANALYSIS_OUTPUTS["annotation_file"],
                contig_table,
            )

        elif lower.endswith((".gb", ".gbk", ".gbff", ".genbank")):
            shutil.copy2(source_annotation, SHARP_ANALYSIS_OUTPUTS["annotation_genbank"])

            cds_table = parse_genbank_to_cds_table_and_gff(
                source_annotation,
                contig_table,
                SHARP_ANALYSIS_OUTPUTS["annotation_file"],
            )

        else:
            raise RuntimeError("Unsupported annotation format. Use GFF3 or GenBank.")

    elif annotation_backend in {"bakta", "bakta_cached_light_db", "bakta_colab_ready", "bakta_completed"}:
        annotation_source = annotation_backend

        bakta_outputs = run_bakta_annotation(genome_fasta)

        shutil.copy2(bakta_outputs["gff"], SHARP_ANALYSIS_OUTPUTS["annotation_file"])

        if bakta_outputs.get("gbff"):
            shutil.copy2(bakta_outputs["gbff"], SHARP_ANALYSIS_OUTPUTS["annotation_genbank"])
        else:
            SHARP_ANALYSIS_OUTPUTS["annotation_genbank"].write_text("", encoding="utf-8")

        source_faa = bakta_outputs["faa"]
        source_ffn = bakta_outputs["ffn"]

        cds_table = parse_gff_to_cds_table(
            SHARP_ANALYSIS_OUTPUTS["annotation_file"],
            contig_table,
        )

    else:
        raise RuntimeError(
            "No valid annotation backend is available. "
            f"Current backend: {annotation_backend or SHARP_BACKEND.get('annotation_backend')}"
        )

    _, cds_table = standardize_protein_fasta(
        source_faa=source_faa,
        cds_table=cds_table,
        output_faa=SHARP_ANALYSIS_OUTPUTS["protein_fasta"],
    )

    standardize_cds_fasta(
        source_ffn=source_ffn,
        cds_table=cds_table,
        output_ffn=SHARP_ANALYSIS_OUTPUTS["cds_nucleotide_fasta"],
    )

    cds_table.to_csv(
        SHARP_ANALYSIS_OUTPUTS["cds_table"],
        sep="\t",
        index=False,
    )

    crosswalk_cols = [
        "pid",
        "pid_original",
        "protein_fasta_id",
        "id_attr",
        "name_attr",
        "nucleotide",
        "start",
        "end",
        "strand",
        "locus_tag",
        "protein_id",
        "gene",
        "product",
    ]

    for col in crosswalk_cols:
        if col not in cds_table.columns:
            cds_table[col] = ""

    cds_table[crosswalk_cols].to_csv(
        SHARP_ANALYSIS_OUTPUTS["id_crosswalk"],
        sep="\t",
        index=False,
    )

    contract = {
        "source": annotation_source,
        "workflow": workflow,
        "input_mode": input_mode,
        "annotation_mode": annotation_mode,
        "annotation_backend": annotation_backend,
        "genome_sha256": GENOME_SHA256,
        "genome_hash_short": GENOME_HASH_SHORT,
        "genome_fasta": str(SHARP_ANALYSIS_OUTPUTS["standard_genome_fasta"]),
        "annotation_file": str(SHARP_ANALYSIS_OUTPUTS["annotation_file"]),
        "annotation_genbank": str(SHARP_ANALYSIS_OUTPUTS["annotation_genbank"]),
        "protein_fasta": str(SHARP_ANALYSIS_OUTPUTS["protein_fasta"]),
        "cds_nucleotide_fasta": str(SHARP_ANALYSIS_OUTPUTS["cds_nucleotide_fasta"]),
        "cds_table": str(SHARP_ANALYSIS_OUTPUTS["cds_table"]),
        "contig_table": str(SHARP_ANALYSIS_OUTPUTS["contig_table"]),
        "id_crosswalk": str(SHARP_ANALYSIS_OUTPUTS["id_crosswalk"]),
        "bakta_run_dir": str(BAKTA_RUN_DIR),
        "bakta_db_path": str(BAKTA_DB_PATH),
        "n_cds": int(len(cds_table)),
        "n_contigs": int(len(contig_table)),
    }

    return contract, cds_table, contig_table

# =====================================================================
# Evidence steps
# =====================================================================

def write_pending_table(path, columns, reason):
    """Write a standardized pending table."""
    df = pd.DataFrame([{col: "" for col in columns}])
    df["status"] = "pending"
    df["reason"] = reason
    df.to_csv(path, sep="\t", index=False)
    return df

def write_empty_table(path, columns):
    """Write an empty TSV table with fixed columns."""
    df = pd.DataFrame(columns=columns)
    df.to_csv(path, sep="\t", index=False)
    return df

def run_fimo_step(genome_fasta):
    """Run FIMO motif search when available and preserve all raw hits."""
    motif_file = Path(SHARP_INTERNAL_RESOURCES.get("heptamer_meme", ""))
    output_path = SHARP_ANALYSIS_OUTPUTS["fimo_hits"]

    columns = [
        "fimo_hit_id",
        "motif_id",
        "motif_alt_id",
        "sequence_name",
        "start",
        "stop",
        "strand",
        "score",
        "p_value",
        "q_value",
        "matched_sequence",
    ]

    if not motif_file.exists():
        record_core_event("FIMO", "pending", f"missing motif file: {motif_file}")
        return write_pending_table(
            output_path,
            columns,
            "S(H)ARP motif file is not loaded.",
        )

    fimo_bin = command_available("fimo")

    if not fimo_bin:
        record_core_event("FIMO", "pending", "fimo executable not found")
        return write_pending_table(
            output_path,
            columns,
            "FIMO executable not found.",
        )

    result = run_cmd(
        [
            fimo_bin,
            "--text",
            "--thresh",
            str(FIMO_PVALUE_THRESHOLD),
            str(motif_file),
            str(genome_fasta),
        ],
        step_label="Running FIMO motif search",
        check=False,
        update_every=20,
    )

    rows = []

    for line in result.stdout.splitlines():
        if not line.strip() or line.startswith("#"):
            continue

        parts = line.rstrip("\n").split("\t")

        if len(parts) < 10:
            continue

        if parts[0] == "motif_id":
            continue

        rows.append({
            "fimo_hit_id": f"fimo_{len(rows) + 1:08d}",
            "motif_id": parts[0],
            "motif_alt_id": parts[1],
            "sequence_name": parts[2],
            "start": parts[3],
            "stop": parts[4],
            "strand": parts[5],
            "score": parts[6],
            "p_value": parts[7],
            "q_value": parts[8],
            "matched_sequence": parts[9],
        })

    df = pd.DataFrame(rows, columns=columns)

    for col in ["start", "stop", "score", "p_value", "q_value"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df.to_csv(output_path, sep="\t", index=False)

    record_core_event("FIMO", "ok", f"{len(df)} raw motif hit(s)")

    return df

def run_sarp_hmm_step(protein_fasta, id_to_pid):
    """Run SARP HMM search when HMM resource is available."""
    sarp_hmm = Path(SHARP_INTERNAL_RESOURCES.get("sarp_hmm", ""))
    output_path = SHARP_ANALYSIS_OUTPUTS["sarp_hits"]

    columns = [
        "target_name",
        "target_pid",
        "target_accession",
        "query_name",
        "query_accession",
        "full_evalue",
        "full_score",
        "full_bias",
        "description",
    ]

    if not sarp_hmm.exists():
        record_core_event("SARP HMM", "pending", f"missing HMM file: {sarp_hmm}")
        return write_pending_table(
            output_path,
            columns,
            "S(H)ARP SARP HMM file is not loaded.",
        )

    hmmsearch_bin = command_available("hmmsearch")

    if not hmmsearch_bin:
        record_core_event("SARP HMM", "pending", "hmmsearch executable not found")
        return write_pending_table(
            output_path,
            columns,
            "hmmsearch executable not found.",
        )

    tblout = HMM_DIR_RUN / "sharp_sarp_hmmsearch.tblout"

    run_cmd(
        [
            hmmsearch_bin,
            "--tblout",
            str(tblout),
            "--noali",
            "-E",
            str(SARP_HMM_EVALUE_THRESHOLD),
            str(sarp_hmm),
            str(protein_fasta),
        ],
        step_label="Running SARP HMM search",
        check=False,
        update_every=20,
    )

    rows = []

    if tblout.exists():
        with open(tblout, "r", encoding="utf-8", errors="replace") as handle:
            for line in handle:
                if not line.strip() or line.startswith("#"):
                    continue

                parts = line.strip().split(maxsplit=18)

                if len(parts) < 6:
                    continue

                target_name = parts[0]
                target_pid = map_external_id_to_pid(target_name, id_to_pid)

                rows.append({
                    "target_name": target_name,
                    "target_pid": target_pid or "",
                    "target_accession": parts[1],
                    "query_name": parts[2],
                    "query_accession": parts[3],
                    "full_evalue": parts[4],
                    "full_score": parts[5],
                    "full_bias": parts[6] if len(parts) > 6 else "",
                    "description": parts[18] if len(parts) > 18 else "",
                })

    df = pd.DataFrame(rows, columns=columns)

    if not df.empty:
        df["full_evalue"] = pd.to_numeric(df["full_evalue"], errors="coerce")
        df["full_score"] = pd.to_numeric(df["full_score"], errors="coerce")

    df.to_csv(output_path, sep="\t", index=False)

    mapped = int((df["target_pid"].astype(str).str.len() > 0).sum()) if not df.empty else 0
    record_core_event("SARP HMM", "ok", f"{len(df)} hit(s), {mapped} mapped")

    return df

def run_domain_hmmscan_step(protein_fasta, id_to_pid):
    """Run domain HMM scan when domain database is available."""
    domain_hmm = Path(SHARP_INTERNAL_RESOURCES.get("domain_models_hmm", ""))
    output_path = SHARP_ANALYSIS_OUTPUTS["domain_hits"]

    columns = [
        "target_name",
        "query_name",
        "query_pid",
        "domain_evalue",
        "domain_score",
        "description",
    ]

    if not domain_hmm.exists():
        record_core_event("Domain HMM", "pending", f"missing domain HMM DB: {domain_hmm}")
        return write_pending_table(
            output_path,
            columns,
            "S(H)ARP domain HMM database is not loaded.",
        )

    hmmscan_bin = command_available("hmmscan")
    hmmpress_bin = command_available("hmmpress")

    if not hmmscan_bin:
        record_core_event("Domain HMM", "pending", "hmmscan executable not found")
        return write_pending_table(
            output_path,
            columns,
            "hmmscan executable not found.",
        )

    if hmmpress_bin:
        run_cmd(
            [hmmpress_bin, "-f", str(domain_hmm)],
            step_label="Indexing domain HMM database",
            check=False,
            update_every=20,
        )

    domtblout = DOMAIN_DIR_RUN / "sharp_domain_hmmscan.domtblout"

    run_cmd(
        [
            hmmscan_bin,
            "--domtblout",
            str(domtblout),
            "--noali",
            str(domain_hmm),
            str(protein_fasta),
        ],
        step_label="Running domain hmmscan",
        check=False,
        update_every=20,
    )

    rows = []

    if domtblout.exists():
        with open(domtblout, "r", encoding="utf-8", errors="replace") as handle:
            for line in handle:
                if not line.strip() or line.startswith("#"):
                    continue

                parts = line.strip().split(maxsplit=22)

                if len(parts) < 14:
                    continue

                query_name = parts[3]
                query_pid = map_external_id_to_pid(query_name, id_to_pid)

                rows.append({
                    "target_name": parts[0],
                    "query_name": query_name,
                    "query_pid": query_pid or "",
                    "domain_evalue": parts[12],
                    "domain_score": parts[13],
                    "description": parts[22] if len(parts) > 22 else "",
                })

    df = pd.DataFrame(rows, columns=columns)

    if not df.empty:
        df["domain_evalue"] = pd.to_numeric(df["domain_evalue"], errors="coerce")
        df["domain_score"] = pd.to_numeric(df["domain_score"], errors="coerce")

    df.to_csv(output_path, sep="\t", index=False)

    record_core_event("Domain HMM", "ok", f"{len(df)} domain hit(s)")

    return df

def run_embedding_step(protein_fasta):
    """Write pending embedding table unless embedding backend is available."""
    output_path = SHARP_ANALYSIS_OUTPUTS["embedding_scores"]

    embedding_model_path = SHARP_INTERNAL_RESOURCES.get("embedding_model", "")
    embedding_reference_path = SHARP_INTERNAL_RESOURCES.get("embedding_reference", "")

    embedding_model = Path(embedding_model_path) if embedding_model_path else None
    embedding_reference = Path(embedding_reference_path) if embedding_reference_path else None

    if embedding_model is None or embedding_reference is None:
        reason = "Embedding model/reference database is not loaded."
    elif not embedding_model.exists() or not embedding_reference.exists():
        reason = "Embedding model/reference database is not loaded."
    else:
        reason = "Embedding backend is pending implementation in this notebook."

    df = pd.DataFrame([{
        "pid": "",
        "embedding_label": "",
        "embedding_score": "",
        "status": "pending",
        "reason": reason,
    }])

    df.to_csv(output_path, sep="\t", index=False)

    record_core_event("Embeddings", "pending", reason)

    return df

# =====================================================================
# FIMO interpretation based on the original Git pipeline
# =====================================================================

def valid_hits_df(df):
    """Return True for a non-pending evidence hit table."""
    return (
        isinstance(df, pd.DataFrame)
        and not df.empty
        and "status" not in df.columns
    )

def build_cds_index_by_contig(cds_df):
    """Build CDS coordinate indexes by contig."""
    index = {}

    for contig, subdf in cds_df.groupby("nucleotide", sort=False):
        subdf = subdf.sort_values(["start", "end"], kind="mergesort").reset_index(drop=True)
        by_end = subdf.sort_values(["end", "start"], kind="mergesort").reset_index(drop=True)

        index[str(contig)] = {
            "by_start": subdf,
            "starts": subdf["start"].astype(int).tolist(),
            "ends_by_start": subdf["end"].astype(int).tolist(),
            "by_end": by_end,
            "ends": by_end["end"].astype(int).tolist(),
            "starts_by_end": by_end["start"].astype(int).tolist(),
        }

    return index

def detect_intragenic_for_hit(contig_index, hit_start, hit_stop):
    """Return whether a motif interval overlaps a CDS on the same contig."""
    if contig_index is None:
        return False

    ends = contig_index["ends"]
    starts_by_end = contig_index["starts_by_end"]

    if not ends:
        return False

    left = 0
    right = len(ends)

    while left < right:
        mid = (left + right) // 2
        if ends[mid] < hit_start:
            left = mid + 1
        else:
            right = mid

    idx = left

    if idx >= len(ends):
        return False

    return int(starts_by_end[idx]) <= int(hit_stop)

def get_next_cds_for_hit(contig_index, hit_start, hit_stop, hit_strand):
    """Get the next CDS according to motif strand, following the original pipeline logic."""
    if contig_index is None:
        return None, None

    if hit_strand == "+":
        starts = contig_index["starts"]
        by_start = contig_index["by_start"]

        left = 0
        right = len(starts)

        while left < right:
            mid = (left + right) // 2
            if starts[mid] <= hit_stop:
                left = mid + 1
            else:
                right = mid

        idx = left

        if idx >= len(by_start):
            return None, None

        row = by_start.iloc[idx]
        distance_to_next_cds = int(row["start"]) - int(hit_stop)

        return row, distance_to_next_cds

    if hit_strand == "-":
        ends = contig_index["ends"]
        by_end = contig_index["by_end"]

        left = 0
        right = len(ends)

        while left < right:
            mid = (left + right) // 2
            if ends[mid] < hit_start:
                left = mid + 1
            else:
                right = mid

        idx = left - 1

        if idx < 0:
            return None, None

        row = by_end.iloc[idx]
        distance_to_next_cds = int(hit_start) - int(row["end"])

        return row, distance_to_next_cds

    return None, None

def process_fimo_hits_original_logic(fimo_df, cds_df):
    """Preserve raw FIMO hits and create regulatory seed hits following the original Git logic."""
    fimo_to_cds_cols = [
        "fimo_hit_id",
        "motif_id",
        "motif_alt_id",
        "sequence_name",
        "start",
        "stop",
        "strand",
        "score",
        "p_value",
        "q_value",
        "matched_sequence",
        "previous_hit_stop",
        "distance",
        "next_pid",
        "next_pid_original",
        "next_locus_tag",
        "next_protein_id",
        "next_gene",
        "next_product",
        "next_cds_start",
        "next_cds_end",
        "next_cds_strand",
        "distance_to_next_cds",
        "intragenic",
        "is_repeat_distance_candidate",
        "is_regulatory_candidate",
    ]

    repeat_cols = fimo_to_cds_cols
    regulatory_cols = fimo_to_cds_cols

    if not valid_hits_df(fimo_df):
        fimo_to_cds = write_empty_table(SHARP_ANALYSIS_OUTPUTS["fimo_to_cds"], fimo_to_cds_cols)
        repeat_hits = write_empty_table(SHARP_ANALYSIS_OUTPUTS["fimo_repeat_hits"], repeat_cols)
        regulatory_hits = write_empty_table(SHARP_ANALYSIS_OUTPUTS["fimo_regulatory_hits"], regulatory_cols)

        record_core_event("FIMO interpretation", "ok", "no raw FIMO hits available")
        return fimo_to_cds, repeat_hits, regulatory_hits

    temp = fimo_df.copy()

    if "fimo_hit_id" not in temp.columns:
        temp["fimo_hit_id"] = [f"fimo_{i + 1:08d}" for i in range(len(temp))]

    temp["start"] = pd.to_numeric(temp["start"], errors="coerce")
    temp["stop"] = pd.to_numeric(temp["stop"], errors="coerce")
    temp["score"] = pd.to_numeric(temp.get("score", ""), errors="coerce")
    temp["p_value"] = pd.to_numeric(temp.get("p_value", ""), errors="coerce")
    temp["q_value"] = pd.to_numeric(temp.get("q_value", ""), errors="coerce")

    temp = temp.dropna(subset=["sequence_name", "start", "stop", "strand"]).copy()
    temp["start"] = temp["start"].astype(int)
    temp["stop"] = temp["stop"].astype(int)

    temp = temp.sort_values(
        ["sequence_name", "strand", "start", "stop"],
        kind="mergesort",
    ).reset_index(drop=True)

    temp["previous_hit_stop"] = temp.groupby(["sequence_name", "strand"])["stop"].shift(1)
    temp["distance"] = temp["start"] - temp["previous_hit_stop"]

    cds_index = build_cds_index_by_contig(cds_df)

    mapped_rows = []

    for _, hit in temp.iterrows():
        contig = str(hit["sequence_name"])
        contig_index = cds_index.get(contig)

        hit_start = int(hit["start"])
        hit_stop = int(hit["stop"])
        hit_strand = str(hit["strand"])

        intragenic = detect_intragenic_for_hit(
            contig_index=contig_index,
            hit_start=hit_start,
            hit_stop=hit_stop,
        )

        next_row, distance_to_next_cds = get_next_cds_for_hit(
            contig_index=contig_index,
            hit_start=hit_start,
            hit_stop=hit_stop,
            hit_strand=hit_strand,
        )

        if next_row is None:
            next_pid = ""
            next_pid_original = ""
            next_locus_tag = ""
            next_protein_id = ""
            next_gene = ""
            next_product = ""
            next_cds_start = ""
            next_cds_end = ""
            next_cds_strand = ""
            distance_to_next_cds_value = ""
        else:
            next_pid = str(next_row.get("pid", ""))
            next_pid_original = str(next_row.get("pid_original", ""))
            next_locus_tag = str(next_row.get("locus_tag", ""))
            next_protein_id = str(next_row.get("protein_id", ""))
            next_gene = str(next_row.get("gene", ""))
            next_product = str(next_row.get("product", ""))
            next_cds_start = int(next_row.get("start", 0))
            next_cds_end = int(next_row.get("end", 0))
            next_cds_strand = int(next_row.get("strand", 0))
            distance_to_next_cds_value = int(distance_to_next_cds) if distance_to_next_cds is not None else ""

        distance_value = hit.get("distance", "")
        distance_numeric = pd.to_numeric(distance_value, errors="coerce")

        is_repeat_distance_candidate = bool(
            pd.notna(distance_numeric)
            and distance_numeric > FIMO_REPEAT_DISTANCE_MIN
            and distance_numeric <= FIMO_REPEAT_DISTANCE_MAX
        )

        is_regulatory_candidate = bool(
            is_repeat_distance_candidate
            and not bool(intragenic)
            and bool(next_pid)
        )

        mapped_rows.append({
            "fimo_hit_id": hit.get("fimo_hit_id", ""),
            "motif_id": hit.get("motif_id", ""),
            "motif_alt_id": hit.get("motif_alt_id", ""),
            "sequence_name": contig,
            "start": hit_start,
            "stop": hit_stop,
            "strand": hit_strand,
            "score": hit.get("score", ""),
            "p_value": hit.get("p_value", ""),
            "q_value": hit.get("q_value", ""),
            "matched_sequence": hit.get("matched_sequence", ""),
            "previous_hit_stop": hit.get("previous_hit_stop", ""),
            "distance": distance_value,
            "next_pid": next_pid,
            "next_pid_original": next_pid_original,
            "next_locus_tag": next_locus_tag,
            "next_protein_id": next_protein_id,
            "next_gene": next_gene,
            "next_product": next_product,
            "next_cds_start": next_cds_start,
            "next_cds_end": next_cds_end,
            "next_cds_strand": next_cds_strand,
            "distance_to_next_cds": distance_to_next_cds_value,
            "intragenic": bool(intragenic),
            "is_repeat_distance_candidate": is_repeat_distance_candidate,
            "is_regulatory_candidate": is_regulatory_candidate,
        })

    fimo_to_cds = pd.DataFrame(mapped_rows, columns=fimo_to_cds_cols)

    if not fimo_to_cds.empty:
        for col in ["score", "p_value", "q_value", "previous_hit_stop", "distance", "distance_to_next_cds"]:
            if col in fimo_to_cds.columns:
                fimo_to_cds[col] = pd.to_numeric(fimo_to_cds[col], errors="coerce")

    repeat_hits = fimo_to_cds[
        fimo_to_cds["is_repeat_distance_candidate"] == True
    ].copy()

    regulatory_hits = fimo_to_cds[
        fimo_to_cds["is_regulatory_candidate"] == True
    ].copy()

    fimo_to_cds.to_csv(SHARP_ANALYSIS_OUTPUTS["fimo_to_cds"], sep="\t", index=False)
    repeat_hits.to_csv(SHARP_ANALYSIS_OUTPUTS["fimo_repeat_hits"], sep="\t", index=False)
    regulatory_hits.to_csv(SHARP_ANALYSIS_OUTPUTS["fimo_regulatory_hits"], sep="\t", index=False)

    unique_regulatory_pids = regulatory_hits["next_pid"].dropna().astype(str).replace("", pd.NA).dropna().nunique()

    record_core_event(
        "FIMO interpretation",
        "ok",
        (
            f"raw={len(fimo_to_cds)}; "
            f"repeat_distance={len(repeat_hits)}; "
            f"regulatory_intergenic={len(regulatory_hits)}; "
            f"unique_regulatory_pids={unique_regulatory_pids}"
        ),
    )

    return fimo_to_cds, repeat_hits, regulatory_hits

# =====================================================================
# Neighborhoods and candidate regions
# =====================================================================

def write_empty_ndf_and_regions():
    """Write empty NDF, candidate-region, seed, index, and evidence-link tables."""
    seed_cols = [
        "seed_pid",
        "seed_type",
        "seed_priority",
        "n_fimo_regulatory_hits",
        "n_sarp_hits",
        "best_sarp_evalue",
        "best_sarp_score",
        "seed_reasons",
    ]

    index_cols = [
        "block_id",
        "anchor_pid",
        "anchor_gene_index",
        "nucleotide",
        "window_start_gene_index",
        "window_end_gene_index",
        "window_start_bp",
        "window_end_bp",
        "n_genes",
        "seed_reasons",
        "has_sarp",
        "has_fimo_regulatory",
        "region_score",
        "region_class",
    ]

    link_cols = [
        "block_id",
        "anchor_pid",
        "evidence_type",
        "evidence_id",
        "evidence_pid",
        "motif_id",
        "sequence_name",
        "start",
        "stop",
        "strand",
        "distance",
        "intragenic",
        "description",
    ]

    ndf_cols = [
        "block_id",
        "organism",
        "nucleotide",
        "nlen",
        "pid",
        "start",
        "end",
        "strand",
        "query",
        "gene",
        "locus_tag",
        "protein_id",
        "product",
        "domain",
        "pfam",
        "source_anchor",
        "region_score",
        "region_class",
    ]

    region_cols = [
        "block_id",
        "anchor_pid",
        "n_genes",
        "has_sarp",
        "has_fimo",
        "n_fimo_regulatory_hits",
        "n_sarp_hits",
        "region_score",
        "region_class",
        "status",
    ]

    seed_table = pd.DataFrame(columns=seed_cols)
    neighborhood_index = pd.DataFrame(columns=index_cols)
    evidence_links = pd.DataFrame(columns=link_cols)
    ndf = pd.DataFrame(columns=ndf_cols)
    regions = pd.DataFrame(columns=region_cols)

    seed_table.to_csv(SHARP_ANALYSIS_OUTPUTS["neighborhood_seed_table"], sep="\t", index=False)
    neighborhood_index.to_csv(SHARP_ANALYSIS_OUTPUTS["neighborhood_index"], sep="\t", index=False)
    evidence_links.to_csv(SHARP_ANALYSIS_OUTPUTS["neighborhood_evidence_links"], sep="\t", index=False)
    ndf.to_csv(SHARP_ANALYSIS_OUTPUTS["ndf"], sep="\t", index=False)
    regions.to_csv(SHARP_ANALYSIS_OUTPUTS["regions"], sep="\t", index=False)

    return seed_table, neighborhood_index, evidence_links, ndf, regions

def build_seed_table(cds_df, fimo_regulatory_df, sarp_df):
    """Build unique PID seeds following the original Git pipeline logic."""
    valid_pids = set(cds_df["pid"].astype(str))

    seed_records = {}

    if valid_hits_df(fimo_regulatory_df):
        for pid, sub in fimo_regulatory_df.groupby("next_pid", dropna=True):
            pid = str(pid).strip()

            if not pid or pid not in valid_pids:
                continue

            seed = seed_records.setdefault(pid, {
                "seed_pid": pid,
                "seed_type": set(),
                "seed_priority": 4,
                "n_fimo_regulatory_hits": 0,
                "n_sarp_hits": 0,
                "best_sarp_evalue": "",
                "best_sarp_score": "",
                "seed_reasons": set(),
            })

            seed["seed_type"].add("fimo_regulatory")
            seed["seed_reasons"].add("fimo_regulatory_repeat")
            seed["n_fimo_regulatory_hits"] += int(len(sub))

    if valid_hits_df(sarp_df):
        temp = sarp_df.copy()

        if "full_evalue" in temp.columns:
            temp["full_evalue"] = pd.to_numeric(temp["full_evalue"], errors="coerce")
            temp = temp[temp["full_evalue"] <= SARP_HMM_EVALUE_THRESHOLD]

        if "full_score" in temp.columns:
            temp["full_score"] = pd.to_numeric(temp["full_score"], errors="coerce")
            temp = temp[temp["full_score"] >= SARP_HMM_SCORE_THRESHOLD]

        for pid, sub in temp.groupby("target_pid", dropna=True):
            pid = str(pid).strip()

            if not pid or pid not in valid_pids:
                continue

            seed = seed_records.setdefault(pid, {
                "seed_pid": pid,
                "seed_type": set(),
                "seed_priority": 1,
                "n_fimo_regulatory_hits": 0,
                "n_sarp_hits": 0,
                "best_sarp_evalue": "",
                "best_sarp_score": "",
                "seed_reasons": set(),
            })

            seed["seed_type"].add("sarp_hmm")
            seed["seed_reasons"].add("sarp_hmm")
            seed["n_sarp_hits"] += int(len(sub))
            seed["seed_priority"] = min(seed["seed_priority"], 1)

            if "full_evalue" in sub.columns:
                best_evalue = pd.to_numeric(sub["full_evalue"], errors="coerce").min()
                if pd.notna(best_evalue):
                    seed["best_sarp_evalue"] = float(best_evalue)

            if "full_score" in sub.columns:
                best_score = pd.to_numeric(sub["full_score"], errors="coerce").max()
                if pd.notna(best_score):
                    seed["best_sarp_score"] = float(best_score)

    rows = []

    for pid, seed in seed_records.items():
        seed_types = sorted(seed["seed_type"])
        seed_reasons = sorted(seed["seed_reasons"])

        if "sarp_hmm" in seed_types and "fimo_regulatory" in seed_types:
            final_type = "sarp_hmm_and_fimo_regulatory"
            final_priority = 1
        elif "sarp_hmm" in seed_types:
            final_type = "sarp_hmm"
            final_priority = 1
        elif "fimo_regulatory" in seed_types:
            final_type = "fimo_regulatory"
            final_priority = 4
        else:
            final_type = "candidate"
            final_priority = 9

        rows.append({
            "seed_pid": pid,
            "seed_type": final_type,
            "seed_priority": final_priority,
            "n_fimo_regulatory_hits": seed["n_fimo_regulatory_hits"],
            "n_sarp_hits": seed["n_sarp_hits"],
            "best_sarp_evalue": seed["best_sarp_evalue"],
            "best_sarp_score": seed["best_sarp_score"],
            "seed_reasons": ";".join(seed_reasons),
        })

    seed_table = pd.DataFrame(rows)

    if not seed_table.empty:
        seed_table = seed_table.sort_values(
            ["seed_priority", "seed_pid"],
            kind="mergesort",
        ).reset_index(drop=True)

    seed_table.to_csv(
        SHARP_ANALYSIS_OUTPUTS["neighborhood_seed_table"],
        sep="\t",
        index=False,
    )

    return seed_table

def build_domain_label_by_pid(domain_df):
    """Build a simple domain label mapping for NDF display."""
    domain_label_by_pid = {}

    if valid_hits_df(domain_df):
        for pid, sub in domain_df.groupby("query_pid"):
            pid = str(pid).strip()

            if not pid:
                continue

            labels = [
                str(x)
                for x in sub["target_name"].dropna().unique().tolist()
                if str(x).strip()
            ]

            if labels:
                domain_label_by_pid[pid] = ";".join(labels[:4])

    return domain_label_by_pid

def build_fimo_links_by_pid(fimo_regulatory_df):
    """Build regulatory FIMO evidence links grouped by seed PID."""
    links_by_pid = {}

    if not valid_hits_df(fimo_regulatory_df):
        return links_by_pid

    for _, row in fimo_regulatory_df.iterrows():
        pid = str(row.get("next_pid", "")).strip()

        if not pid:
            continue

        links_by_pid.setdefault(pid, []).append(row.to_dict())

    return links_by_pid

def build_sarp_links_by_pid(sarp_df):
    """Build SARP HMM evidence links grouped by seed PID."""
    links_by_pid = {}

    if not valid_hits_df(sarp_df):
        return links_by_pid

    for _, row in sarp_df.iterrows():
        pid = str(row.get("target_pid", "")).strip()

        if not pid:
            continue

        links_by_pid.setdefault(pid, []).append(row.to_dict())

    return links_by_pid

def build_ndf_and_regions(cds_df, fimo_regulatory_df, sarp_df, domain_df, embedding_df):
    """Build NDF from unique PID seeds, preserving raw FIMO in separate linked tables."""
    seed_table = build_seed_table(
        cds_df=cds_df,
        fimo_regulatory_df=fimo_regulatory_df,
        sarp_df=sarp_df,
    )

    if seed_table.empty:
        record_core_event("Neighborhoods", "ok", "no candidate PID seeds detected")
        return write_empty_ndf_and_regions()

    domain_label_by_pid = build_domain_label_by_pid(domain_df)
    fimo_links_by_pid = build_fimo_links_by_pid(fimo_regulatory_df)
    sarp_links_by_pid = build_sarp_links_by_pid(sarp_df)

    ndf_rows = []
    region_rows = []
    index_rows = []
    evidence_link_rows = []

    cds_by_pid = cds_df.set_index("pid", drop=False)

    for block_idx, seed in enumerate(seed_table.itertuples(index=False), start=1):
        anchor_pid = str(seed.seed_pid)

        if anchor_pid not in cds_by_pid.index:
            continue

        anchor_row = cds_by_pid.loc[anchor_pid]

        if isinstance(anchor_row, pd.DataFrame):
            anchor_row = anchor_row.iloc[0]

        contig = anchor_row["nucleotide"]
        anchor_gene_index = int(anchor_row["gene_index"])

        window = cds_df[
            (cds_df["nucleotide"] == contig)
            & (cds_df["gene_index"] >= anchor_gene_index - NEIGHBORHOOD_BEFORE_GENES)
            & (cds_df["gene_index"] <= anchor_gene_index + NEIGHBORHOOD_AFTER_GENES)
        ].copy()

        if window.empty:
            continue

        block_id = f"sharp_block_{block_idx:06d}"

        has_sarp = "sarp_hmm" in str(seed.seed_type)
        has_fimo = "fimo_regulatory" in str(seed.seed_type)

        region_score = 0
        region_score += 2 if has_sarp else 0
        region_score += 1 if has_fimo else 0

        if has_sarp and has_fimo:
            region_class = "sarp_and_regulatory_motif_supported"
        elif has_sarp:
            region_class = "sarp_supported"
        elif has_fimo:
            region_class = "regulatory_motif_supported"
        else:
            region_class = "candidate"

        window_start_gene_index = int(window["gene_index"].min())
        window_end_gene_index = int(window["gene_index"].max())
        window_start_bp = int(window["start"].min())
        window_end_bp = int(window["end"].max())

        index_rows.append({
            "block_id": block_id,
            "anchor_pid": anchor_pid,
            "anchor_gene_index": anchor_gene_index,
            "nucleotide": contig,
            "window_start_gene_index": window_start_gene_index,
            "window_end_gene_index": window_end_gene_index,
            "window_start_bp": window_start_bp,
            "window_end_bp": window_end_bp,
            "n_genes": int(len(window)),
            "seed_reasons": str(seed.seed_reasons),
            "has_sarp": bool(has_sarp),
            "has_fimo_regulatory": bool(has_fimo),
            "region_score": region_score,
            "region_class": region_class,
        })

        for fimo_hit in fimo_links_by_pid.get(anchor_pid, []):
            evidence_link_rows.append({
                "block_id": block_id,
                "anchor_pid": anchor_pid,
                "evidence_type": "fimo_regulatory_repeat",
                "evidence_id": fimo_hit.get("fimo_hit_id", ""),
                "evidence_pid": anchor_pid,
                "motif_id": fimo_hit.get("motif_id", ""),
                "sequence_name": fimo_hit.get("sequence_name", ""),
                "start": fimo_hit.get("start", ""),
                "stop": fimo_hit.get("stop", ""),
                "strand": fimo_hit.get("strand", ""),
                "distance": fimo_hit.get("distance", ""),
                "intragenic": fimo_hit.get("intragenic", ""),
                "description": f"intergenic repeat motif mapped to downstream PID {anchor_pid}",
            })

        for sarp_hit in sarp_links_by_pid.get(anchor_pid, []):
            evidence_link_rows.append({
                "block_id": block_id,
                "anchor_pid": anchor_pid,
                "evidence_type": "sarp_hmm",
                "evidence_id": sarp_hit.get("target_name", ""),
                "evidence_pid": anchor_pid,
                "motif_id": "",
                "sequence_name": "",
                "start": "",
                "stop": "",
                "strand": "",
                "distance": "",
                "intragenic": "",
                "description": (
                    f"score={sarp_hit.get('full_score', '')}; "
                    f"evalue={sarp_hit.get('full_evalue', '')}"
                ),
            })

        for _, row in window.iterrows():
            pid = row["pid"]
            is_query = 1 if pid == anchor_pid else 0

            if pid == anchor_pid and has_sarp:
                domain_label = "SARP-like regulator"
            else:
                domain_label = domain_label_by_pid.get(
                    pid,
                    row.get("product", "") or "unannotated",
                )

            ndf_rows.append({
                "block_id": block_id,
                "organism": row.get("organism", SHARP_SAMPLE.get("organism_name", "unknown")),
                "nucleotide": row["nucleotide"],
                "nlen": row.get("nlen", ""),
                "pid": pid,
                "start": row["start"],
                "end": row["end"],
                "strand": row["strand"],
                "query": is_query,
                "gene": row.get("gene", ""),
                "locus_tag": row.get("locus_tag", ""),
                "protein_id": row.get("protein_id", ""),
                "product": row.get("product", ""),
                "domain": domain_label,
                "pfam": domain_label,
                "source_anchor": str(seed.seed_reasons),
                "region_score": region_score,
                "region_class": region_class,
            })

        region_rows.append({
            "block_id": block_id,
            "anchor_pid": anchor_pid,
            "n_genes": int(len(window)),
            "has_sarp": bool(has_sarp),
            "has_fimo": bool(has_fimo),
            "n_fimo_regulatory_hits": int(seed.n_fimo_regulatory_hits),
            "n_sarp_hits": int(seed.n_sarp_hits),
            "region_score": region_score,
            "region_class": region_class,
            "status": "candidate_region",
        })

    neighborhood_index = pd.DataFrame(index_rows)
    evidence_links = pd.DataFrame(evidence_link_rows)
    ndf = pd.DataFrame(ndf_rows)
    regions = pd.DataFrame(region_rows)

    neighborhood_index.to_csv(
        SHARP_ANALYSIS_OUTPUTS["neighborhood_index"],
        sep="\t",
        index=False,
    )

    evidence_links.to_csv(
        SHARP_ANALYSIS_OUTPUTS["neighborhood_evidence_links"],
        sep="\t",
        index=False,
    )

    ndf.to_csv(
        SHARP_ANALYSIS_OUTPUTS["ndf"],
        sep="\t",
        index=False,
    )

    regions.to_csv(
        SHARP_ANALYSIS_OUTPUTS["regions"],
        sep="\t",
        index=False,
    )

    record_core_event(
        "Neighborhoods",
        "ok",
        (
            f"seeds={len(seed_table)}; "
            f"candidate_regions={len(regions)}; "
            f"ndf_rows={len(ndf)}; "
            f"evidence_links={len(evidence_links)}"
        ),
    )

    return seed_table, neighborhood_index, evidence_links, ndf, regions

# =====================================================================
# Main execution
# =====================================================================

SHARP_CORE_STATUS = {
    "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "finished_at": None,
    "status": "running",
    "error": None,
    "traceback": None,
}

SHARP_ANNOTATION_CONTRACT = {}
cds_df = pd.DataFrame()
contig_df = pd.DataFrame()
fimo_df = pd.DataFrame()
fimo_to_cds_df = pd.DataFrame()
fimo_repeat_df = pd.DataFrame()
fimo_regulatory_df = pd.DataFrame()
sarp_df = pd.DataFrame()
domain_df = pd.DataFrame()
embedding_df = pd.DataFrame()
seed_table = pd.DataFrame()
neighborhood_index = pd.DataFrame()
evidence_links = pd.DataFrame()
ndf = pd.DataFrame()
regions = pd.DataFrame()

try:
    if not SHARP_DEPENDENCIES_READY:
        blockers = (
            SHARP_DEPENDENCY_STATUS.get("blockers")
            or SHARP_DEPENDENCY_STATUS.get("dependency_blockers")
            or []
        )

        blocker_text = (
            "\n".join(f"- {item}" for item in blockers)
            if blockers
            else "No explicit blocker was reported."
        )

        raise RuntimeError(
            "S(H)ARP dependencies are not ready.\n\n"
            f"{blocker_text}"
        )

    update_core_status(
        phase="Building annotation contract",
        message="S(H)ARP is standardizing genome, annotation, proteins, CDS table, and ID crosswalk.",
        progress=18,
        rows=[
            ("Annotation backend", annotation_backend),
            ("Genome FASTA", GENOME_FASTA),
            ("Bakta DB", BAKTA_DB_PATH or "not required"),
        ],
        history_label="Annotation contract started",
    )

    SHARP_ANNOTATION_CONTRACT, cds_df, contig_df = obtain_annotation_contract()

    record_core_event(
        "Annotation contract",
        "ok",
        f"{SHARP_ANNOTATION_CONTRACT['n_cds']} CDS across {SHARP_ANNOTATION_CONTRACT['n_contigs']} contig(s)",
    )

    update_core_status(
        phase="Annotation ready",
        message="Standardized annotation files were generated.",
        progress=48,
        rows=[
            ("CDS parsed", SHARP_ANNOTATION_CONTRACT["n_cds"]),
            ("Contigs", SHARP_ANNOTATION_CONTRACT["n_contigs"]),
            ("Annotation", SHARP_ANNOTATION_CONTRACT["annotation_file"]),
            ("Proteins", SHARP_ANNOTATION_CONTRACT["protein_fasta"]),
        ],
        history_label="Annotation contract ready",
    )

    id_to_pid = build_id_to_pid_map(cds_df)

    update_core_status(
        phase="Running evidence steps",
        message="S(H)ARP is evaluating motif, HMM, domain, and embedding evidence when resources are available.",
        progress=58,
        rows=[
            ("Protein FASTA", SHARP_ANNOTATION_CONTRACT["protein_fasta"]),
            ("FIMO", "available" if command_available("fimo") else "pending"),
            ("hmmsearch", command_available("hmmsearch") or "not available"),
            ("hmmscan", command_available("hmmscan") or "not available"),
        ],
        history_label="Evidence steps started",
    )

    fimo_df = run_fimo_step(SHARP_ANNOTATION_CONTRACT["genome_fasta"])

    update_core_status(
        phase="Interpreting FIMO motif logic",
        message="S(H)ARP is preserving raw FIMO hits, calculating repeat distances, mapping hits to downstream CDS, and selecting intergenic regulatory seeds.",
        progress=64,
        rows=[
            ("Raw FIMO hits", count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_hits"])),
            ("Raw FIMO table", SHARP_ANALYSIS_OUTPUTS["fimo_hits"]),
        ],
        history_label="FIMO interpretation started",
    )

    fimo_to_cds_df, fimo_repeat_df, fimo_regulatory_df = process_fimo_hits_original_logic(
        fimo_df=fimo_df,
        cds_df=cds_df,
    )

    update_core_status(
        phase="Motif evidence processed",
        message="FIMO motif evidence was processed using the original Git-style repeat/CDS/intergenic logic.",
        progress=68,
        rows=[
            ("Raw FIMO hits", count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_hits"])),
            ("FIMO to CDS rows", count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_to_cds"])),
            ("FIMO repeat hits", count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_repeat_hits"])),
            ("FIMO regulatory hits", count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_regulatory_hits"])),
        ],
        history_label="Motif evidence finished",
    )

    sarp_df = run_sarp_hmm_step(
        SHARP_ANNOTATION_CONTRACT["protein_fasta"],
        id_to_pid=id_to_pid,
    )

    update_core_status(
        phase="SARP HMM evidence processed",
        message="SARP/BtaD-like regulator evidence was processed or marked as pending.",
        progress=74,
        rows=[
            ("SARP HMM rows", count_real_rows(SHARP_ANALYSIS_OUTPUTS["sarp_hits"])),
            ("SARP HMM table", SHARP_ANALYSIS_OUTPUTS["sarp_hits"]),
        ],
        history_label="SARP HMM evidence finished",
    )

    domain_df = run_domain_hmmscan_step(
        SHARP_ANNOTATION_CONTRACT["protein_fasta"],
        id_to_pid=id_to_pid,
    )

    update_core_status(
        phase="Domain evidence processed",
        message="Domain evidence was processed or marked as pending.",
        progress=80,
        rows=[
            ("Domain rows", count_real_rows(SHARP_ANALYSIS_OUTPUTS["domain_hits"])),
            ("Domain table", SHARP_ANALYSIS_OUTPUTS["domain_hits"]),
        ],
        history_label="Domain evidence finished",
    )

    embedding_df = run_embedding_step(
        SHARP_ANNOTATION_CONTRACT["protein_fasta"],
    )

    update_core_status(
        phase="Building neighborhoods",
        message="S(H)ARP is building neighborhoods from unique PID seeds instead of expanding one neighborhood per raw FIMO hit.",
        progress=88,
        rows=[
            ("CDS table", SHARP_ANALYSIS_OUTPUTS["cds_table"]),
            ("Seed table", SHARP_ANALYSIS_OUTPUTS["neighborhood_seed_table"]),
            ("Neighborhood table", SHARP_ANALYSIS_OUTPUTS["ndf"]),
            ("Candidate regions", SHARP_ANALYSIS_OUTPUTS["regions"]),
        ],
        history_label="Neighborhood construction started",
    )

    seed_table, neighborhood_index, evidence_links, ndf, regions = build_ndf_and_regions(
        cds_df=cds_df,
        fimo_regulatory_df=fimo_regulatory_df,
        sarp_df=sarp_df,
        domain_df=domain_df,
        embedding_df=embedding_df,
    )

    SHARP_CORE_STATUS["status"] = "completed"
    SHARP_CORE_STATUS["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

except Exception as exc:
    SHARP_CORE_STATUS["status"] = "failed"
    SHARP_CORE_STATUS["error"] = str(exc)
    SHARP_CORE_STATUS["traceback"] = traceback.format_exc()
    SHARP_CORE_STATUS["finished_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

    record_core_event("Core analysis", "failed", str(exc))

    if DEBUG_CORE:
        raise

# =====================================================================
# Manifest and final counts
# =====================================================================

output_counts = {
    "cds": int(len(cds_df)) if isinstance(cds_df, pd.DataFrame) else 0,
    "contigs": int(len(contig_df)) if isinstance(contig_df, pd.DataFrame) else 0,
    "fimo_hits": count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_hits"]),
    "fimo_to_cds": count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_to_cds"]),
    "fimo_repeat_hits": count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_repeat_hits"]),
    "fimo_regulatory_hits": count_real_rows(SHARP_ANALYSIS_OUTPUTS["fimo_regulatory_hits"]),
    "sarp_hmm_hits": count_real_rows(SHARP_ANALYSIS_OUTPUTS["sarp_hits"]),
    "domain_hits": count_real_rows(SHARP_ANALYSIS_OUTPUTS["domain_hits"]),
    "embedding_scores": count_real_rows(SHARP_ANALYSIS_OUTPUTS["embedding_scores"]),
    "neighborhood_seeds": count_real_rows(SHARP_ANALYSIS_OUTPUTS["neighborhood_seed_table"]),
    "neighborhood_index_rows": count_real_rows(SHARP_ANALYSIS_OUTPUTS["neighborhood_index"]),
    "neighborhood_evidence_links": count_real_rows(SHARP_ANALYSIS_OUTPUTS["neighborhood_evidence_links"]),
    "ndf_rows": count_real_rows(SHARP_ANALYSIS_OUTPUTS["ndf"]),
    "candidate_regions": count_real_rows(SHARP_ANALYSIS_OUTPUTS["regions"]),
}

SHARP_PIPELINE_MANIFEST = {
    "status": SHARP_CORE_STATUS,
    "events": CORE_EVENTS,
    "sample": SHARP_SAMPLE,
    "workflow": workflow,
    "input_mode": input_mode,
    "annotation_mode": annotation_mode,
    "annotation_backend": annotation_backend,
    "annotation_contract": SHARP_ANNOTATION_CONTRACT,
    "outputs": path_to_string(SHARP_ANALYSIS_OUTPUTS),
    "counts": output_counts,
    "fimo_logic": {
        "raw_hits_preserved": True,
        "distance_formula": "distance = current_start - previous_stop within sequence_name and strand",
        "repeat_distance_min_exclusive": FIMO_REPEAT_DISTANCE_MIN,
        "repeat_distance_max_inclusive": FIMO_REPEAT_DISTANCE_MAX,
        "regulatory_seed_rule": "repeat-distance hit, mapped to next CDS according to motif strand, and intragenic == False",
        "neighborhood_seed_unit": "unique PID, not raw FIMO hit",
    },
    "backend": path_to_string(SHARP_BACKEND),
    "tool_status": path_to_string(SHARP_TOOL_STATUS),
    "python_status": path_to_string(SHARP_PYTHON_STATUS),
}

write_json(
    SHARP_ANALYSIS_OUTPUTS["pipeline_manifest"],
    SHARP_PIPELINE_MANIFEST,
)

context.update({
    "core_status": SHARP_CORE_STATUS["status"],
    "pipeline_manifest": str(SHARP_ANALYSIS_OUTPUTS["pipeline_manifest"]),
    "annotation_contract": SHARP_ANNOTATION_CONTRACT,
    "analysis_outputs": path_to_string(SHARP_ANALYSIS_OUTPUTS),
    "core_counts": output_counts,
})

write_json(SHARP_CONTEXT_FILE, context)

globals().update({
    "SHARP_CORE_STATUS": SHARP_CORE_STATUS,
    "SHARP_PIPELINE_MANIFEST": SHARP_PIPELINE_MANIFEST,
    "SHARP_ANALYSIS_OUTPUTS": SHARP_ANALYSIS_OUTPUTS,
    "SHARP_ANNOTATION_CONTRACT": SHARP_ANNOTATION_CONTRACT,
    "SHARP_SAMPLE": SHARP_SAMPLE,
    "ANNOTATION_DIR": ANNOTATION_DIR,
    "BAKTA_OUTPUT_DIR": BAKTA_OUTPUT_DIR,
    "BAKTA_RUN_DIR": BAKTA_RUN_DIR,
    "TABLE_DIR": TABLE_DIR,
    "TABLES_DIR": TABLE_DIR,
    "LOG_DIR": LOG_DIR,
    "LOGS_DIR": LOG_DIR,
    "FIMO_DIR": FIMO_DIR,
    "HMM_DIR_RUN": HMM_DIR_RUN,
    "DOMAIN_DIR_RUN": DOMAIN_DIR_RUN,
    "EMBEDDING_DIR_RUN": EMBEDDING_DIR_RUN,
    "TMP_DIR": TMP_DIR,
    "cds_df": cds_df,
    "contig_df": contig_df,
    "fimo_df": fimo_df,
    "fimo_to_cds_df": fimo_to_cds_df,
    "fimo_repeat_df": fimo_repeat_df,
    "fimo_regulatory_df": fimo_regulatory_df,
    "sarp_df": sarp_df,
    "domain_df": domain_df,
    "embedding_df": embedding_df,
    "seed_table": seed_table,
    "neighborhood_index": neighborhood_index,
    "evidence_links": evidence_links,
    "ndf": ndf,
    "regions": regions,
})

if SHARP_CORE_STATUS["status"] == "completed":
    final_message = "Core analysis completed. Continue to the report/export cell."
    final_blockers = []
else:
    final_message = "Core analysis failed. Check the error and command logs before continuing."
    final_blockers = [SHARP_CORE_STATUS.get("error", "Unknown error")]

update_core_status(
    phase="S(H)ARP core analysis completed" if SHARP_CORE_STATUS["status"] == "completed" else "S(H)ARP core analysis failed",
    message=final_message,
    progress=100 if SHARP_CORE_STATUS["status"] == "completed" else 90,
    rows=[
        ("Status", SHARP_CORE_STATUS["status"]),
        ("Annotation source", SHARP_ANNOTATION_CONTRACT.get("source", "not generated")),
        ("CDS parsed", output_counts["cds"]),
        ("Contigs", output_counts["contigs"]),
        ("Raw FIMO hits", output_counts["fimo_hits"]),
        ("FIMO repeat hits", output_counts["fimo_repeat_hits"]),
        ("FIMO regulatory hits", output_counts["fimo_regulatory_hits"]),
        ("SARP HMM hits", output_counts["sarp_hmm_hits"]),
        ("Domain hits", output_counts["domain_hits"]),
        ("Neighborhood seeds", output_counts["neighborhood_seeds"]),
        ("Candidate regions", output_counts["candidate_regions"]),
        ("NDF", SHARP_ANALYSIS_OUTPUTS["ndf"]),
        ("Regions", SHARP_ANALYSIS_OUTPUTS["regions"]),
        ("Manifest", SHARP_ANALYSIS_OUTPUTS["pipeline_manifest"]),
    ],
    blockers=final_blockers,
    done=True,
    history_label="Core analysis finished",
)

print("S(H)ARP core analysis finished.")
print("status:", SHARP_CORE_STATUS["status"])
print("annotation_source:", SHARP_ANNOTATION_CONTRACT.get("source", "not generated"))
print("cds_parsed:", output_counts["cds"])
print("contigs:", output_counts["contigs"])
print("raw_fimo_hits:", output_counts["fimo_hits"])
print("fimo_to_cds:", output_counts["fimo_to_cds"])
print("fimo_repeat_hits:", output_counts["fimo_repeat_hits"])
print("fimo_regulatory_hits:", output_counts["fimo_regulatory_hits"])
print("sarp_hmm_hits:", output_counts["sarp_hmm_hits"])
print("domain_hits:", output_counts["domain_hits"])
print("neighborhood_seeds:", output_counts["neighborhood_seeds"])
print("neighborhood_index_rows:", output_counts["neighborhood_index_rows"])
print("neighborhood_evidence_links:", output_counts["neighborhood_evidence_links"])
print("ndf_rows:", output_counts["ndf_rows"])
print("candidate_regions:", output_counts["candidate_regions"])
print("manifest:", SHARP_ANALYSIS_OUTPUTS["pipeline_manifest"])
print("raw_fimo_table:", SHARP_ANALYSIS_OUTPUTS["fimo_hits"])
print("fimo_to_cds_table:", SHARP_ANALYSIS_OUTPUTS["fimo_to_cds"])
print("fimo_repeat_table:", SHARP_ANALYSIS_OUTPUTS["fimo_repeat_hits"])
print("fimo_regulatory_table:", SHARP_ANALYSIS_OUTPUTS["fimo_regulatory_hits"])
print("seed_table:", SHARP_ANALYSIS_OUTPUTS["neighborhood_seed_table"])
print("ndf:", SHARP_ANALYSIS_OUTPUTS["ndf"])
print("regions:", SHARP_ANALYSIS_OUTPUTS["regions"])

if SHARP_CORE_STATUS["status"] != "completed":
    print()
    print("Error:")
    print(SHARP_CORE_STATUS.get("error", "Unknown error"))
    print()
    print("Traceback:")
    print(SHARP_CORE_STATUS.get("traceback", ""))
    raise RuntimeError(SHARP_CORE_STATUS.get("error", "S(H)ARP core analysis failed."))

In [ ]:
# @title Build S(H)ARP interactive HTML report and complete results package { display-mode: "form" }
# @markdown Generate the final interactive S(H)ARP report and one complete downloadable results package.
# @markdown
# @markdown No user configuration is required.

from pathlib import Path
from IPython.display import display, HTML
import os
import re
import json
import html
import zipfile
import time
import base64
import mimetypes
import pandas as pd

# =====================================================================
# Internal options
# =====================================================================

SHOW_REPORT_INLINE = True
INCLUDE_LOGS_IN_EXPORT = True
INCLUDE_NORMALIZED_INPUTS_IN_EXPORT = True
MAX_TABLE_ROWS_IN_REPORT = 200
MAX_EMBEDDED_ZIP_BUTTON_MB = 80
PRINT_TECHNICAL_PATHS = False

# =====================================================================
# Required runtime objects
# =====================================================================

required_globals = [
    "SHARP_CORE_STATUS",
    "SHARP_ANALYSIS_OUTPUTS",
    "SHARP_SAMPLE",
    "RUN_DIR",
]

missing = [name for name in required_globals if name not in globals()]

if missing:
    raise RuntimeError(
        "Run the S(H)ARP core analysis cell before this report cell. Missing: "
        + ", ".join(missing)
    )

if SHARP_CORE_STATUS.get("status") != "completed":
    raise RuntimeError(
        "S(H)ARP core analysis did not complete successfully. "
        "Run or fix the core analysis cell before exporting results."
    )

RUN_DIR = Path(RUN_DIR).resolve()
TABLES_DIR = Path(globals().get("TABLES_DIR", RUN_DIR / "tables")).resolve()
ANNOTATION_DIR = Path(globals().get("ANNOTATION_DIR", RUN_DIR / "annotation")).resolve()
REPORT_DIR = Path(globals().get("REPORT_DIR", RUN_DIR / "report")).resolve()
LOGS_DIR = Path(globals().get("LOGS_DIR", RUN_DIR / "logs")).resolve()
CONFIG_DIR = Path(globals().get("CONFIG_DIR", RUN_DIR.parent.parent / "config")).resolve()
NORMALIZED_DIR = Path(globals().get("NORMALIZED_DIR", RUN_DIR.parent.parent / "data" / "normalized")).resolve()

REPORT_DIR.mkdir(parents=True, exist_ok=True)

REPORT_HTML = REPORT_DIR / "sharp_results.html"
RESOURCE_MANIFEST_JSON = REPORT_DIR / "sharp_resource_manifest.json"
RESOURCE_CHECKLIST_TSV = REPORT_DIR / "sharp_resource_checklist.tsv"
PACKAGE_MANIFEST_JSON = REPORT_DIR / "sharp_package_manifest.json"

EXPORT_ZIP = RUN_DIR / f"{SHARP_SAMPLE.get('analysis_name', 'sharp_run')}_sharp_complete_results.zip"

SHARP_CORE_RESULTS = globals().get("SHARP_CORE_RESULTS", {})
SHARP_INTERNAL_RESOURCES = globals().get("SHARP_INTERNAL_RESOURCES", {})
SHARP_BACKEND = globals().get("SHARP_BACKEND", {})
SHARP_DEPENDENCY_STATUS = globals().get("SHARP_DEPENDENCY_STATUS", {})
SHARP_PIPELINE_MANIFEST = globals().get("SHARP_PIPELINE_MANIFEST", {})
SHARP_CONTEXT_FILE = Path(
    globals().get("SHARP_CONTEXT_FILE", CONFIG_DIR / "sharp_notebook_context.json")
)

annotation_mode_current = globals().get(
    "annotation_mode",
    SHARP_PIPELINE_MANIFEST.get("annotation_mode", SHARP_CORE_STATUS.get("annotation_mode", "unknown")),
)

input_mode_current = globals().get(
    "input_mode",
    SHARP_PIPELINE_MANIFEST.get("input_mode", SHARP_SAMPLE.get("input_mode", "unknown")),
)

requires_bakta_db_current = bool(
    input_mode_current == "genome_fasta_only"
    or annotation_mode_current == "auto_from_genome"
    or SHARP_BACKEND.get("requires_bakta_db", False)
    or SHARP_BACKEND.get("input_requires_bakta_db", False)
)

# =====================================================================
# Helpers
# =====================================================================

def read_tsv(path):
    """Read a TSV file safely."""
    if path is None:
        return pd.DataFrame()

    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path, sep="\t")
    except Exception:
        return pd.DataFrame()

def count_real_rows(df, candidate_only=False):
    """Count rows while treating pending one-row placeholders as zero."""
    if df is None or df.empty:
        return 0

    if "status" in df.columns:
        statuses = set(df["status"].dropna().astype(str).str.lower())

        if statuses and statuses <= {"pending"}:
            return 0

    if candidate_only and "status" in df.columns:
        df = df[df["status"].astype(str) == "candidate_region"]

    return int(len(df))

def safe(value):
    """HTML-escape a value."""
    return html.escape("" if value is None else str(value))

def file_size_label(path):
    """Return a human-readable file size."""
    path = Path(path)

    if not path.exists():
        return "not found"

    size = path.stat().st_size

    if size < 1024:
        return f"{size} B"

    if size < 1024 ** 2:
        return f"{size / 1024:.1f} KB"

    if size < 1024 ** 3:
        return f"{size / (1024 ** 2):.1f} MB"

    return f"{size / (1024 ** 3):.2f} GB"

def status_label(df):
    """Return a compact status label for a table."""
    if df is None or df.empty:
        return "empty"

    if "status" in df.columns:
        statuses = set(df["status"].dropna().astype(str).str.lower())

        if statuses and statuses <= {"pending"}:
            reason = ""

            if "reason" in df.columns and len(df["reason"].dropna()):
                reason = str(df["reason"].dropna().iloc[0])

            return "pending" + (f": {reason}" if reason else "")

    return "completed"

def json_ready(value):
    """Convert values to JSON-serializable objects."""
    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}

    if isinstance(value, list):
        return [json_ready(v) for v in value]

    if isinstance(value, tuple):
        return [json_ready(v) for v in value]

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value

def resource_exists(path):
    """Return True if a resource path exists and is non-empty."""
    if path is None:
        return False

    path_text = str(path).strip()

    if not path_text:
        return False

    path = Path(path_text)

    if not path.exists():
        return False

    if path.is_file():
        return path.stat().st_size > 0

    if path.is_dir():
        try:
            return any(path.iterdir())
        except Exception:
            return False

    return False

def hmm_is_pressed(hmm_path):
    """Return True if HMMER pressed files exist."""
    if hmm_path is None or not str(hmm_path).strip():
        return False

    hmm_path = Path(hmm_path)

    return all(
        Path(str(hmm_path) + suffix).exists()
        for suffix in [".h3f", ".h3i", ".h3m", ".h3p"]
    )

def resource_path(key, fallback=""):
    """Resolve an internal resource path."""
    value = SHARP_INTERNAL_RESOURCES.get(key, fallback)

    if value is None:
        return None

    value = str(value).strip()

    if not value:
        return None

    return Path(value)

def make_safe_table_id(value):
    """Make a safe HTML table ID."""
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(value)).strip("_") or "table"

def interactive_table_html(df, title, table_id, max_rows=200, empty_message="No rows available."):
    """Build searchable/sortable HTML table markup."""
    table_id = make_safe_table_id(table_id)

    if df is None or df.empty:
        return f"""
        <section class="panel-section">
          <h3>{safe(title)}</h3>
          <p class="muted">{safe(empty_message)}</p>
        </section>
        """

    view = df.head(max_rows).copy()
    total_rows = len(df)
    shown_rows = len(view)

    table = view.to_html(
        index=False,
        escape=True,
        classes="sharp-table interactive-table",
        table_id=table_id,
    )

    return f"""
    <section class="panel-section">
      <div class="section-head">
        <h3>{safe(title)}</h3>
        <span class="row-note">showing {shown_rows} of {total_rows} row(s)</span>
      </div>
      <input
        class="table-search"
        type="text"
        placeholder="Search this table..."
        oninput="filterTable('{table_id}', this.value)"
      >
      <div class="table-wrap">
        {table}
      </div>
    </section>
    """

def output_file_category(path):
    """Classify an output file category."""
    path = Path(path)
    path_text = str(path)

    if "annotation" in path_text:
        return "annotation"

    if "tables" in path_text:
        return "tables"

    if (
        "fimo" in path_text
        or "hmm" in path_text
        or "domains" in path_text
        or "embeddings" in path_text
    ):
        return "evidence"

    if "report" in path_text:
        return "report"

    if "logs" in path_text:
        return "logs"

    if "config" in path_text:
        return "metadata"

    return "metadata"

def data_uri_for_file(path, mime_type=None):
    """Create a data URI for a file."""
    path = Path(path)

    if mime_type is None:
        guessed, _ = mimetypes.guess_type(str(path))
        mime_type = guessed or "application/octet-stream"

    content = path.read_bytes()
    encoded = base64.b64encode(content).decode("ascii")

    return f"data:{mime_type};base64,{encoded}"

def add_to_staging(staged_files, path, arcname, label="", category=""):
    """Add a file to the ZIP staging dictionary."""
    path = Path(path)

    if not path.exists() or not path.is_file():
        return

    arcname = str(Path(arcname))

    if arcname in staged_files:
        return

    staged_files[arcname] = {
        "path": path,
        "arcname": arcname,
        "label": label or path.name,
        "category": category or output_file_category(path),
        "size": path.stat().st_size,
    }

def add_file_to_zip_once(zip_handle, staged_item, written_arc_names):
    """Write one file to ZIP only once."""
    arcname = staged_item["arcname"]

    if arcname in written_arc_names:
        return

    zip_handle.write(staged_item["path"], arcname=arcname)
    written_arc_names.add(arcname)

def kpi_card(title, value, subtitle=""):
    """Build a KPI card."""
    return f"""
    <div class="kpi-card">
      <div class="kpi-title">{safe(title)}</div>
      <div class="kpi-value">{safe(value)}</div>
      <div class="kpi-subtitle">{safe(subtitle)}</div>
    </div>
    """

def read_json_if_exists(path):
    """Read JSON file if available."""
    path = Path(path)

    if not path.exists():
        return {}

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}

def write_json(path, data):
    """Write JSON file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(json_ready(data), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

# =====================================================================
# Load core outputs
# =====================================================================

cds = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("cds_table", ""))
contigs = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("contig_table", ""))

fimo = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("fimo_hits", ""))
fimo_to_cds = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("fimo_to_cds", ""))
fimo_repeat = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("fimo_repeat_hits", ""))
fimo_regulatory = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("fimo_regulatory_hits", ""))

sarp = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("sarp_hits", ""))
domain = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("domain_hits", ""))
embedding = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("embedding_scores", ""))

seed_table = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("neighborhood_seed_table", ""))
neighborhood_index = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("neighborhood_index", ""))
evidence_links = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("neighborhood_evidence_links", ""))

ndf = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("ndf", ""))
regions = read_tsv(SHARP_ANALYSIS_OUTPUTS.get("regions", ""))

n_cds = count_real_rows(cds)
n_contigs = count_real_rows(contigs)

n_fimo = count_real_rows(fimo)
n_fimo_to_cds = count_real_rows(fimo_to_cds)
n_fimo_repeat = count_real_rows(fimo_repeat)
n_fimo_regulatory = count_real_rows(fimo_regulatory)

n_sarp = count_real_rows(sarp)
n_domain = count_real_rows(domain)
n_embedding = count_real_rows(embedding)

n_seeds = count_real_rows(seed_table)
n_neighborhood_index = count_real_rows(neighborhood_index)
n_evidence_links = count_real_rows(evidence_links)
n_ndf = count_real_rows(ndf)
n_regions = count_real_rows(regions, candidate_only=True)

annotation_source = (
    SHARP_PIPELINE_MANIFEST
    .get("annotation_contract", {})
    .get("source", SHARP_CORE_STATUS.get("annotation_source", "NA"))
)

genome_hash = (
    SHARP_PIPELINE_MANIFEST
    .get("annotation_contract", {})
    .get("genome_hash_short", SHARP_SAMPLE.get("genome_hash_short", "NA"))
)

created_at = time.strftime("%Y-%m-%d %H:%M:%S")

top_regions = regions.copy()

if not top_regions.empty and "region_score" in top_regions.columns:
    top_regions["region_score"] = pd.to_numeric(top_regions["region_score"], errors="coerce")

    sort_columns = ["region_score"]
    ascending = [False]

    if "block_id" in top_regions.columns:
        sort_columns.append("block_id")
        ascending.append(True)

    top_regions = top_regions.sort_values(sort_columns, ascending=ascending)

# =====================================================================
# FIMO funnel summary
# =====================================================================

def percent_label(numerator, denominator):
    """Return a percentage label."""
    if denominator is None or denominator == 0:
        return "NA"

    return f"{100 * numerator / denominator:.2f}%"

fimo_funnel_df = pd.DataFrame([
    {
        "stage": "Raw FIMO hits",
        "rows": n_fimo,
        "retained_from_previous": "100.00%" if n_fimo else "NA",
        "interpretation": "All motif hits produced by FIMO; preserved without filtering.",
        "output_file": Path(SHARP_ANALYSIS_OUTPUTS.get("fimo_hits", "")).name,
    },
    {
        "stage": "FIMO to CDS",
        "rows": n_fimo_to_cds,
        "retained_from_previous": percent_label(n_fimo_to_cds, n_fimo),
        "interpretation": "All raw hits mapped to CDS context and intragenic/intergenic state.",
        "output_file": Path(SHARP_ANALYSIS_OUTPUTS.get("fimo_to_cds", "")).name,
    },
    {
        "stage": "Repeat-distance hits",
        "rows": n_fimo_repeat,
        "retained_from_previous": percent_label(n_fimo_repeat, n_fimo_to_cds),
        "interpretation": "Hits satisfying the original repeat-spacing logic: 2 < distance <= 15.",
        "output_file": Path(SHARP_ANALYSIS_OUTPUTS.get("fimo_repeat_hits", "")).name,
    },
    {
        "stage": "Regulatory FIMO hits",
        "rows": n_fimo_regulatory,
        "retained_from_previous": percent_label(n_fimo_regulatory, n_fimo_repeat),
        "interpretation": "Repeat-distance hits that are intergenic and mapped to the next CDS according to motif strand.",
        "output_file": Path(SHARP_ANALYSIS_OUTPUTS.get("fimo_regulatory_hits", "")).name,
    },
    {
        "stage": "Neighborhood seeds",
        "rows": n_seeds,
        "retained_from_previous": percent_label(n_seeds, n_fimo_regulatory),
        "interpretation": "Unique PID anchors used for neighborhood construction.",
        "output_file": Path(SHARP_ANALYSIS_OUTPUTS.get("neighborhood_seed_table", "")).name,
    },
    {
        "stage": "Candidate regions",
        "rows": n_regions,
        "retained_from_previous": percent_label(n_regions, n_seeds),
        "interpretation": "Neighborhoods built around unique PID anchors.",
        "output_file": Path(SHARP_ANALYSIS_OUTPUTS.get("regions", "")).name,
    },
])

FIMO_FUNNEL_TSV = REPORT_DIR / "sharp_fimo_evidence_funnel.tsv"
fimo_funnel_df.to_csv(FIMO_FUNNEL_TSV, sep="\t", index=False)

# =====================================================================
# Resource completeness checklist
# =====================================================================

heptamer_meme = resource_path("heptamer_meme")
sarp_hmm = resource_path("sarp_hmm")
domain_models_hmm = resource_path("domain_models_hmm")
embedding_reference = resource_path("embedding_reference")
scoring_config = resource_path("scoring_config")
report_module = resource_path("report_module")

bakta_db_path = (
    SHARP_BACKEND.get("bakta_db_path")
    or os.environ.get("BAKTA_DB", "")
    or ""
)

resource_rows = [
    {
        "resource": "heptarepeats2.meme",
        "purpose": "FIMO motif search for heptamer repeats / regulatory motifs",
        "path": str(heptamer_meme or ""),
        "required_for_complete_results": True,
        "required_for_this_run": True,
        "loaded": resource_exists(heptamer_meme),
        "note": "Needed to populate FIMO motif hits.",
    },
    {
        "resource": "sarp_custom.hmm",
        "purpose": "HMM search for SARP/BtaD-like regulators",
        "path": str(sarp_hmm or ""),
        "required_for_complete_results": True,
        "required_for_this_run": True,
        "loaded": resource_exists(sarp_hmm),
        "note": "Needed to populate SARP HMM evidence.",
    },
    {
        "resource": "domain_models.hmm",
        "purpose": "hmmscan domain evidence for candidate proteins",
        "path": str(domain_models_hmm or ""),
        "required_for_complete_results": True,
        "required_for_this_run": True,
        "loaded": resource_exists(domain_models_hmm),
        "note": "Needed to populate domain evidence.",
    },
    {
        "resource": "domain_models.hmm.h3*",
        "purpose": "Pressed HMMER index files for domain_models.hmm",
        "path": str(domain_models_hmm or "") + ".h3f/.h3i/.h3m/.h3p",
        "required_for_complete_results": False,
        "required_for_this_run": resource_exists(domain_models_hmm),
        "loaded": hmm_is_pressed(domain_models_hmm) if resource_exists(domain_models_hmm) else False,
        "note": "Can be generated automatically with hmmpress when domain_models.hmm is available.",
    },
    {
        "resource": "reference_embeddings.parquet",
        "purpose": "Reference embedding database for similarity/scoring",
        "path": str(embedding_reference or ""),
        "required_for_complete_results": True,
        "required_for_this_run": True,
        "loaded": resource_exists(embedding_reference),
        "note": "Needed to compare candidate proteins against curated references.",
    },
    {
        "resource": "sharp_scoring_config.json",
        "purpose": "Scoring thresholds and neighborhood parameters",
        "path": str(scoring_config or ""),
        "required_for_complete_results": True,
        "required_for_this_run": True,
        "loaded": resource_exists(scoring_config),
        "note": "Usually generated by the notebook, but should be curated for final scoring.",
    },
    {
        "resource": "operon_fig_colab.py",
        "purpose": "Optional rich operon/neighborhood visualization",
        "path": str(report_module or ""),
        "required_for_complete_results": False,
        "required_for_this_run": False,
        "loaded": resource_exists(report_module),
        "note": "Optional, but recommended for richer final reports.",
    },
    {
        "resource": "Bakta DB",
        "purpose": "Genome-only annotation backend",
        "path": str(bakta_db_path),
        "required_for_complete_results": False,
        "required_for_this_run": requires_bakta_db_current,
        "loaded": resource_exists(bakta_db_path),
        "note": "Required only for genome-only FASTA input. Not required for FASTA + GFF3/GBFF + FAA input.",
    },
]

resource_df = pd.DataFrame(resource_rows)

resource_df["status"] = resource_df.apply(
    lambda row: (
        "loaded"
        if row["loaded"]
        else (
            "missing_required"
            if row["required_for_this_run"] or row["required_for_complete_results"]
            else "optional_missing"
        )
    ),
    axis=1,
)

missing_required_df = resource_df[
    (resource_df["loaded"] == False)
    & (
        (resource_df["required_for_complete_results"] == True)
        | (resource_df["required_for_this_run"] == True)
    )
].copy()

missing_optional_df = resource_df[
    (resource_df["loaded"] == False)
    & (resource_df["required_for_complete_results"] == False)
    & (resource_df["required_for_this_run"] == False)
].copy()

resource_df.to_csv(RESOURCE_CHECKLIST_TSV, sep="\t", index=False)

write_json(
    RESOURCE_MANIFEST_JSON,
    {
        "created_at": created_at,
        "input_mode": input_mode_current,
        "annotation_mode": annotation_mode_current,
        "requires_bakta_db_for_this_run": requires_bakta_db_current,
        "resources": resource_rows,
        "missing_required_for_complete_results": missing_required_df["resource"].tolist(),
        "missing_optional": missing_optional_df["resource"].tolist(),
    },
)

# =====================================================================
# Resource status HTML
# =====================================================================

if missing_required_df.empty:
    missing_resource_html = """
    <div class="notice ok-notice">
      <strong>Resource status:</strong> all core S(H)ARP resources required for complete evidence are currently loaded.
    </div>
    """
else:
    missing_items = ""

    for _, row in missing_required_df.iterrows():
        missing_items += (
            "<li>"
            f"<strong>{safe(row['resource'])}</strong> — {safe(row['purpose'])}<br>"
            f"<span class='muted'>{safe(row['note'])}</span>"
            "</li>"
        )

    missing_resource_html = f"""
    <div class="notice warn-notice">
      <strong>Resources still needed for complete results:</strong>
      <ul>{missing_items}</ul>
    </div>
    """

# =====================================================================
# Output file inventory for report
# =====================================================================

output_rows = []

for key, value in SHARP_ANALYSIS_OUTPUTS.items():
    path = Path(value)

    output_rows.append({
        "label": key,
        "category": output_file_category(path),
        "file": path.name,
        "exists": path.exists(),
        "size": file_size_label(path) if path.exists() else "missing",
    })

for label, path in [
    ("interactive_html_report", REPORT_HTML),
    ("resource_manifest", RESOURCE_MANIFEST_JSON),
    ("resource_checklist", RESOURCE_CHECKLIST_TSV),
    ("fimo_evidence_funnel", FIMO_FUNNEL_TSV),
    ("package_manifest", PACKAGE_MANIFEST_JSON),
]:
    output_rows.append({
        "label": label,
        "category": "report",
        "file": Path(path).name,
        "exists": Path(path).exists(),
        "size": file_size_label(path) if Path(path).exists() else "pending",
    })

output_files_df = pd.DataFrame(output_rows)

# =====================================================================
# Build interactive HTML report
# =====================================================================

fimo_funnel_table = interactive_table_html(
    fimo_funnel_df,
    "FIMO evidence funnel",
    "fimo_funnel_table",
    max_rows=50,
)

top_regions_table = interactive_table_html(
    top_regions,
    "Candidate regions",
    "candidate_regions_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="No candidate regions were detected.",
)

seed_table_html = interactive_table_html(
    seed_table,
    "Neighborhood seed table",
    "neighborhood_seed_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="No neighborhood seeds were detected.",
)

neighborhood_index_html = interactive_table_html(
    neighborhood_index,
    "Neighborhood index",
    "neighborhood_index_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="No neighborhood index rows were generated.",
)

evidence_links_html = interactive_table_html(
    evidence_links,
    "Neighborhood evidence links",
    "neighborhood_evidence_links_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="No neighborhood evidence links were generated.",
)

ndf_table_html = interactive_table_html(
    ndf,
    "Neighborhood dataframe",
    "ndf_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="The neighborhood dataframe is empty.",
)

raw_fimo_table_html = interactive_table_html(
    fimo,
    "Raw FIMO hits",
    "raw_fimo_hits_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="No raw FIMO hits were detected.",
)

fimo_to_cds_table_html = interactive_table_html(
    fimo_to_cds,
    "FIMO to CDS mapping",
    "fimo_to_cds_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="No FIMO to CDS mapping rows were generated.",
)

fimo_repeat_table_html = interactive_table_html(
    fimo_repeat,
    "FIMO repeat-distance hits",
    "fimo_repeat_hits_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="No repeat-distance FIMO hits were detected.",
)

fimo_regulatory_table_html = interactive_table_html(
    fimo_regulatory,
    "FIMO regulatory hits",
    "fimo_regulatory_hits_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="No regulatory FIMO hits were detected.",
)

sarp_table_html = interactive_table_html(
    sarp,
    "SARP HMM hits",
    "sarp_hmm_hits_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="SARP HMM evidence is empty or pending.",
)

domain_table_html = interactive_table_html(
    domain,
    "Domain HMM hits",
    "domain_hits_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="Domain HMM evidence is empty or pending.",
)

embedding_table_html = interactive_table_html(
    embedding,
    "Embedding scores",
    "embedding_scores_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="Embedding evidence is empty or pending.",
)

resource_table_html = interactive_table_html(
    resource_df,
    "Resource checklist",
    "resource_checklist_table",
    max_rows=100,
)

output_files_table_html = interactive_table_html(
    output_files_df,
    "Output file inventory",
    "output_file_inventory_table",
    max_rows=300,
)

annotation_table_html = interactive_table_html(
    cds,
    "CDS table",
    "cds_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="CDS table is empty.",
)

contig_table_html = interactive_table_html(
    contigs,
    "Contig table",
    "contig_table",
    max_rows=MAX_TABLE_ROWS_IN_REPORT,
    empty_message="Contig table is empty.",
)

fimo_bar_max = max(n_fimo, 1)

def funnel_bar(label, value, total):
    """Build one funnel bar."""
    width = 0 if total == 0 else max(2, min(100, 100 * value / total))

    return f"""
    <div class="funnel-row">
      <div class="funnel-label">{safe(label)}</div>
      <div class="funnel-bar-wrap">
        <div class="funnel-bar" style="width:{width:.2f}%"></div>
      </div>
      <div class="funnel-value">{safe(value)}</div>
    </div>
    """

funnel_visual_html = f"""
<div class="funnel-card">
  {funnel_bar("Raw FIMO hits", n_fimo, fimo_bar_max)}
  {funnel_bar("FIMO to CDS", n_fimo_to_cds, fimo_bar_max)}
  {funnel_bar("Repeat-distance hits", n_fimo_repeat, fimo_bar_max)}
  {funnel_bar("Regulatory FIMO hits", n_fimo_regulatory, fimo_bar_max)}
  {funnel_bar("Neighborhood seeds", n_seeds, fimo_bar_max)}
  {funnel_bar("Candidate regions", n_regions, fimo_bar_max)}
</div>
"""

report = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>S(H)ARP results report</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
:root {{
  --sharp-bg:#97003F;
  --sharp-dark:#760032;
  --sharp-deep:#4D0022;
  --sharp-pink:#D8AFC0;
  --sharp-soft:#F1DDE6;
  --sharp-white:#FFFFFF;
  --sharp-ink:#2b1020;
  --sharp-muted:#6f5060;
  --sharp-line:rgba(77,0,34,0.16);
}}
* {{
  box-sizing:border-box;
}}
body {{
  margin:0;
  background:#faf7f9;
  color:var(--sharp-ink);
  font-family:Inter,Arial,sans-serif;
  line-height:1.5;
}}
.hero {{
  background:linear-gradient(135deg,var(--sharp-bg),var(--sharp-dark));
  color:var(--sharp-white);
  padding:34px 38px;
}}
.wordmark {{
  font-family:Bodoni 72,Bodoni MT,Didot,Georgia,Times New Roman,serif;
  font-size:44px;
  letter-spacing:0.6px;
}}
.subtitle {{
  color:var(--sharp-soft);
  text-transform:uppercase;
  letter-spacing:1.8px;
  font-size:12px;
  margin-top:4px;
}}
.hero-grid {{
  margin-top:22px;
  display:grid;
  grid-template-columns:repeat(4,minmax(0,1fr));
  gap:14px;
}}
.kpi-card {{
  background:rgba(255,255,255,0.12);
  border:1px solid rgba(255,255,255,0.22);
  border-radius:16px;
  padding:14px 16px;
}}
.kpi-title {{
  color:var(--sharp-soft);
  font-size:12px;
  text-transform:uppercase;
  letter-spacing:1px;
}}
.kpi-value {{
  font-size:28px;
  font-weight:900;
  margin-top:4px;
}}
.kpi-subtitle {{
  color:var(--sharp-soft);
  font-size:12px;
  margin-top:3px;
}}
.main {{
  padding:24px 34px 44px;
}}
.tabs {{
  display:flex;
  flex-wrap:wrap;
  gap:8px;
  margin-bottom:18px;
}}
.tab-button {{
  border:1px solid var(--sharp-line);
  background:white;
  color:var(--sharp-deep);
  padding:10px 14px;
  border-radius:999px;
  font-weight:800;
  cursor:pointer;
}}
.tab-button.active {{
  background:var(--sharp-deep);
  color:white;
}}
.tab-panel {{
  display:none;
}}
.tab-panel.active {{
  display:block;
}}
.panel {{
  background:white;
  border:1px solid var(--sharp-line);
  border-radius:18px;
  padding:18px;
  margin-bottom:18px;
  box-shadow:0 8px 26px rgba(77,0,34,0.06);
}}
.panel h2 {{
  margin:0 0 10px;
  color:var(--sharp-deep);
}}
.panel-section {{
  margin-top:18px;
}}
.section-head {{
  display:flex;
  align-items:baseline;
  justify-content:space-between;
  gap:12px;
}}
.row-note {{
  color:var(--sharp-muted);
  font-size:12px;
}}
.muted {{
  color:var(--sharp-muted);
}}
.notice {{
  border-radius:14px;
  padding:14px 16px;
  margin:16px 0;
}}
.warn-notice {{
  background:#fff1d6;
  border:1px solid #e5b95a;
}}
.ok-notice {{
  background:#edf8ef;
  border:1px solid #6dbb7d;
}}
.resource-warn {{
  margin-top:16px;
  border-radius:14px;
  padding:14px 16px;
  background:#fff1d6;
  border:1px solid #e5b95a;
}}
.table-search {{
  width:100%;
  max-width:420px;
  margin:8px 0 10px;
  padding:9px 12px;
  border:1px solid var(--sharp-line);
  border-radius:10px;
}}
.table-wrap {{
  overflow:auto;
  border:1px solid var(--sharp-line);
  border-radius:12px;
}}
.sharp-table {{
  border-collapse:collapse;
  width:100%;
  min-width:900px;
  font-size:12px;
}}
.sharp-table th {{
  background:var(--sharp-deep);
  color:white;
  padding:8px;
  text-align:left;
  position:sticky;
  top:0;
  cursor:pointer;
}}
.sharp-table td {{
  border-top:1px solid var(--sharp-line);
  padding:7px 8px;
  vertical-align:top;
}}
.sharp-table tr:nth-child(even) td {{
  background:#fcf7fa;
}}
.grid-2 {{
  display:grid;
  grid-template-columns:1fr 1fr;
  gap:16px;
}}
.funnel-card {{
  background:#fbf3f7;
  border:1px solid var(--sharp-line);
  border-radius:16px;
  padding:16px;
  margin-top:12px;
}}
.funnel-row {{
  display:grid;
  grid-template-columns:190px 1fr 90px;
  gap:12px;
  align-items:center;
  margin:8px 0;
}}
.funnel-label {{
  font-weight:800;
  color:var(--sharp-deep);
}}
.funnel-bar-wrap {{
  height:18px;
  border-radius:999px;
  background:#ead4df;
  overflow:hidden;
}}
.funnel-bar {{
  height:18px;
  border-radius:999px;
  background:var(--sharp-bg);
}}
.funnel-value {{
  text-align:right;
  font-weight:900;
}}
.code {{
  font-family:ui-monospace,SFMono-Regular,Menlo,Monaco,Consolas,monospace;
  background:#f4eef2;
  border-radius:8px;
  padding:2px 6px;
}}
@media (max-width:900px) {{
  .hero-grid {{
    grid-template-columns:repeat(2,minmax(0,1fr));
  }}
  .grid-2 {{
    grid-template-columns:1fr;
  }}
}}
</style>
<script>
function showTab(tabId) {{
  document.querySelectorAll(".tab-panel").forEach(function(panel) {{
    panel.classList.remove("active");
  }});

  document.querySelectorAll(".tab-button").forEach(function(button) {{
    button.classList.remove("active");
  }});

  var panel = document.getElementById(tabId);
  var button = document.querySelector('[data-tab="' + tabId + '"]');

  if (panel) {{
    panel.classList.add("active");
  }}

  if (button) {{
    button.classList.add("active");
  }}
}}

function filterTable(tableId, query) {{
  var table = document.getElementById(tableId);

  if (!table) {{
    return;
  }}

  query = String(query || "").toLowerCase();

  var rows = table.querySelectorAll("tbody tr");

  rows.forEach(function(row) {{
    var text = row.innerText.toLowerCase();
    row.style.display = text.includes(query) ? "" : "none";
  }});
}}

function sortTable(table, columnIndex) {{
  var tbody = table.tBodies[0];

  if (!tbody) {{
    return;
  }}

  var rows = Array.from(tbody.rows);
  var currentDirection = table.getAttribute("data-sort-direction") || "asc";
  var nextDirection = currentDirection === "asc" ? "desc" : "asc";

  rows.sort(function(a, b) {{
    var aText = a.cells[columnIndex] ? a.cells[columnIndex].innerText.trim() : "";
    var bText = b.cells[columnIndex] ? b.cells[columnIndex].innerText.trim() : "";

    var aNum = Number(aText.replace(/,/g, ""));
    var bNum = Number(bText.replace(/,/g, ""));

    if (!Number.isNaN(aNum) && !Number.isNaN(bNum)) {{
      return nextDirection === "asc" ? aNum - bNum : bNum - aNum;
    }}

    return nextDirection === "asc"
      ? aText.localeCompare(bText)
      : bText.localeCompare(aText);
  }});

  rows.forEach(function(row) {{
    tbody.appendChild(row);
  }});

  table.setAttribute("data-sort-direction", nextDirection);
}}

document.addEventListener("DOMContentLoaded", function() {{
  document.querySelectorAll("table.interactive-table").forEach(function(table) {{
    table.querySelectorAll("th").forEach(function(th, columnIndex) {{
      th.addEventListener("click", function() {{
        sortTable(table, columnIndex);
      }});
    }});
  }});

  showTab("summary");
}});
</script>
</head>
<body>
<header class="hero">
  <div class="wordmark">S(H)ARP</div>
  <div class="subtitle">Streptomyces Hidden Antibiotic Regulated Pathways — interactive report</div>

  <div class="hero-grid">
    {kpi_card("Candidate regions", n_regions, "neighborhoods selected")}
    {kpi_card("CDS", n_cds, f"{n_contigs} contig(s)")}
    {kpi_card("Raw FIMO hits", n_fimo, "all preserved")}
    {kpi_card("Regulatory FIMO hits", n_fimo_regulatory, "used as PID seeds")}
  </div>
</header>

<main class="main">
  <div class="tabs">
    <button class="tab-button active" data-tab="summary" onclick="showTab('summary')">Summary</button>
    <button class="tab-button" data-tab="fimo" onclick="showTab('fimo')">FIMO funnel</button>
    <button class="tab-button" data-tab="regions" onclick="showTab('regions')">Regions</button>
    <button class="tab-button" data-tab="evidence" onclick="showTab('evidence')">Evidence</button>
    <button class="tab-button" data-tab="annotation" onclick="showTab('annotation')">Annotation</button>
    <button class="tab-button" data-tab="resources" onclick="showTab('resources')">Resources</button>
    <button class="tab-button" data-tab="files" onclick="showTab('files')">Files</button>
  </div>

  <section id="summary" class="tab-panel active">
    <div class="panel">
      <h2>Run summary</h2>
      <p>
        This report was generated at <span class="code">{safe(created_at)}</span>.
        Annotation source: <span class="code">{safe(annotation_source)}</span>.
        Genome hash: <span class="code">{safe(genome_hash)}</span>.
      </p>
      <p>
        The current Cell 3 logic preserves all raw FIMO hits, maps them to CDS context,
        applies the original repeat-distance/intergenic regulatory logic, and builds
        neighborhoods from unique PID anchors rather than from every raw motif hit.
      </p>
      {missing_resource_html}
      <div class="grid-2">
        <div>
          <h3>Evidence status</h3>
          <ul>
            <li>Raw FIMO hits: <strong>{n_fimo}</strong></li>
            <li>FIMO to CDS rows: <strong>{n_fimo_to_cds}</strong></li>
            <li>FIMO repeat-distance hits: <strong>{n_fimo_repeat}</strong></li>
            <li>FIMO regulatory hits: <strong>{n_fimo_regulatory}</strong></li>
            <li>SARP HMM hits: <strong>{n_sarp}</strong> — {safe(status_label(sarp))}</li>
            <li>Domain hits: <strong>{n_domain}</strong> — {safe(status_label(domain))}</li>
            <li>Embedding hits: <strong>{n_embedding}</strong> — {safe(status_label(embedding))}</li>
          </ul>
        </div>
        <div>
          <h3>Neighborhood status</h3>
          <ul>
            <li>Neighborhood seeds: <strong>{n_seeds}</strong></li>
            <li>Neighborhood index rows: <strong>{n_neighborhood_index}</strong></li>
            <li>Evidence links: <strong>{n_evidence_links}</strong></li>
            <li>NDF rows: <strong>{n_ndf}</strong></li>
            <li>Candidate regions: <strong>{n_regions}</strong></li>
          </ul>
        </div>
      </div>
      {funnel_visual_html}
    </div>
  </section>

  <section id="fimo" class="tab-panel">
    <div class="panel">
      <h2>FIMO evidence funnel</h2>
      <p>
        The raw FIMO table is preserved in full. The regulatory candidate subset follows
        the original Git-style logic: motifs are sorted by contig/strand/start, repeat
        distance is calculated as current motif start minus previous motif stop, hits with
        <span class="code">2 &lt; distance &lt;= 15</span> are retained, intragenic hits are removed,
        and the remaining hits are mapped to the next CDS according to motif strand.
      </p>
      {funnel_visual_html}
      {fimo_funnel_table}
      {fimo_regulatory_table_html}
      {fimo_repeat_table_html}
      {fimo_to_cds_table_html}
      {raw_fimo_table_html}
    </div>
  </section>

  <section id="regions" class="tab-panel">
    <div class="panel">
      <h2>Candidate regions and neighborhoods</h2>
      <p>
        Neighborhoods are constructed from unique PID seeds. With the current defaults,
        each region spans up to {NEIGHBORHOOD_BEFORE_GENES if 'NEIGHBORHOOD_BEFORE_GENES' in globals() else 10}
        genes before and {NEIGHBORHOOD_AFTER_GENES if 'NEIGHBORHOOD_AFTER_GENES' in globals() else 10}
        genes after the anchor.
      </p>
      {top_regions_table}
      {seed_table_html}
      {neighborhood_index_html}
      {evidence_links_html}
      {ndf_table_html}
    </div>
  </section>

  <section id="evidence" class="tab-panel">
    <div class="panel">
      <h2>Non-FIMO evidence</h2>
      <p>
        These tables are populated when the corresponding S(H)ARP resources are available.
      </p>
      {sarp_table_html}
      {domain_table_html}
      {embedding_table_html}
    </div>
  </section>

  <section id="annotation" class="tab-panel">
    <div class="panel">
      <h2>Annotation</h2>
      <p>
        Standardized annotation files are included in the complete ZIP package.
      </p>
      {contig_table_html}
      {annotation_table_html}
    </div>
  </section>

  <section id="resources" class="tab-panel">
    <div class="panel">
      <h2>Resource checklist</h2>
      {missing_resource_html}
      {resource_table_html}
    </div>
  </section>

  <section id="files" class="tab-panel">
    <div class="panel">
      <h2>Output files</h2>
      <p>
        The ZIP package contains the complete run directory, report files, manifests,
        normalized inputs when available, and configuration files needed for reproducibility.
      </p>
      {output_files_table_html}
    </div>
  </section>
</main>
</body>
</html>
"""

REPORT_HTML.write_text(report, encoding="utf-8")

# =====================================================================
# Stage complete results package
# =====================================================================

staged_files = {}

add_to_staging(
    staged_files,
    REPORT_HTML,
    "report/sharp_results.html",
    label="interactive_html_report",
    category="report",
)

add_to_staging(
    staged_files,
    RESOURCE_MANIFEST_JSON,
    "report/sharp_resource_manifest.json",
    label="resource_manifest",
    category="report",
)

add_to_staging(
    staged_files,
    RESOURCE_CHECKLIST_TSV,
    "report/sharp_resource_checklist.tsv",
    label="resource_checklist",
    category="report",
)

add_to_staging(
    staged_files,
    FIMO_FUNNEL_TSV,
    "report/sharp_fimo_evidence_funnel.tsv",
    label="fimo_evidence_funnel",
    category="report",
)

for path in sorted(RUN_DIR.rglob("*")):
    if not path.is_file():
        continue

    if path.resolve() == EXPORT_ZIP.resolve():
        continue

    if not INCLUDE_LOGS_IN_EXPORT and LOGS_DIR in path.parents:
        continue

    relative = path.relative_to(RUN_DIR)

    add_to_staging(
        staged_files,
        path,
        f"run/{relative}",
        label=path.name,
        category=output_file_category(path),
    )

if CONFIG_DIR.exists():
    for path in sorted(CONFIG_DIR.glob("sharp_*.json")):
        add_to_staging(
            staged_files,
            path,
            f"config/{path.name}",
            label=path.name,
            category="metadata",
        )

if INCLUDE_NORMALIZED_INPUTS_IN_EXPORT and NORMALIZED_DIR.exists():
    for path in sorted(NORMALIZED_DIR.glob("*")):
        if path.is_file():
            add_to_staging(
                staged_files,
                path,
                f"input_normalized/{path.name}",
                label=path.name,
                category="input",
            )

package_manifest = {
    "created_at": created_at,
    "analysis_name": SHARP_SAMPLE.get("analysis_name", "sharp_run"),
    "report_html": "report/sharp_results.html",
    "counts": {
        "contigs": n_contigs,
        "cds": n_cds,
        "raw_fimo_hits": n_fimo,
        "fimo_to_cds": n_fimo_to_cds,
        "fimo_repeat_hits": n_fimo_repeat,
        "fimo_regulatory_hits": n_fimo_regulatory,
        "sarp_hmm_hits": n_sarp,
        "domain_hits": n_domain,
        "embedding_hits": n_embedding,
        "neighborhood_seeds": n_seeds,
        "neighborhood_index_rows": n_neighborhood_index,
        "neighborhood_evidence_links": n_evidence_links,
        "ndf_rows": n_ndf,
        "candidate_regions": n_regions,
    },
    "files": [
        {
            "arcname": item["arcname"],
            "label": item["label"],
            "category": item["category"],
            "size_bytes": item["size"],
        }
        for item in staged_files.values()
    ],
}

write_json(PACKAGE_MANIFEST_JSON, package_manifest)

add_to_staging(
    staged_files,
    PACKAGE_MANIFEST_JSON,
    "report/sharp_package_manifest.json",
    label="package_manifest",
    category="report",
)

if EXPORT_ZIP.exists():
    EXPORT_ZIP.unlink()

written_arc_names = set()

with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zip_handle:
    for item in staged_files.values():
        add_file_to_zip_once(zip_handle, item, written_arc_names)

# =====================================================================
# Public output metadata
# =====================================================================

SHARP_REPORT_OUTPUTS = {
    "report_html": str(REPORT_HTML),
    "export_zip": str(EXPORT_ZIP),
    "resource_manifest": str(RESOURCE_MANIFEST_JSON),
    "resource_checklist": str(RESOURCE_CHECKLIST_TSV),
    "package_manifest": str(PACKAGE_MANIFEST_JSON),
    "fimo_evidence_funnel": str(FIMO_FUNNEL_TSV),
    "created_at": created_at,
    "counts": {
        "contigs": n_contigs,
        "cds": n_cds,
        "raw_fimo_hits": n_fimo,
        "fimo_to_cds": n_fimo_to_cds,
        "fimo_repeat_hits": n_fimo_repeat,
        "fimo_regulatory_hits": n_fimo_regulatory,
        "sarp_hmm_hits": n_sarp,
        "domain_hits": n_domain,
        "embedding_hits": n_embedding,
        "neighborhood_seeds": n_seeds,
        "neighborhood_index_rows": n_neighborhood_index,
        "neighborhood_evidence_links": n_evidence_links,
        "ndf_rows": n_ndf,
        "candidate_regions": n_regions,
    },
    "missing_required_resources": missing_required_df["resource"].tolist(),
    "missing_optional_resources": missing_optional_df["resource"].tolist(),
}

globals()["SHARP_REPORT_OUTPUTS"] = SHARP_REPORT_OUTPUTS

context = read_json_if_exists(SHARP_CONTEXT_FILE)

context.update({
    "report_outputs": SHARP_REPORT_OUTPUTS,
    "report_html": str(REPORT_HTML),
    "export_zip": str(EXPORT_ZIP),
    "resource_manifest": str(RESOURCE_MANIFEST_JSON),
    "resource_checklist": str(RESOURCE_CHECKLIST_TSV),
    "package_manifest": str(PACKAGE_MANIFEST_JSON),
    "fimo_evidence_funnel": str(FIMO_FUNNEL_TSV),
})

write_json(SHARP_CONTEXT_FILE, context)

# =====================================================================
# Build one-button download control
# =====================================================================

zip_size_mb = EXPORT_ZIP.stat().st_size / (1024 ** 2)

colab_callback_ready = False

try:
    from google.colab import output, files

    def sharp_download_complete_package():
        files.download(str(EXPORT_ZIP))

    output.register_callback(
        "sharp.download_complete_package",
        sharp_download_complete_package,
    )

    colab_callback_ready = True

except Exception:
    colab_callback_ready = False

if colab_callback_ready:
    download_control_html = """
    <button
      class="button"
      onclick="google.colab.kernel.invokeFunction('sharp.download_complete_package', [], {})"
    >
      Download complete S(H)ARP package (.zip)
    </button>
    """

elif zip_size_mb <= MAX_EMBEDDED_ZIP_BUTTON_MB:
    zip_data_uri = data_uri_for_file(EXPORT_ZIP, "application/zip")

    download_control_html = f"""
    <a
      class="button"
      href="{html.escape(zip_data_uri)}"
      download="{html.escape(EXPORT_ZIP.name)}"
    >
      Download complete S(H)ARP package (.zip)
    </a>
    """

else:
    download_control_html = """
    <div class="disabled-button">
      ZIP is too large for an embedded browser button
    </div>
    """

if missing_required_df.empty:
    missing_summary_html = """
    <div class="ok-summary">
      All core S(H)ARP resources required for complete evidence are currently loaded.
    </div>
    """
else:
    missing_summary_items = ""

    for _, row in missing_required_df.iterrows():
        missing_summary_items += (
            "<li>"
            f"<strong>{safe(row['resource'])}</strong> — {safe(row['purpose'])}"
            "</li>"
        )

    missing_summary_html = f"""
    <div class="resource-warn">
      <strong>Resources still needed for complete results:</strong>
      <ul>{missing_summary_items}</ul>
    </div>
    """

# =====================================================================
# Final public panel
# =====================================================================

display(HTML(f"""
<style>
.sharp-final-panel {{
  background:linear-gradient(135deg,#97003F,#760032);
  color:white;
  padding:20px 22px;
  border-radius:18px;
  margin:14px 0;
  font-family:Inter,Arial,sans-serif;
  box-shadow:0 10px 32px rgba(0,0,0,0.18);
}}
.sharp-final-panel h2 {{
  margin:0 0 8px;
  font-size:22px;
}}
.sharp-final-panel p {{
  margin:6px 0;
  color:#F1DDE6;
}}
.sharp-final-grid {{
  display:grid;
  grid-template-columns:repeat(4,minmax(0,1fr));
  gap:10px;
  margin:14px 0;
}}
.sharp-final-kpi {{
  background:rgba(255,255,255,0.12);
  border:1px solid rgba(255,255,255,0.22);
  border-radius:14px;
  padding:11px 12px;
}}
.sharp-final-kpi .label {{
  color:#F1DDE6;
  font-size:11px;
  text-transform:uppercase;
  letter-spacing:1px;
}}
.sharp-final-kpi .value {{
  font-size:23px;
  font-weight:900;
  margin-top:3px;
}}
.button {{
  display:inline-block;
  border:0;
  background:#D8AFC0;
  color:#4D0022;
  padding:12px 16px;
  border-radius:999px;
  font-weight:900;
  cursor:pointer;
  text-decoration:none;
  margin-top:10px;
}}
.disabled-button {{
  display:inline-block;
  background:#F1DDE6;
  color:#4D0022;
  padding:12px 16px;
  border-radius:999px;
  font-weight:900;
  margin-top:10px;
}}
.resource-warn {{
  margin-top:14px;
  border-radius:14px;
  padding:12px 14px;
  background:#fff1d6;
  color:#4D0022;
  border:1px solid #e5b95a;
}}
.ok-summary {{
  margin-top:14px;
  border-radius:14px;
  padding:12px 14px;
  background:#edf8ef;
  color:#17451f;
  border:1px solid #6dbb7d;
}}
@media (max-width:900px) {{
  .sharp-final-grid {{
    grid-template-columns:repeat(2,minmax(0,1fr));
  }}
}}
</style>

<div class="sharp-final-panel">
  <h2>Interactive report and complete package ready.</h2>
  <p>The interactive HTML report is shown below and is also included inside the complete ZIP package.</p>
  <p>ZIP package size: <code>{safe(file_size_label(EXPORT_ZIP))}</code></p>
  <p>Files included: <code>{safe(len(staged_files))}</code></p>

  <div class="sharp-final-grid">
    <div class="sharp-final-kpi">
      <div class="label">Raw FIMO hits</div>
      <div class="value">{safe(n_fimo)}</div>
    </div>
    <div class="sharp-final-kpi">
      <div class="label">Regulatory FIMO</div>
      <div class="value">{safe(n_fimo_regulatory)}</div>
    </div>
    <div class="sharp-final-kpi">
      <div class="label">Neighborhood seeds</div>
      <div class="value">{safe(n_seeds)}</div>
    </div>
    <div class="sharp-final-kpi">
      <div class="label">Candidate regions</div>
      <div class="value">{safe(n_regions)}</div>
    </div>
  </div>

  <div class="actions">
    {download_control_html}
  </div>

  {missing_summary_html}
</div>
"""))

# =====================================================================
# Compact console summary
# =====================================================================

print("S(H)ARP report/export finished.")
print("download_button_ready:", True)
print("zip_size:", file_size_label(EXPORT_ZIP))
print("files_in_package:", len(staged_files))
print("html_report_included_in_zip:", True)
print("contigs:", n_contigs)
print("cds:", n_cds)
print("raw_fimo_hits:", n_fimo)
print("fimo_to_cds:", n_fimo_to_cds)
print("fimo_repeat_hits:", n_fimo_repeat)
print("fimo_regulatory_hits:", n_fimo_regulatory)
print("sarp_hmm_hits:", n_sarp)
print("domain_hits:", n_domain)
print("embedding_hits:", n_embedding)
print("neighborhood_seeds:", n_seeds)
print("neighborhood_index_rows:", n_neighborhood_index)
print("neighborhood_evidence_links:", n_evidence_links)
print("ndf_rows:", n_ndf)
print("candidate_regions:", n_regions)

if missing_required_df.empty:
    print("missing_required_resources: none")
else:
    print("missing_required_resources:")
    for resource_name in missing_required_df["resource"].tolist():
        print("-", resource_name)

if PRINT_TECHNICAL_PATHS:
    print()
    print("Technical paths:")
    print("report_html:", REPORT_HTML)
    print("export_zip:", EXPORT_ZIP)
    print("resource_manifest:", RESOURCE_MANIFEST_JSON)
    print("resource_checklist:", RESOURCE_CHECKLIST_TSV)
    print("fimo_evidence_funnel:", FIMO_FUNNEL_TSV)
    print("package_manifest:", PACKAGE_MANIFEST_JSON)

# =====================================================================
# Inline interactive report preview
# =====================================================================

if SHOW_REPORT_INLINE:
    print()
    print("Inline interactive report preview:")

    try:
        report_html_text = REPORT_HTML.read_text(encoding="utf-8")

        display(HTML(f"""
        <div style="
            border:1px solid rgba(151,0,63,0.25);
            border-radius:16px;
            overflow:hidden;
            margin-top:10px;
            background:white;
        ">
          <iframe
            srcdoc="{html.escape(report_html_text, quote=True)}"
            width="100%"
            height="760"
            style="border:0; background:white;"
          ></iframe>
        </div>
        """))

    except Exception as exc:
        print("Inline preview was not available:", exc)